In [ ]:
# ============================================================
# PHASE 1 (V4-IN-PLACE): REVIEWER-DRIVEN DATA PREP
# Better negative coverage + patient-wise feature extraction
# Keeps core output filenames/keys compatible with later phases
# ============================================================

import os
import sys
import gc
import json
import glob
import time
import tarfile
import random
import warnings
import platform
from dataclasses import dataclass, asdict
from datetime import datetime

import numpy as np
import pandas as pd
import nibabel as nib
from PIL import Image

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import timm

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================

@dataclass
class Config:
    SEED: int = 42

    BRATS2021_TAR: str = "/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_Training_Data.tar"
    BRATS2021_EXTRACT_DIR: str = "/kaggle/temp/brats2021_extracted"
    BRATS2020_DIR: str = "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
    WORK_DIR: str = "/kaggle/working"

    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS: int = 2

    IMG_SIZE: int = 224
    MODALITIES: tuple = ("t1", "t1ce", "t2", "flair")
    FEATURE_DIM: int = 384
    BACKBONE: str = "vit_small_patch16_224"

    TUMOR_PIXEL_THRESHOLD: int = 100
    THRESHOLD_VALUES: tuple = (50, 100, 200)

    MAX_SELECTED_SLICES: int = 48
    MAX_POSITIVE_FRACTION: float = 0.60
    BOUNDARY_NEG_RADIUS: int = 6
    FAR_NEG_MIN_DISTANCE: int = 12
    TARGET_BOUNDARY_NEG: int = 12
    TARGET_FAR_NEG: int = 10
    MIN_SELECTED_IF_NO_TUMOR: int = 24

    TRAIN_RATIO: float = 0.68
    VAL_RATIO: float = 0.16
    TUNE_RATIO: float = 0.08
    TEST_RATIO: float = 0.08

    BATCH_SIZE: int = 16
    EXTRACT_EXTERNAL_2020: bool = False
    SAVE_SPLIT_CSVS: bool = True

CFG = Config()

SLICE_ROLE_TO_ID = {
    "far_neg": 0,
    "boundary_neg": 1,
    "positive": 2,
    "mid_neg": 3,
    "no_tumor": 4,
}
ID_TO_SLICE_ROLE = {v: k for k, v in SLICE_ROLE_TO_ID.items()}


# ============================================================
# UTILS
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)

def find_first(patterns):
    for p in patterns:
        files = glob.glob(p)
        if files:
            return files[0]
    return None

def percentile_normalize(vol):
    vol = np.asarray(vol, dtype=np.float32)
    p1, p99 = np.percentile(vol, [1, 99])
    vol = np.clip(vol, p1, p99)
    mean, std = vol.mean(), vol.std()
    if std > 1e-6:
        vol = (vol - mean) / std
    vmin, vmax = vol.min(), vol.max()
    if vmax - vmin > 1e-6:
        vol = (vol - vmin) / (vmax - vmin)
    return vol.astype(np.float32)

def save_env_info():
    info = {
        "timestamp": datetime.now().isoformat(),
        "python_version": sys.version,
        "platform": platform.platform(),
        "device": CFG.DEVICE,
        "config": asdict(CFG)
    }
    try:
        import psutil
        info["system_ram_gb"] = round(psutil.virtual_memory().total / 1e9, 2)
    except Exception:
        info["system_ram_gb"] = None

    if torch.cuda.is_available():
        info["gpu_name"] = torch.cuda.get_device_name(0)
        info["gpu_mem_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
        info["cuda_version"] = torch.version.cuda
        info["torch_version"] = torch.__version__

    save_json(info, os.path.join(CFG.WORK_DIR, "env_info_v3.json"))

def spread_sample(indices, k):
    indices = sorted(set(int(x) for x in indices))
    if k <= 0 or len(indices) == 0:
        return []
    if len(indices) <= k:
        return indices

    pos = np.linspace(0, len(indices) - 1, k)
    picked = []
    used = set()
    for p in pos:
        j = int(round(float(p)))
        j = max(0, min(j, len(indices) - 1))
        if j not in used:
            used.add(j)
            picked.append(indices[j])

    if len(picked) < k:
        for x in indices:
            if x not in picked:
                picked.append(x)
            if len(picked) == k:
                break

    return sorted(picked[:k])

def nearest_positive_distance(all_indices, pos_indices):
    if len(pos_indices) == 0:
        return {int(i): -1 for i in all_indices}
    pos_arr = np.asarray(sorted(pos_indices), dtype=np.int64)
    out = {}
    for i in all_indices:
        out[int(i)] = int(np.min(np.abs(pos_arr - int(i))))
    return out

def sample_no_tumor_indices(depth, k):
    k = min(int(k), int(depth))
    if k <= 0:
        return []

    center_k = min(k // 2, max(8, k // 3))
    mid = depth // 2
    half = center_k // 2
    center_block = list(range(max(0, mid - half), min(depth, mid - half + center_k)))

    remain = k - len(center_block)
    others = [i for i in range(depth) if i not in set(center_block)]
    spread = spread_sample(others, remain)
    selected = sorted(set(center_block + spread))

    if len(selected) < k:
        for i in range(depth):
            if i not in selected:
                selected.append(i)
            if len(selected) == k:
                break

    return sorted(selected[:k])


# ============================================================
# DATA DISCOVERY
# ============================================================

def extract_2021_if_needed():
    ensure_dir(CFG.BRATS2021_EXTRACT_DIR)
    existing = glob.glob(os.path.join(CFG.BRATS2021_EXTRACT_DIR, "BraTS2021_*"))
    if not existing:
        existing = glob.glob(os.path.join(CFG.BRATS2021_EXTRACT_DIR, "*", "BraTS2021_*"))
    if len(existing) > 100:
        print(f"✓ BraTS2021 already extracted: {len(existing)} folders")
        return CFG.BRATS2021_EXTRACT_DIR

    print("Extracting BraTS2021...")
    with tarfile.open(CFG.BRATS2021_TAR, "r") as tar:
        tar.extractall(CFG.BRATS2021_EXTRACT_DIR)
    print("✓ Extraction complete")
    return CFG.BRATS2021_EXTRACT_DIR

def list_patient_dirs(base_dir, tag="BraTS2021"):
    if tag == "BraTS2021":
        dirs = sorted(glob.glob(os.path.join(base_dir, "BraTS2021_*")))
        if not dirs:
            dirs = sorted(glob.glob(os.path.join(base_dir, "*", "BraTS2021_*")))
    else:
        dirs = sorted(glob.glob(os.path.join(base_dir, "BraTS20*")))
        if not dirs:
            dirs = sorted(glob.glob(os.path.join(base_dir, "*", "BraTS20*")))
    return [d for d in dirs if os.path.isdir(d)]

def get_bundle(patient_dir):
    bundle = {}
    for mod in CFG.MODALITIES:
        bundle[mod] = find_first([
            os.path.join(patient_dir, f"*_{mod}.nii.gz"),
            os.path.join(patient_dir, f"*_{mod}.nii"),
            os.path.join(patient_dir, f"*{mod}.nii.gz"),
            os.path.join(patient_dir, f"*{mod}.nii"),
        ])
    bundle["seg"] = find_first([
        os.path.join(patient_dir, "*_seg.nii.gz"),
        os.path.join(patient_dir, "*_seg.nii"),
        os.path.join(patient_dir, "*seg.nii.gz"),
        os.path.join(patient_dir, "*seg.nii"),
    ])
    return bundle


# ============================================================
# REVIEWER-DRIVEN SLICE SELECTION
# ============================================================

def select_slices_v4(
    tumor_pixels,
    threshold,
    max_selected,
    max_positive_fraction,
    boundary_neg_radius,
    far_neg_min_distance,
    target_boundary_neg,
    target_far_neg,
    min_no_tumor
):
    tumor_pixels = np.asarray(tumor_pixels, dtype=np.int64)
    depth = len(tumor_pixels)
    all_idx = list(range(depth))
    pos = np.where(tumor_pixels >= threshold)[0].astype(np.int64).tolist()

    if len(pos) == 0:
        selected = sample_no_tumor_indices(depth, min(min_no_tumor, max_selected))
        role_map = {int(i): "no_tumor" for i in selected}
        dist_map = {int(i): -1 for i in selected}
        meta = {
            "has_positive": 0,
            "first_pos": -1,
            "last_pos": -1,
            "roi_start": -1,
            "roi_end": -1,
            "n_selected": int(len(selected)),
            "n_selected_positive": 0,
            "n_selected_boundary_neg": 0,
            "n_selected_far_neg": 0,
            "n_selected_mid_neg": 0,
            "n_selected_no_tumor": int(len(selected)),
        }
        return selected, role_map, dist_map, meta

    first_pos = int(min(pos))
    last_pos = int(max(pos))
    roi_start = max(0, first_pos - boundary_neg_radius)
    roi_end = min(depth - 1, last_pos + boundary_neg_radius)

    dist_map_full = nearest_positive_distance(all_idx, pos)
    neg = [i for i in all_idx if i not in set(pos)]

    boundary_neg = [i for i in neg if dist_map_full[i] <= boundary_neg_radius]
    far_neg = [i for i in neg if dist_map_full[i] >= far_neg_min_distance]
    mid_neg = [i for i in neg if i not in set(boundary_neg) and i not in set(far_neg)]

    max_pos_keep = max(1, int(round(max_selected * max_positive_fraction)))
    sel_pos = spread_sample(pos, min(len(pos), max_pos_keep))

    remaining = max_selected - len(sel_pos)

    boundary_neg_sorted = sorted(
        boundary_neg,
        key=lambda x: (dist_map_full[x], abs(x - (first_pos + last_pos) / 2.0))
    )
    sel_boundary = boundary_neg_sorted[:min(target_boundary_neg, remaining, len(boundary_neg_sorted))]
    remaining -= len(sel_boundary)

    sel_far = spread_sample(far_neg, min(target_far_neg, remaining, len(far_neg)))
    remaining -= len(sel_far)

    mid_sorted = sorted(mid_neg, key=lambda x: (dist_map_full[x], abs(x - (first_pos + last_pos) / 2.0)))
    sel_mid = spread_sample(mid_sorted, min(remaining, len(mid_sorted)))
    remaining -= len(sel_mid)

    selected = sorted(set(sel_pos + sel_boundary + sel_far + sel_mid))

    if len(selected) < max_selected:
        remaining_pool = [i for i in all_idx if i not in set(selected)]
        fill = spread_sample(remaining_pool, max_selected - len(selected))
        selected = sorted(set(selected + fill))

    if len(selected) > max_selected:
        selected = spread_sample(selected, max_selected)

    sel_set = set(selected)
    boundary_set = set(sel_boundary) & sel_set
    far_set = set(sel_far) & sel_set
    pos_set = set(sel_pos) & sel_set

    role_map = {}
    for i in selected:
        if i in pos_set:
            role_map[int(i)] = "positive"
        elif i in boundary_set:
            role_map[int(i)] = "boundary_neg"
        elif i in far_set:
            role_map[int(i)] = "far_neg"
        else:
            role_map[int(i)] = "mid_neg"

    dist_map = {int(i): int(dist_map_full[i]) for i in selected}

    meta = {
        "has_positive": 1,
        "first_pos": int(first_pos),
        "last_pos": int(last_pos),
        "roi_start": int(roi_start),
        "roi_end": int(roi_end),
        "n_selected": int(len(selected)),
        "n_selected_positive": int(sum(role_map[i] == "positive" for i in selected)),
        "n_selected_boundary_neg": int(sum(role_map[i] == "boundary_neg" for i in selected)),
        "n_selected_far_neg": int(sum(role_map[i] == "far_neg" for i in selected)),
        "n_selected_mid_neg": int(sum(role_map[i] == "mid_neg" for i in selected)),
        "n_selected_no_tumor": 0,
    }
    return selected, role_map, dist_map, meta


# ============================================================
# METADATA BUILD
# ============================================================

def build_metadata(dataset_dir, dataset_tag="BraTS2021"):
    patient_dirs = list_patient_dirs(dataset_dir, dataset_tag)
    print(f"\nBuilding metadata for {dataset_tag}: {len(patient_dirs)} patients")

    patient_rows = []
    slice_rows = []
    row_id = 0

    for i, patient_dir in enumerate(tqdm(patient_dirs, desc=f"{dataset_tag} audit")):
        pid = os.path.basename(patient_dir)
        bundle = get_bundle(patient_dir)

        if bundle["seg"] is None:
            continue
        if any(bundle[m] is None for m in CFG.MODALITIES):
            continue

        try:
            seg = nib.load(bundle["seg"]).get_fdata()
        except Exception:
            continue

        depth = seg.shape[2]
        tumor_pixels = [int(np.sum(seg[:, :, z] > 0)) for z in range(depth)]

        selected, role_map, dist_map, meta = select_slices_v4(
            tumor_pixels=tumor_pixels,
            threshold=CFG.TUMOR_PIXEL_THRESHOLD,
            max_selected=CFG.MAX_SELECTED_SLICES,
            max_positive_fraction=CFG.MAX_POSITIVE_FRACTION,
            boundary_neg_radius=CFG.BOUNDARY_NEG_RADIUS,
            far_neg_min_distance=CFG.FAR_NEG_MIN_DISTANCE,
            target_boundary_neg=CFG.TARGET_BOUNDARY_NEG,
            target_far_neg=CFG.TARGET_FAR_NEG,
            min_no_tumor=CFG.MIN_SELECTED_IF_NO_TUMOR
        )

        patient_rows.append({
            "patient_id": pid,
            "patient_path": patient_dir,
            "depth": int(depth),
            "total_tumor_voxels": int(np.sum(tumor_pixels)),
            "max_tumor_pixels": int(np.max(tumor_pixels)),
            "positive_slices_thr50": int(np.sum(np.array(tumor_pixels) >= 50)),
            "positive_slices_thr100": int(np.sum(np.array(tumor_pixels) >= 100)),
            "positive_slices_thr200": int(np.sum(np.array(tumor_pixels) >= 200)),
            "selected_slices": int(meta["n_selected"]),
            "selected_positive": int(meta["n_selected_positive"]),
            "selected_boundary_neg": int(meta["n_selected_boundary_neg"]),
            "selected_far_neg": int(meta["n_selected_far_neg"]),
            "selected_mid_neg": int(meta["n_selected_mid_neg"]),
            "selected_no_tumor": int(meta["n_selected_no_tumor"]),
            "has_positive": int(meta["has_positive"]),
            "roi_start": int(meta["roi_start"]),
            "roi_end": int(meta["roi_end"]),
            "first_pos": int(meta["first_pos"]),
            "last_pos": int(meta["last_pos"]),
        })

        for order_idx, z in enumerate(selected):
            tp = int(tumor_pixels[z])
            label = int(tp >= CFG.TUMOR_PIXEL_THRESHOLD)
            role = role_map[int(z)]

            slice_rows.append({
                "row_id": row_id,
                "patient_id": pid,
                "patient_path": patient_dir,
                "slice_idx": int(z),
                "slice_order": int(order_idx),
                "tumor_pixels": tp,
                "label": label,
                "slice_role": role,
                "slice_role_id": int(SLICE_ROLE_TO_ID[role]),
                "distance_to_positive": int(dist_map[int(z)]),
                "roi_start": int(meta["roi_start"]),
                "roi_end": int(meta["roi_end"]),
                "has_positive_patient": int(meta["has_positive"]),
                "t1_path": bundle["t1"],
                "t1ce_path": bundle["t1ce"],
                "t2_path": bundle["t2"],
                "flair_path": bundle["flair"],
                "seg_path": bundle["seg"],
                "modality_mask_t1": 1,
                "modality_mask_t1ce": 1,
                "modality_mask_t2": 1,
                "modality_mask_flair": 1
            })
            row_id += 1

        del seg
        if (i + 1) % 25 == 0:
            clear_memory()

    patient_df = pd.DataFrame(patient_rows)
    slice_df = pd.DataFrame(slice_rows)
    return patient_df, slice_df

def threshold_sensitivity(slice_df):
    rows = []
    for thr in CFG.THRESHOLD_VALUES:
        labels = (slice_df["tumor_pixels"].values >= thr).astype(int)
        rows.append({
            "threshold": int(thr),
            "n_slices": int(len(labels)),
            "positive_slices": int(labels.sum()),
            "negative_slices": int(len(labels) - labels.sum()),
            "positive_pct": float(100 * labels.mean()) if len(labels) else 0.0
        })
    sens_df = pd.DataFrame(rows)
    sens_df.to_csv(os.path.join(CFG.WORK_DIR, "threshold_sensitivity_v3.csv"), index=False)

def save_selection_reports(patient_df, slice_df):
    role_summary = slice_df.groupby("slice_role").agg(
        n_slices=("label", "size"),
        positive_labels=("label", "sum")
    ).reset_index()
    role_summary["negative_labels"] = role_summary["n_slices"] - role_summary["positive_labels"]
    role_summary.to_csv(os.path.join(CFG.WORK_DIR, "slice_role_summary_v3.csv"), index=False)

    split_role_summary = slice_df.groupby(["split", "slice_role"]).size().reset_index(name="n_slices")
    split_role_summary.to_csv(os.path.join(CFG.WORK_DIR, "split_slice_role_summary_v3.csv"), index=False)

    selection_policy = {
        "phase": "phase1_v4_in_place",
        "tumor_pixel_threshold": CFG.TUMOR_PIXEL_THRESHOLD,
        "max_selected_slices": CFG.MAX_SELECTED_SLICES,
        "max_positive_fraction": CFG.MAX_POSITIVE_FRACTION,
        "boundary_neg_radius": CFG.BOUNDARY_NEG_RADIUS,
        "far_neg_min_distance": CFG.FAR_NEG_MIN_DISTANCE,
        "target_boundary_neg": CFG.TARGET_BOUNDARY_NEG,
        "target_far_neg": CFG.TARGET_FAR_NEG,
        "min_selected_if_no_tumor": CFG.MIN_SELECTED_IF_NO_TUMOR,
        "slice_role_to_id": SLICE_ROLE_TO_ID,
        "timestamp": datetime.now().isoformat()
    }
    save_json(selection_policy, os.path.join(CFG.WORK_DIR, "selection_policy_v3.json"))


# ============================================================
# SPLITS
# ============================================================

def robust_qcut(values, q=4):
    uniq = np.unique(values)
    if len(uniq) < 2:
        return np.array(["single"] * len(values))
    bins = min(q, len(uniq))
    result = pd.qcut(values, q=bins, duplicates="drop").astype(str)
    return np.asarray(result)

def prepare_safe_strata(values, min_count=2):
    s = pd.Series(values).astype(str).copy()

    # Repeatedly merge too-small groups into "rare"
    for _ in range(10):
        counts = s.value_counts()
        rare = counts[counts < min_count].index.tolist()
        if len(rare) == 0:
            break
        s = s.where(~s.isin(rare), "rare")

    counts = s.value_counts()

    # Stratification impossible if fewer than 2 strata remain
    # or any remaining stratum still has < 2 samples
    if len(counts) < 2 or counts.min() < min_count:
        return None

    return s.values

def split_ids_with_optional_stratify(ids, strata, test_size, seed):
    ids = np.asarray(ids)
    if strata is not None:
        strata = prepare_safe_strata(strata, min_count=2)
    if strata is None:
        return train_test_split(ids, test_size=test_size, random_state=seed, shuffle=True)
    return train_test_split(ids, test_size=test_size, random_state=seed, shuffle=True, stratify=strata)

def make_patient_splits(patient_df, slice_df):
    patient_tmp = patient_df.copy()

    frac = patient_tmp["positive_slices_thr100"] / patient_tmp["depth"].clip(lower=1)
    s1 = robust_qcut(frac.values, q=4)
    s2 = robust_qcut(np.log1p(patient_tmp["max_tumor_pixels"].values), q=4)

    # Only use s3 if the column exists in your new Phase 1
    if "selected_boundary_neg" in patient_tmp.columns:
        s3 = robust_qcut(patient_tmp["selected_boundary_neg"].values, q=4)
        patient_tmp["stratum"] = (
            s1.astype(str) + "__" + s2.astype(str) + "__" + s3.astype(str)
        )
    else:
        patient_tmp["stratum"] = s1.astype(str) + "__" + s2.astype(str)

    patient_tmp["stratum"] = prepare_safe_strata(patient_tmp["stratum"].values, min_count=2)
    if patient_tmp["stratum"] is None:
        patient_tmp["stratum"] = np.array(["all"] * len(patient_tmp), dtype=object)

    pids = patient_tmp["patient_id"].values
    strata = patient_tmp["stratum"]

    temp_ratio = CFG.VAL_RATIO + CFG.TUNE_RATIO + CFG.TEST_RATIO
    train_ids, temp_ids = split_ids_with_optional_stratify(
        pids, strata, test_size=temp_ratio, seed=CFG.SEED
    )

    temp_df = patient_tmp[patient_tmp["patient_id"].isin(temp_ids)].copy()
    temp_pids = temp_df["patient_id"].values
    temp_strata = temp_df["stratum"].values if "stratum" in temp_df.columns else None

    val_share = CFG.VAL_RATIO / temp_ratio
    val_ids, rem_ids = split_ids_with_optional_stratify(
        temp_pids, temp_strata, test_size=(1.0 - val_share), seed=CFG.SEED
    )

    rem_df = temp_df[temp_df["patient_id"].isin(rem_ids)].copy()
    rem_pids = rem_df["patient_id"].values
    rem_strata = rem_df["stratum"].values if "stratum" in rem_df.columns else None

    tune_share = CFG.TUNE_RATIO / (CFG.TUNE_RATIO + CFG.TEST_RATIO)
    tune_ids, test_ids = split_ids_with_optional_stratify(
        rem_pids, rem_strata, test_size=(1.0 - tune_share), seed=CFG.SEED
    )

    split_map = {}
    for x in train_ids:
        split_map[x] = "train"
    for x in val_ids:
        split_map[x] = "val"
    for x in tune_ids:
        split_map[x] = "tune"
    for x in test_ids:
        split_map[x] = "test"

    patient_df = patient_df.copy()
    slice_df = slice_df.copy()
    patient_df["split"] = patient_df["patient_id"].map(split_map)
    slice_df["split"] = slice_df["patient_id"].map(split_map)

    assert set(train_ids).isdisjoint(set(val_ids))
    assert set(train_ids).isdisjoint(set(tune_ids))
    assert set(train_ids).isdisjoint(set(test_ids))
    assert set(val_ids).isdisjoint(set(tune_ids))
    assert set(val_ids).isdisjoint(set(test_ids))
    assert set(tune_ids).isdisjoint(set(test_ids))

    train_slice = slice_df[slice_df["split"] == "train"].copy()
    train_pat = train_slice.groupby("patient_id")["label"].max()
    n0 = int((train_pat == 0).sum())
    n1 = int((train_pat == 1).sum())
    total = max(n0 + n1, 1)
    bag_class_weights = {
        "0": float(total / max(2 * n0, 1)),
        "1": float(total / max(2 * n1, 1))
    }

    n0s = int((train_slice["label"] == 0).sum())
    n1s = int((train_slice["label"] == 1).sum())
    totals = max(n0s + n1s, 1)
    slice_class_weights = {
        "0": float(totals / max(2 * n0s, 1)),
        "1": float(totals / max(2 * n1s, 1))
    }

    split_info = {
        "seed": CFG.SEED,
        "config": asdict(CFG),
        "train_patients": sorted(train_ids.tolist()),
        "val_patients": sorted(val_ids.tolist()),
        "tune_patients": sorted(tune_ids.tolist()),
        "test_patients": sorted(test_ids.tolist()),
        "bag_class_weights": bag_class_weights,
        "slice_class_weights": slice_class_weights,
        "timestamp": datetime.now().isoformat()
    }
    save_json(split_info, os.path.join(CFG.WORK_DIR, "patient_splits_v3.json"))
    return patient_df, slice_df, split_info

# ============================================================
# FEATURE EXTRACTOR
# ============================================================

class FrozenViT(nn.Module):
    def __init__(self, model_name=CFG.BACKBONE):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.backbone.eval()
        for p in self.backbone.parameters():
            p.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            return self.backbone(x)

class SlicePreprocessor:
    def __init__(self):
        self.transform = transforms.Compose([
            transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def __call__(self, arr2d):
        arr2d = np.asarray(arr2d, dtype=np.float32)
        arr2d = np.clip(arr2d, 0.0, 1.0)
        img = Image.fromarray((arr2d * 255).astype(np.uint8)).convert("RGB")
        return self.transform(img)

def load_norm_vol(path):
    vol = nib.load(path).get_fdata()
    vol = percentile_normalize(vol)
    return vol.astype(np.float32)

def extract_features_for_split(df, split_name):
    print(f"\nExtracting features for {split_name}: {len(df)} slices")
    if len(df) == 0:
        out = {
            "features_t1": np.zeros((0, CFG.FEATURE_DIM), dtype=np.float32),
            "features_t1ce": np.zeros((0, CFG.FEATURE_DIM), dtype=np.float32),
            "features_t2": np.zeros((0, CFG.FEATURE_DIM), dtype=np.float32),
            "features_flair": np.zeros((0, CFG.FEATURE_DIM), dtype=np.float32),
            "labels": np.zeros((0,), dtype=np.int64),
            "patient_ids": np.array([], dtype=str),
            "slice_idxs": np.zeros((0,), dtype=np.int64),
            "modality_mask": np.zeros((0, 4), dtype=np.int64),
            "tumor_pixels": np.zeros((0,), dtype=np.int64),
            "distance_to_positive": np.zeros((0,), dtype=np.int64),
            "slice_role_id": np.zeros((0,), dtype=np.int64),
            "has_positive_patient": np.zeros((0,), dtype=np.int64),
        }
        np.savez_compressed(os.path.join(CFG.WORK_DIR, f"{split_name}_features_4mod_v3.npz"), **out)
        return out

    model = FrozenViT().to(CFG.DEVICE)
    preproc = SlicePreprocessor()

    feats_t1, feats_t1ce, feats_t2, feats_flair = [], [], [], []
    labels_all = []
    patient_ids_all = []
    slice_idxs_all = []
    mod_masks_all = []
    tumor_pixels_all = []
    distance_all = []
    role_id_all = []
    has_positive_all = []

    patient_groups = list(df.groupby("patient_id", sort=True))

    for _, pdf in tqdm(patient_groups, desc=f"Features {split_name}"):
        pdf = pdf.sort_values("slice_idx").reset_index(drop=True)

        try:
            vol_t1 = load_norm_vol(pdf.loc[0, "t1_path"])
            vol_t1ce = load_norm_vol(pdf.loc[0, "t1ce_path"])
            vol_t2 = load_norm_vol(pdf.loc[0, "t2_path"])
            vol_flair = load_norm_vol(pdf.loc[0, "flair_path"])
        except Exception:
            continue

        slice_indices = pdf["slice_idx"].astype(int).tolist()
        labels_np = pdf["label"].astype(np.int64).values
        mod_masks_np = pdf[[
            "modality_mask_t1", "modality_mask_t1ce", "modality_mask_t2", "modality_mask_flair"
        ]].astype(np.int64).values
        tumor_pixels_np = pdf["tumor_pixels"].astype(np.int64).values
        distance_np = pdf["distance_to_positive"].astype(np.int64).values
        role_id_np = pdf["slice_role_id"].astype(np.int64).values
        has_pos_np = pdf["has_positive_patient"].astype(np.int64).values
        pid_np = pdf["patient_id"].astype(str).values

        for start in range(0, len(slice_indices), CFG.BATCH_SIZE):
            end = min(start + CFG.BATCH_SIZE, len(slice_indices))
            idxs = slice_indices[start:end]

            batch_t1 = torch.stack([preproc(vol_t1[:, :, z]) for z in idxs], dim=0).to(CFG.DEVICE)
            batch_t1ce = torch.stack([preproc(vol_t1ce[:, :, z]) for z in idxs], dim=0).to(CFG.DEVICE)
            batch_t2 = torch.stack([preproc(vol_t2[:, :, z]) for z in idxs], dim=0).to(CFG.DEVICE)
            batch_flair = torch.stack([preproc(vol_flair[:, :, z]) for z in idxs], dim=0).to(CFG.DEVICE)

            f_t1 = model(batch_t1).cpu().numpy().astype(np.float32)
            f_t1ce = model(batch_t1ce).cpu().numpy().astype(np.float32)
            f_t2 = model(batch_t2).cpu().numpy().astype(np.float32)
            f_flair = model(batch_flair).cpu().numpy().astype(np.float32)

            feats_t1.append(f_t1)
            feats_t1ce.append(f_t1ce)
            feats_t2.append(f_t2)
            feats_flair.append(f_flair)

            labels_all.append(labels_np[start:end])
            patient_ids_all.extend(pid_np[start:end].tolist())
            slice_idxs_all.append(np.asarray(idxs, dtype=np.int64))
            mod_masks_all.append(mod_masks_np[start:end].astype(np.int64))
            tumor_pixels_all.append(tumor_pixels_np[start:end].astype(np.int64))
            distance_all.append(distance_np[start:end].astype(np.int64))
            role_id_all.append(role_id_np[start:end].astype(np.int64))
            has_positive_all.append(has_pos_np[start:end].astype(np.int64))

        del vol_t1, vol_t1ce, vol_t2, vol_flair
        clear_memory()

    out = {
        "features_t1": np.concatenate(feats_t1, axis=0).astype(np.float32),
        "features_t1ce": np.concatenate(feats_t1ce, axis=0).astype(np.float32),
        "features_t2": np.concatenate(feats_t2, axis=0).astype(np.float32),
        "features_flair": np.concatenate(feats_flair, axis=0).astype(np.float32),
        "labels": np.concatenate(labels_all, axis=0).astype(np.int64),
        "patient_ids": np.asarray(patient_ids_all),
        "slice_idxs": np.concatenate(slice_idxs_all, axis=0).astype(np.int64),
        "modality_mask": np.concatenate(mod_masks_all, axis=0).astype(np.int64),
        "tumor_pixels": np.concatenate(tumor_pixels_all, axis=0).astype(np.int64),
        "distance_to_positive": np.concatenate(distance_all, axis=0).astype(np.int64),
        "slice_role_id": np.concatenate(role_id_all, axis=0).astype(np.int64),
        "has_positive_patient": np.concatenate(has_positive_all, axis=0).astype(np.int64),
    }

    save_path = os.path.join(CFG.WORK_DIR, f"{split_name}_features_4mod_v3.npz")
    np.savez_compressed(save_path, **out)
    print(f"✓ saved: {save_path}")

    del model
    clear_memory()
    return out


# ============================================================
# EXTERNAL 2020 (OPTIONAL)
# ============================================================

def extract_external_2020():
    if not CFG.EXTRACT_EXTERNAL_2020:
        print("Skipping BraTS2020 external extraction (default OFF)")
        return

    if not os.path.exists(CFG.BRATS2020_DIR):
        print("BraTS2020 path not found, skipping")
        return

    patient_df, slice_df = build_metadata(CFG.BRATS2020_DIR, "BraTS2020")
    patient_df.to_csv(os.path.join(CFG.WORK_DIR, "brats2020_patient_summary_v3.csv"), index=False)
    slice_df.to_csv(os.path.join(CFG.WORK_DIR, "brats2020_slice_metadata_v3.csv"), index=False)
    extract_features_for_split(slice_df, "brats2020_external")


# ============================================================
# MAIN
# ============================================================

def run_phase1():
    start = time.time()
    ensure_dir(CFG.WORK_DIR)
    set_seed(CFG.SEED)
    save_env_info()

    print("=" * 72)
    print("PHASE 1 (V4-IN-PLACE): REVIEWER-DRIVEN DATA PREP")
    print("=" * 72)
    print(f"Device: {CFG.DEVICE}")
    print(f"Backbone: {CFG.BACKBONE}")
    print(f"External BraTS2020 extraction: {CFG.EXTRACT_EXTERNAL_2020}")
    print("=" * 72)

    brats2021_root = extract_2021_if_needed()

    patient_df, slice_df = build_metadata(brats2021_root, "BraTS2021")
    patient_df.to_csv(os.path.join(CFG.WORK_DIR, "patient_summary_v3.csv"), index=False)
    slice_df.to_csv(os.path.join(CFG.WORK_DIR, "slice_metadata_v3.csv"), index=False)

    threshold_sensitivity(slice_df)
    patient_df, slice_df, split_info = make_patient_splits(patient_df, slice_df)

    if CFG.SAVE_SPLIT_CSVS:
        for split_name in ["train", "val", "tune", "test"]:
            sdf = slice_df[slice_df["split"] == split_name].sort_values(["patient_id", "slice_idx"])
            sdf.to_csv(os.path.join(CFG.WORK_DIR, f"{split_name}_slice_metadata_v3.csv"), index=False)

    split_patient_summary = patient_df.groupby("split")["patient_id"].nunique().reset_index(name="patients")
    split_slice_summary = slice_df.groupby("split").agg(
        slices=("label", "size"),
        positive_slices=("label", "sum"),
    ).reset_index()
    split_slice_summary["negative_slices"] = split_slice_summary["slices"] - split_slice_summary["positive_slices"]

    split_patient_summary.to_csv(os.path.join(CFG.WORK_DIR, "split_patient_summary_v3.csv"), index=False)
    split_slice_summary.to_csv(os.path.join(CFG.WORK_DIR, "split_slice_summary_v3.csv"), index=False)

    save_selection_reports(patient_df, slice_df)

    train_df = slice_df[slice_df["split"] == "train"].sort_values(["patient_id", "slice_idx"]).copy()
    val_df = slice_df[slice_df["split"] == "val"].sort_values(["patient_id", "slice_idx"]).copy()
    tune_df = slice_df[slice_df["split"] == "tune"].sort_values(["patient_id", "slice_idx"]).copy()
    test_df = slice_df[slice_df["split"] == "test"].sort_values(["patient_id", "slice_idx"]).copy()

    extract_features_for_split(train_df, "train")
    extract_features_for_split(val_df, "val")
    extract_features_for_split(tune_df, "tune")
    extract_features_for_split(test_df, "test")

    extract_external_2020()

    manifest = {
        "phase": "phase1_v4_in_place_reviewer_driven",
        "timestamp": datetime.now().isoformat(),
        "config": asdict(CFG),
        "outputs": [
            "env_info_v3.json",
            "patient_summary_v3.csv",
            "slice_metadata_v3.csv",
            "threshold_sensitivity_v3.csv",
            "patient_splits_v3.json",
            "split_patient_summary_v3.csv",
            "split_slice_summary_v3.csv",
            "slice_role_summary_v3.csv",
            "split_slice_role_summary_v3.csv",
            "selection_policy_v3.json",
            "train_features_4mod_v3.npz",
            "val_features_4mod_v3.npz",
            "tune_features_4mod_v3.npz",
            "test_features_4mod_v3.npz"
        ]
    }
    save_json(manifest, os.path.join(CFG.WORK_DIR, "phase1_manifest_v3.json"))

    elapsed = (time.time() - start) / 60.0
    print("\n" + "=" * 72)
    print("✓ PHASE 1 COMPLETE")
    print("=" * 72)
    print(f"Time: {elapsed:.1f} min")
    print(split_patient_summary.to_string(index=False))
    print(split_slice_summary.to_string(index=False))
    print("=" * 72)

if __name__ == "__main__":
    run_phase1()


In [ ]:
import os
import shutil
from pathlib import Path

SRC = Path("/kaggle/input/datasets/absanim/phase-1-output")   # change only if Kaggle changes the folder name
DST = Path("/kaggle/working")

if not SRC.exists():
    print("Available input folders:")
    for p in Path("/kaggle/input").iterdir():
        print(" -", p.name)
    raise FileNotFoundError(f"Not found: {SRC}")

copied = 0
for item in SRC.iterdir():
    dst_path = DST / item.name

    if item.is_file():
        shutil.copy2(item, dst_path)
        copied += 1
    elif item.is_dir():
        if dst_path.exists():
            shutil.rmtree(dst_path)
        shutil.copytree(item, dst_path)
        copied += 1

print(f"Copied {copied} items from {SRC} to {DST}")
print("\nNow in /kaggle/working:")
for x in sorted(os.listdir(DST)):
    print(" -", x)


In [ ]:
# ============================================================
# PHASE 2 (V6): REVIEWER-DRIVEN SLICE-CENTER TRAINING
# Works with Phase 1 V4-IN-PLACE outputs
# Focus: better macro-F1 + specificity via balanced center windows
# ============================================================

import os
import gc
import json
import time
import math
import copy
import random
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, balanced_accuracy_score,
    average_precision_score
)

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================

class Config:
    PHASE1_DIR = "/kaggle/working"
    WORK_DIR = "/kaggle/working"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    SEEDS = [42, 123, 456]

    NUM_CLASSES = 2
    D_IN = 384
    D_MODEL = 128

    MAX_SLICES = 48              # must be >= Phase 1 selected max
    TRAIN_WINDOW = 11            # odd number, local window around center slice
    TOP_K = 4
    PROTO_MOMENTUM = 0.90

    TOTAL_EPOCHS = 32
    LR = 8e-4
    WEIGHT_DECAY = 1e-4
    BATCH_SIZE = 32
    EVAL_BATCH_SIZE = 64
    NUM_WORKERS = 0
    EVAL_EVERY = 1
    PATIENCE = 8
    CLIP_GRAD_NORM = 1.0

    MODALITY_DROPOUT = 0.10
    FIG_DPI = 300

    CENTER_GAMMA_POS = 1.5
    CENTER_GAMMA_NEG = 3.0
    CENTER_LOSS_WEIGHT = 1.00
    BAG_LOSS_WEIGHT = 0.20
    SMOOTHNESS_LOSS_WEIGHT = 0.06
    DIVERSITY_LOSS_WEIGHT = 0.01

    SAMPLER_POSITIVE_BOOST = 1.20
    SAMPLER_BOUNDARY_NEG_BOOST = 1.45
    SAMPLER_FAR_NEG_BOOST = 1.20
    SAMPLER_NO_TUMOR_BOOST = 1.10

    ROLE_LOSS_MULTIPLIER = {
        0: 1.25,   # far_neg
        1: 1.60,   # boundary_neg
        2: 1.00,   # positive
        3: 1.00,   # mid_neg
        4: 1.15,   # no_tumor
    }

CFG = Config()

SLICE_ROLE_TO_ID = {
    "far_neg": 0,
    "boundary_neg": 1,
    "positive": 2,
    "mid_neg": 3,
    "no_tumor": 4,
}
ID_TO_SLICE_ROLE = {v: k for k, v in SLICE_ROLE_TO_ID.items()}


# ============================================================
# UTILS
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def safe_auc(labels, prob1):
    labels = np.asarray(labels)
    prob1 = np.asarray(prob1)
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return float("nan")
    try:
        return float(roc_auc_score(labels, prob1))
    except Exception:
        return float("nan")

def safe_pr_auc(labels, prob1):
    labels = np.asarray(labels)
    prob1 = np.asarray(prob1)
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return float("nan")
    try:
        return float(average_precision_score(labels, prob1))
    except Exception:
        return float("nan")

def nanmean_or_zero(x):
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0 or np.all(np.isnan(x)):
        return 0.0
    return float(np.nanmean(x))

def nanstd_or_zero(x):
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0 or np.all(np.isnan(x)):
        return 0.0
    return float(np.nanstd(x))


# ============================================================
# LOAD DATA
# ============================================================

def load_npz(name):
    path = os.path.join(CFG.PHASE1_DIR, f"{name}_features_4mod_v3.npz")
    with np.load(path, allow_pickle=True) as data:
        out = {k: data[k] for k in data.files}
    return out

def _zeros_if_missing(data_dict, key, shape, dtype):
    if key in data_dict:
        return data_dict[key]
    return np.zeros(shape, dtype=dtype)

def _validate_data_dict(data_dict, split_name="unknown"):
    required = [
        "features_t1", "features_t1ce", "features_t2", "features_flair",
        "labels", "patient_ids", "slice_idxs", "modality_mask"
    ]
    missing = [k for k in required if k not in data_dict]
    if missing:
        raise KeyError(f"{split_name} missing keys: {missing}")

    n = len(data_dict["labels"])
    for k in required:
        if len(data_dict[k]) != n:
            raise ValueError(f"{split_name}: key '{k}' length mismatch for {k}")

def load_all_data():
    with open(os.path.join(CFG.PHASE1_DIR, "patient_splits_v3.json"), "r") as f:
        splits = json.load(f)

    train_data = load_npz("train")
    val_data = load_npz("val")
    tune_data = load_npz("tune")
    test_data = load_npz("test")

    for name, d in [("train", train_data), ("val", val_data), ("tune", tune_data), ("test", test_data)]:
        _validate_data_dict(d, name)

    print("=" * 72)
    print("LOADING PHASE 1 V4 DATA")
    print("=" * 72)
    print(f"Train slices: {len(train_data['labels'])}")
    print(f"Val slices:   {len(val_data['labels'])}")
    print(f"Tune slices:  {len(tune_data['labels'])}")
    print(f"Test slices:  {len(test_data['labels'])}")
    print(f"Slice class weights: {splits['slice_class_weights']}")
    print("=" * 72 + "\n")

    return splits, train_data, val_data, tune_data, test_data


# ============================================================
# PATIENT BAG DATASET
# kept for compatibility with later phases
# ============================================================

class PatientBagDataset(Dataset):
    def __init__(self, data_dict, max_slices=48):
        self.max_slices = max_slices
        self.items = []

        patient_ids = np.asarray(data_dict["patient_ids"]).astype(str)
        slice_idxs = np.asarray(data_dict["slice_idxs"]).astype(np.int64)

        f_t1 = np.asarray(data_dict["features_t1"], dtype=np.float32)
        f_t1ce = np.asarray(data_dict["features_t1ce"], dtype=np.float32)
        f_t2 = np.asarray(data_dict["features_t2"], dtype=np.float32)
        f_flair = np.asarray(data_dict["features_flair"], dtype=np.float32)
        labels = np.asarray(data_dict["labels"], dtype=np.int64)
        mod_mask = np.asarray(data_dict["modality_mask"], dtype=np.int64)

        n = len(labels)
        dist = _zeros_if_missing(data_dict, "distance_to_positive", (n,), np.int64).astype(np.int64)
        role_id = _zeros_if_missing(data_dict, "slice_role_id", (n,), np.int64).astype(np.int64)
        tumor_pixels = _zeros_if_missing(data_dict, "tumor_pixels", (n,), np.int64).astype(np.int64)
        has_positive_patient = _zeros_if_missing(data_dict, "has_positive_patient", (n,), np.int64).astype(np.int64)

        pid_to_indices = {}
        for i, pid in enumerate(patient_ids):
            pid_to_indices.setdefault(pid, []).append(i)

        for pid in sorted(pid_to_indices.keys()):
            idxs = np.asarray(pid_to_indices[pid], dtype=np.int64)
            order = np.argsort(slice_idxs[idxs], kind="mergesort")
            idxs = idxs[order]

            t1 = f_t1[idxs]
            t1ce = f_t1ce[idxs]
            t2 = f_t2[idxs]
            flair = f_flair[idxs]
            y = labels[idxs]
            mm = mod_mask[idxs]
            sx = slice_idxs[idxs]
            dd = dist[idxs]
            rr = role_id[idxs]
            tp = tumor_pixels[idxs]
            hp = has_positive_patient[idxs]

            valid_len = len(y)
            if valid_len > self.max_slices:
                pos = np.where(y == 1)[0]
                if len(pos) > 0:
                    center = int(np.round(pos.mean()))
                else:
                    center = valid_len // 2
                half = self.max_slices // 2
                lo = max(0, center - half)
                hi = min(valid_len, lo + self.max_slices)
                lo = max(0, hi - self.max_slices)

                t1, t1ce, t2, flair = t1[lo:hi], t1ce[lo:hi], t2[lo:hi], flair[lo:hi]
                y, mm, sx = y[lo:hi], mm[lo:hi], sx[lo:hi]
                dd, rr, tp, hp = dd[lo:hi], rr[lo:hi], tp[lo:hi], hp[lo:hi]
                valid_len = self.max_slices
            else:
                pad = self.max_slices - valid_len
                t1 = np.vstack([t1, np.zeros((pad, t1.shape[1]), dtype=np.float32)])
                t1ce = np.vstack([t1ce, np.zeros((pad, t1ce.shape[1]), dtype=np.float32)])
                t2 = np.vstack([t2, np.zeros((pad, t2.shape[1]), dtype=np.float32)])
                flair = np.vstack([flair, np.zeros((pad, flair.shape[1]), dtype=np.float32)])

                y = np.concatenate([y, np.full(pad, -100, dtype=np.int64)])
                mm = np.vstack([mm, np.zeros((pad, 4), dtype=np.int64)])
                sx = np.concatenate([sx, np.full(pad, -1, dtype=np.int64)])
                dd = np.concatenate([dd, np.full(pad, -1, dtype=np.int64)])
                rr = np.concatenate([rr, np.full(pad, -1, dtype=np.int64)])
                tp = np.concatenate([tp, np.zeros(pad, dtype=np.int64)])
                hp = np.concatenate([hp, np.zeros(pad, dtype=np.int64)])

            seq_mask = torch.zeros(self.max_slices, dtype=torch.bool)
            seq_mask[:valid_len] = True
            bag_mod_mask = (mm[:valid_len].sum(axis=0) > 0).astype(np.int64)
            if bag_mod_mask.sum() == 0:
                bag_mod_mask[:] = 1

            bag_label = int(np.max(y[:valid_len]) > 0)

            self.items.append({
                "t1": torch.tensor(t1, dtype=torch.float32),
                "t1ce": torch.tensor(t1ce, dtype=torch.float32),
                "t2": torch.tensor(t2, dtype=torch.float32),
                "flair": torch.tensor(flair, dtype=torch.float32),
                "bag_label": torch.tensor(bag_label, dtype=torch.long),
                "slice_labels": torch.tensor(y, dtype=torch.long),
                "seq_mask": seq_mask,
                "mod_mask": torch.tensor(bag_mod_mask, dtype=torch.bool),
                "pid": str(pid),
                "slice_idxs": sx.astype(np.int64),
                "distance_to_positive": dd.astype(np.int64),
                "slice_role_id": rr.astype(np.int64),
                "tumor_pixels": tp.astype(np.int64),
                "has_positive_patient": hp.astype(np.int64),
                "valid_len": int(valid_len),
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        x = self.items[idx]
        return (
            x["t1"], x["t1ce"], x["t2"], x["flair"],
            x["bag_label"], x["slice_labels"], x["seq_mask"], x["mod_mask"], x["pid"]
        )


# ============================================================
# CENTER WINDOW DATASET
# training/eval dataset
# ============================================================

class CenterWindowDataset(Dataset):
    def __init__(self, data_dict, window_size=11):
        assert window_size % 2 == 1, "window_size must be odd"
        self.window_size = window_size
        self.half = window_size // 2

        patient_ids = np.asarray(data_dict["patient_ids"]).astype(str)
        slice_idxs = np.asarray(data_dict["slice_idxs"]).astype(np.int64)

        f_t1 = np.asarray(data_dict["features_t1"], dtype=np.float32)
        f_t1ce = np.asarray(data_dict["features_t1ce"], dtype=np.float32)
        f_t2 = np.asarray(data_dict["features_t2"], dtype=np.float32)
        f_flair = np.asarray(data_dict["features_flair"], dtype=np.float32)
        labels = np.asarray(data_dict["labels"], dtype=np.int64)
        mod_mask = np.asarray(data_dict["modality_mask"], dtype=np.int64)

        n = len(labels)
        dist = _zeros_if_missing(data_dict, "distance_to_positive", (n,), np.int64).astype(np.int64)
        role_id = _zeros_if_missing(data_dict, "slice_role_id", (n,), np.int64).astype(np.int64)
        tumor_pixels = _zeros_if_missing(data_dict, "tumor_pixels", (n,), np.int64).astype(np.int64)
        has_positive_patient = _zeros_if_missing(data_dict, "has_positive_patient", (n,), np.int64).astype(np.int64)

        self.patient_data = {}
        pid_to_indices = {}
        for i, pid in enumerate(patient_ids):
            pid_to_indices.setdefault(pid, []).append(i)

        for pid in sorted(pid_to_indices.keys()):
            idxs = np.asarray(pid_to_indices[pid], dtype=np.int64)
            order = np.argsort(slice_idxs[idxs], kind="mergesort")
            idxs = idxs[order]

            self.patient_data[pid] = {
                "t1": f_t1[idxs],
                "t1ce": f_t1ce[idxs],
                "t2": f_t2[idxs],
                "flair": f_flair[idxs],
                "labels": labels[idxs],
                "slice_idxs": slice_idxs[idxs],
                "modality_mask": mod_mask[idxs],
                "distance_to_positive": dist[idxs],
                "slice_role_id": role_id[idxs],
                "tumor_pixels": tumor_pixels[idxs],
                "has_positive_patient": has_positive_patient[idxs],
            }

        self.index = []
        for pid in sorted(self.patient_data.keys()):
            y = self.patient_data[pid]["labels"]
            roles = self.patient_data[pid]["slice_role_id"]
            sx = self.patient_data[pid]["slice_idxs"]
            dd = self.patient_data[pid]["distance_to_positive"]
            tp = self.patient_data[pid]["tumor_pixels"]
            bag_label = int(y.max() > 0)
            for local_idx in range(len(y)):
                self.index.append({
                    "pid": pid,
                    "local_idx": int(local_idx),
                    "center_label": int(y[local_idx]),
                    "bag_label": bag_label,
                    "slice_role_id": int(roles[local_idx]),
                    "slice_idx": int(sx[local_idx]),
                    "distance_to_positive": int(dd[local_idx]),
                    "tumor_pixels": int(tp[local_idx]),
                })

    def __len__(self):
        return len(self.index)

    def _build_window(self, arr, center_idx):
        out = []
        mask = []
        for offset in range(-self.half, self.half + 1):
            j = center_idx + offset
            if 0 <= j < len(arr):
                out.append(arr[j])
                mask.append(True)
            else:
                out.append(np.zeros(arr.shape[1], dtype=np.float32))
                mask.append(False)
        return np.stack(out, axis=0), np.array(mask, dtype=bool)

    def __getitem__(self, idx):
        meta = self.index[idx]
        pid = meta["pid"]
        c = meta["local_idx"]
        d = self.patient_data[pid]

        t1, seq_mask = self._build_window(d["t1"], c)
        t1ce, _ = self._build_window(d["t1ce"], c)
        t2, _ = self._build_window(d["t2"], c)
        flair, _ = self._build_window(d["flair"], c)

        valid_mm = d["modality_mask"]
        bag_mod_mask = (valid_mm.sum(axis=0) > 0).astype(np.int64)
        if bag_mod_mask.sum() == 0:
            bag_mod_mask[:] = 1

        return (
            torch.tensor(t1, dtype=torch.float32),
            torch.tensor(t1ce, dtype=torch.float32),
            torch.tensor(t2, dtype=torch.float32),
            torch.tensor(flair, dtype=torch.float32),
            torch.tensor(int(meta["bag_label"]), dtype=torch.long),
            torch.tensor(int(meta["center_label"]), dtype=torch.long),
            torch.tensor(seq_mask, dtype=torch.bool),
            torch.tensor(bag_mod_mask, dtype=torch.bool),
            torch.tensor(self.half, dtype=torch.long),  # center position always middle
            torch.tensor(int(meta["slice_role_id"]), dtype=torch.long),
            str(pid),
            torch.tensor(int(meta["slice_idx"]), dtype=torch.long),
            torch.tensor(int(meta["distance_to_positive"]), dtype=torch.long),
            torch.tensor(int(meta["tumor_pixels"]), dtype=torch.long),
        )


# ============================================================
# MODEL
# ============================================================

def masked_topk_mean(seq_feats, scores, seq_mask, top_k):
    bsz, _, dim = seq_feats.shape
    out = []
    device = seq_feats.device
    dtype = seq_feats.dtype

    for b in range(bsz):
        valid_idx = torch.where(seq_mask[b])[0]
        if valid_idx.numel() == 0:
            out.append(torch.zeros(dim, device=device, dtype=dtype))
            continue
        vals = scores[b, valid_idx]
        k = min(int(top_k), int(valid_idx.numel()))
        chosen_local = torch.topk(vals, k=k, dim=0).indices
        chosen_idx = valid_idx[chosen_local]
        out.append(seq_feats[b, chosen_idx].mean(dim=0))
    return torch.stack(out, dim=0)

class ReliabilityFusion(nn.Module):
    def __init__(self, d_in=384, d_model=128):
        super().__init__()
        self.proj = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_in, d_model),
                nn.LayerNorm(d_model),
                nn.GELU()
            ) for _ in range(4)
        ])
        self.rel = nn.ModuleList([nn.Linear(d_model, 1) for _ in range(4)])
        self.out = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.LayerNorm(d_model),
            nn.GELU()
        )

    def forward(self, mods, mod_mask):
        proj_feats = []
        rel_logits = []

        for i in range(4):
            p = self.proj[i](mods[i])
            proj_feats.append(p)
            rel_logits.append(self.rel[i](p))

        feat_stack = torch.stack(proj_feats, dim=2)    # [B,S,4,D]
        rel_stack = torch.cat(rel_logits, dim=-1)      # [B,S,4]
        rel_stack = rel_stack.masked_fill(~mod_mask[:, None, :], -1e4)

        weights = torch.softmax(rel_stack, dim=-1)
        fused = (feat_stack * weights.unsqueeze(-1)).sum(dim=2)

        valid = mod_mask[:, None, :, None].float()
        mean_feat = (feat_stack * valid).sum(dim=2) / valid.sum(dim=2).clamp_min(1.0)

        out = self.out(torch.cat([fused, mean_feat], dim=-1))
        return out, weights

class HybridMILPool(nn.Module):
    def __init__(self, d_model=128, top_k=4):
        super().__init__()
        self.top_k = top_k
        self.attn = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.Tanh(),
            nn.Linear(d_model // 2, 1)
        )

    def forward(self, seq_feats, slice_logits, seq_mask):
        probs = torch.softmax(slice_logits, dim=-1)
        entropy = -(probs * torch.log(probs.clamp_min(1e-8))).sum(dim=-1) / math.log(2.0)
        reliability = 1.0 - entropy

        attn_logits = self.attn(seq_feats).squeeze(-1)
        attn_logits = attn_logits + torch.log(reliability.clamp_min(1e-6))
        attn_logits = attn_logits.masked_fill(~seq_mask, -1e9)

        attn_w = torch.softmax(attn_logits, dim=1)
        attn_pool = (seq_feats * attn_w.unsqueeze(-1)).sum(dim=1)

        pos_feat = masked_topk_mean(seq_feats, probs[..., 1], seq_mask, self.top_k)
        neg_feat = masked_topk_mean(seq_feats, probs[..., 0], seq_mask, self.top_k)

        bag_embed = torch.cat([attn_pool, pos_feat, neg_feat], dim=-1)
        return bag_embed, attn_w, reliability

class PUMAMILLite(nn.Module):
    def __init__(self, d_in=384, d_model=128, max_slices=48, top_k=4, proto_momentum=0.9):
        super().__init__()
        self.d_model = d_model
        self.max_slices = max_slices
        self.top_k = top_k
        self.proto_momentum = proto_momentum

        self.fusion = ReliabilityFusion(d_in=d_in, d_model=d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, max_slices, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dim_feedforward=256,
            dropout=0.10,
            batch_first=True,
            activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.slice_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 2)
        )

        self.pool = HybridMILPool(d_model=d_model, top_k=top_k)

        self.register_buffer("prototypes", torch.zeros(2, d_model * 3))
        self.register_buffer("proto_counts", torch.zeros(2))

        self.final_head = nn.Sequential(
            nn.LayerNorm(d_model * 3 + 2),
            nn.Linear(d_model * 3 + 2, d_model),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(d_model, 2)
        )

        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def modality_dropout(self, mod_mask):
        if (not self.training) or CFG.MODALITY_DROPOUT <= 0:
            return mod_mask

        keep = mod_mask.clone()
        rand = torch.rand_like(keep.float())
        keep = keep & (rand > CFG.MODALITY_DROPOUT)

        for i in range(keep.size(0)):
            if keep[i].sum() == 0:
                valid = torch.where(mod_mask[i])[0]
                if valid.numel() > 0:
                    j = valid[torch.randint(0, valid.numel(), (1,), device=valid.device)]
                    keep[i, j] = True
                else:
                    keep[i] = True

        return keep

    def update_prototypes(self, bag_embed, bag_labels):
        with torch.no_grad():
            for c in [0, 1]:
                mask = (bag_labels == c)
                if mask.any():
                    batch_proto = bag_embed[mask].mean(dim=0)
                    if self.proto_counts[c] == 0:
                        self.prototypes[c] = batch_proto
                    else:
                        self.prototypes[c] = (
                            self.proto_momentum * self.prototypes[c] +
                            (1.0 - self.proto_momentum) * batch_proto
                        )
                    self.proto_counts[c] += mask.sum()

    def prototype_sims(self, bag_embed):
        sims = []
        for c in [0, 1]:
            if self.proto_counts[c] <= 0 or torch.norm(self.prototypes[c]) < 1e-8:
                sims.append(torch.zeros(bag_embed.size(0), 1, device=bag_embed.device))
            else:
                s = F.cosine_similarity(
                    bag_embed, self.prototypes[c].unsqueeze(0), dim=-1
                ).unsqueeze(-1)
                sims.append(s)
        return torch.cat(sims, dim=-1)

    def forward(self, t1, t1ce, t2, flair, seq_mask, mod_mask, return_aux=False):
        assert t1.size(1) <= self.max_slices, "Input sequence longer than model max_slices"

        mod_mask_used = self.modality_dropout(mod_mask)
        fused, mod_weights = self.fusion([t1, t1ce, t2, flair], mod_mask_used)

        x = fused + self.pos_embed[:, :t1.size(1), :]
        x = self.encoder(x, src_key_padding_mask=~seq_mask)

        slice_logits = self.slice_head(x)
        bag_embed, attn_w, reliability = self.pool(x, slice_logits, seq_mask)
        proto_s = self.prototype_sims(bag_embed)
        bag_logits = self.final_head(torch.cat([bag_embed, proto_s], dim=-1))

        if return_aux:
            return bag_logits, slice_logits, bag_embed, mod_weights, attn_w, reliability
        return bag_logits


# ============================================================
# LOSSES / METRICS
# ============================================================

def diversity_loss(mod_weights):
    mean_w = mod_weights.mean(dim=1)
    target = torch.full_like(mean_w, 1.0 / mean_w.size(-1))
    return F.mse_loss(mean_w, target)

def temporal_smoothness_loss(slice_logits, seq_mask):
    pos_probs = torch.softmax(slice_logits, dim=-1)[..., 1]
    diffs = pos_probs[:, 1:] - pos_probs[:, :-1]
    pair_mask = seq_mask[:, 1:] & seq_mask[:, :-1]
    if pair_mask.sum() == 0:
        return slice_logits.sum() * 0.0
    return F.smooth_l1_loss(diffs[pair_mask], torch.zeros_like(diffs[pair_mask]))

def role_aware_asymmetric_focal_loss(center_logits, center_targets, role_ids, class_weights):
    prob1 = torch.softmax(center_logits, dim=-1)[:, 1].clamp(1e-6, 1 - 1e-6)
    y = center_targets.float()

    pos_w = float(class_weights[1])
    neg_w = float(class_weights[0])

    role_mult = torch.tensor(
        [CFG.ROLE_LOSS_MULTIPLIER.get(int(r.item()), 1.0) for r in role_ids],
        dtype=torch.float32,
        device=center_logits.device
    )

    pos_term = -pos_w * y * ((1.0 - prob1) ** CFG.CENTER_GAMMA_POS) * torch.log(prob1)
    neg_term = -neg_w * (1.0 - y) * role_mult * (prob1 ** CFG.CENTER_GAMMA_NEG) * torch.log(1.0 - prob1)

    loss = pos_term + neg_term
    return loss.mean()

def calculate_ece(probs, labels, n_bins=15):
    probs = np.asarray(probs)
    labels = np.asarray(labels)

    if len(labels) == 0:
        return 0.0

    conf = np.max(probs, axis=1)
    pred = np.argmax(probs, axis=1)
    acc = (pred == labels).astype(float)

    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.sum() > 0:
            ece += (m.sum() / len(labels)) * abs(acc[m].mean() - conf[m].mean())
    return float(ece)

def metrics_from_probs(probs, labels, threshold=0.5):
    probs = np.asarray(probs)
    labels = np.asarray(labels)

    if len(labels) == 0:
        out = {
            "accuracy": 0.0, "precision": 0.0, "recall": 0.0, "f1": 0.0,
            "auc": float("nan"), "pr_auc": float("nan"), "ece": 0.0,
            "balanced_accuracy": 0.0, "specificity": 0.0, "sensitivity": 0.0
        }
        return out, np.zeros((0,), dtype=np.int64)

    preds = (probs[:, 1] >= threshold).astype(int)

    out = {
        "accuracy": float(accuracy_score(labels, preds)),
        "precision": float(precision_score(labels, preds, average="macro", zero_division=0)),
        "recall": float(recall_score(labels, preds, average="macro", zero_division=0)),
        "f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "auc": safe_auc(labels, probs[:, 1]),
        "pr_auc": safe_pr_auc(labels, probs[:, 1]),
        "ece": float(calculate_ece(probs, labels)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, preds)) if len(np.unique(labels)) > 1 else 0.0,
        "specificity": float(recall_score(labels, preds, pos_label=0, zero_division=0)),
        "sensitivity": float(recall_score(labels, preds, pos_label=1, zero_division=0)),
    }
    return out, preds

def find_best_threshold(probs, labels):
    labels = np.asarray(labels)
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return 0.50, 0.0

    best_t = 0.50
    best_f1 = -1.0
    best_bal = -1.0
    best_spec = -1.0

    for t in np.arange(0.10, 0.91, 0.01):
        preds = (probs[:, 1] >= t).astype(int)
        f1 = f1_score(labels, preds, average="macro", zero_division=0)
        bal = balanced_accuracy_score(labels, preds)
        spec = recall_score(labels, preds, pos_label=0, zero_division=0)

        better = (
            (f1 > best_f1) or
            (abs(f1 - best_f1) < 1e-12 and bal > best_bal) or
            (abs(f1 - best_f1) < 1e-12 and abs(bal - best_bal) < 1e-12 and spec > best_spec)
        )
        if better:
            best_t = float(t)
            best_f1 = float(f1)
            best_bal = float(bal)
            best_spec = float(spec)

    return best_t, best_f1

def fit_temperature(logits, labels, device):
    logits = np.asarray(logits)
    labels = np.asarray(labels)
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return 1.0

    T = nn.Parameter(torch.ones(1, device=device))
    logits_t = torch.tensor(logits, dtype=torch.float32, device=device)
    labels_t = torch.tensor(labels, dtype=torch.long, device=device)

    optimizer = torch.optim.LBFGS([T], lr=0.1, max_iter=50, line_search_fn="strong_wolfe")

    def closure():
        optimizer.zero_grad()
        loss = F.cross_entropy(logits_t / T.clamp_min(0.5), labels_t)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(T.detach().clamp_min(0.5).cpu().item())


# ============================================================
# EVALUATION
# ============================================================

def evaluate_window_dataset(model, dataset, batch_size=64, temperature=1.0):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    all_logits, all_probs, all_labels = [], [], []
    all_pids, all_slice_idxs = [], []
    all_roles, all_dists, all_tumor_pixels = [], [], []

    with torch.no_grad():
        for batch in loader:
            (
                t1, t1ce, t2, flair, bag_y, center_y, seq_mask, mod_mask,
                center_pos, role_id, pid, slice_idx, distance_to_positive, tumor_pixels
            ) = batch

            t1 = t1.to(CFG.DEVICE)
            t1ce = t1ce.to(CFG.DEVICE)
            t2 = t2.to(CFG.DEVICE)
            flair = flair.to(CFG.DEVICE)
            seq_mask = seq_mask.to(CFG.DEVICE)
            mod_mask = mod_mask.to(CFG.DEVICE)
            center_pos = center_pos.to(CFG.DEVICE)

            _, slice_logits, _, _, _, _ = model(
                t1, t1ce, t2, flair, seq_mask, mod_mask, return_aux=True
            )

            center_logits = slice_logits[torch.arange(slice_logits.size(0), device=slice_logits.device), center_pos]
            center_logits = center_logits / float(temperature)
            center_probs = torch.softmax(center_logits, dim=-1)

            all_logits.append(center_logits.cpu().numpy())
            all_probs.append(center_probs.cpu().numpy())
            all_labels.append(center_y.numpy())
            all_pids.extend(list(pid))
            all_slice_idxs.append(slice_idx.numpy())
            all_roles.append(role_id.numpy())
            all_dists.append(distance_to_positive.numpy())
            all_tumor_pixels.append(tumor_pixels.numpy())

    logits = np.concatenate(all_logits, axis=0) if all_logits else np.zeros((0, 2), dtype=np.float32)
    probs = np.concatenate(all_probs, axis=0) if all_probs else np.zeros((0, 2), dtype=np.float32)
    labels = np.concatenate(all_labels, axis=0) if all_labels else np.zeros((0,), dtype=np.int64)
    pids = np.asarray(all_pids)
    slice_idxs = np.concatenate(all_slice_idxs, axis=0) if all_slice_idxs else np.zeros((0,), dtype=np.int64)
    roles = np.concatenate(all_roles, axis=0) if all_roles else np.zeros((0,), dtype=np.int64)
    dists = np.concatenate(all_dists, axis=0) if all_dists else np.zeros((0,), dtype=np.int64)
    tumor_pixels = np.concatenate(all_tumor_pixels, axis=0) if all_tumor_pixels else np.zeros((0,), dtype=np.int64)

    return logits, probs, labels, pids, slice_idxs, roles, dists, tumor_pixels

def evaluate_model(model, data_dict, batch_size=64, temperature=1.0):
    ds = CenterWindowDataset(data_dict, window_size=CFG.TRAIN_WINDOW)
    logits, probs, labels, pids, slice_idxs, _, _, _ = evaluate_window_dataset(
        model, ds, batch_size=batch_size, temperature=temperature
    )
    return logits, probs, labels, pids


# ============================================================
# TRAINING
# ============================================================

def build_train_sampler(dataset):
    weights = []
    for meta in dataset.index:
        y = int(meta["center_label"])
        role = int(meta["slice_role_id"])

        w = 1.0
        if y == 1:
            w *= CFG.SAMPLER_POSITIVE_BOOST
        elif role == SLICE_ROLE_TO_ID["boundary_neg"]:
            w *= CFG.SAMPLER_BOUNDARY_NEG_BOOST
        elif role == SLICE_ROLE_TO_ID["far_neg"]:
            w *= CFG.SAMPLER_FAR_NEG_BOOST
        elif role == SLICE_ROLE_TO_ID["no_tumor"]:
            w *= CFG.SAMPLER_NO_TUMOR_BOOST

        weights.append(w)

    weights = torch.tensor(weights, dtype=torch.double)
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)

def train_single_seed(seed, train_data, val_data, bag_weights, slice_weights):
    set_seed(seed)

    model = PUMAMILLite(
        d_in=CFG.D_IN,
        d_model=CFG.D_MODEL,
        max_slices=CFG.MAX_SLICES,
        top_k=CFG.TOP_K,
        proto_momentum=CFG.PROTO_MOMENTUM
    ).to(CFG.DEVICE)

    train_ds = CenterWindowDataset(train_data, window_size=CFG.TRAIN_WINDOW)
    val_ds = CenterWindowDataset(val_data, window_size=CFG.TRAIN_WINDOW)

    train_sampler = build_train_sampler(train_ds)

    train_dl = DataLoader(
        train_ds,
        batch_size=CFG.BATCH_SIZE,
        sampler=train_sampler,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )

    bag_class_weights = torch.tensor(
        [bag_weights[0], bag_weights[1]],
        dtype=torch.float32,
        device=CFG.DEVICE
    )
    slice_class_weights = torch.tensor(
        [slice_weights[0], slice_weights[1]],
        dtype=torch.float32,
        device=CFG.DEVICE
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.TOTAL_EPOCHS)

    ckpt_path = os.path.join(CFG.WORK_DIR, f"pumamil_best_seed{seed}.pth")

    history_rows = []
    best_score = -1.0
    best_epoch = -1
    no_improve = 0

    print("=" * 72)
    print(f"TRAINING SEED={seed}")
    print("=" * 72)
    print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

    for epoch in range(CFG.TOTAL_EPOCHS):
        model.train()

        ep_total = 0.0
        ep_center = 0.0
        ep_bag = 0.0
        ep_smooth = 0.0
        ep_div = 0.0
        n_batches = 0

        for batch in train_dl:
            (
                t1, t1ce, t2, flair, bag_y, center_y, seq_mask, mod_mask,
                center_pos, role_id, pid, slice_idx, distance_to_positive, tumor_pixels
            ) = batch

            t1 = t1.to(CFG.DEVICE, non_blocking=True)
            t1ce = t1ce.to(CFG.DEVICE, non_blocking=True)
            t2 = t2.to(CFG.DEVICE, non_blocking=True)
            flair = flair.to(CFG.DEVICE, non_blocking=True)
            bag_y = bag_y.to(CFG.DEVICE, non_blocking=True)
            center_y = center_y.to(CFG.DEVICE, non_blocking=True)
            seq_mask = seq_mask.to(CFG.DEVICE, non_blocking=True)
            mod_mask = mod_mask.to(CFG.DEVICE, non_blocking=True)
            center_pos = center_pos.to(CFG.DEVICE, non_blocking=True)
            role_id = role_id.to(CFG.DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            bag_logits, slice_logits, bag_embed, mod_weights, attn_w, reliability = model(
                t1, t1ce, t2, flair, seq_mask, mod_mask, return_aux=True
            )

            center_logits = slice_logits[torch.arange(slice_logits.size(0), device=slice_logits.device), center_pos]

            loss_center = role_aware_asymmetric_focal_loss(
                center_logits, center_y, role_id, slice_class_weights
            )

            loss_bag = F.cross_entropy(bag_logits, bag_y, weight=bag_class_weights)
            loss_smooth = temporal_smoothness_loss(slice_logits, seq_mask)
            loss_div = diversity_loss(mod_weights)

            loss = (
                CFG.CENTER_LOSS_WEIGHT * loss_center +
                CFG.BAG_LOSS_WEIGHT * loss_bag +
                CFG.SMOOTHNESS_LOSS_WEIGHT * loss_smooth +
                CFG.DIVERSITY_LOSS_WEIGHT * loss_div
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.CLIP_GRAD_NORM)
            optimizer.step()

            with torch.no_grad():
                model.update_prototypes(bag_embed.detach(), bag_y.detach())

            ep_total += float(loss.item())
            ep_center += float(loss_center.item())
            ep_bag += float(loss_bag.item())
            ep_smooth += float(loss_smooth.item())
            ep_div += float(loss_div.item())
            n_batches += 1

        scheduler.step()

        row = {
            "epoch": int(epoch + 1),
            "train_total_loss": ep_total / max(n_batches, 1),
            "train_center_loss": ep_center / max(n_batches, 1),
            "train_bag_loss": ep_bag / max(n_batches, 1),
            "train_smoothness_loss": ep_smooth / max(n_batches, 1),
            "train_diversity_loss": ep_div / max(n_batches, 1),
        }

        if ((epoch + 1) % CFG.EVAL_EVERY == 0) or (epoch == CFG.TOTAL_EPOCHS - 1):
            val_logits, val_probs, val_labels, _, _, _, _, _ = evaluate_window_dataset(
                model, val_ds, batch_size=CFG.EVAL_BATCH_SIZE, temperature=1.0
            )
            val_threshold, _ = find_best_threshold(val_probs, val_labels)
            val_metrics, _ = metrics_from_probs(val_probs, val_labels, threshold=val_threshold)

            val_score = (
                0.60 * val_metrics["f1"] +
                0.25 * val_metrics["balanced_accuracy"] +
                0.15 * val_metrics["specificity"]
            )

            row.update({
                "val_threshold": float(val_threshold),
                "val_macro_f1": float(val_metrics["f1"]),
                "val_bal_acc": float(val_metrics["balanced_accuracy"]),
                "val_specificity": float(val_metrics["specificity"]),
                "val_sensitivity": float(val_metrics["sensitivity"]),
                "val_auc": float(val_metrics["auc"]) if not np.isnan(val_metrics["auc"]) else float("nan"),
                "val_pr_auc": float(val_metrics["pr_auc"]) if not np.isnan(val_metrics["pr_auc"]) else float("nan"),
                "val_score": float(val_score),
            })

            print(
                f"Epoch {epoch+1:2d}/{CFG.TOTAL_EPOCHS} | "
                f"Loss={row['train_total_loss']:.4f} | "
                f"Val Macro-F1={row['val_macro_f1']:.4f} | "
                f"Val BalAcc={row['val_bal_acc']:.4f} | "
                f"Val Spec={row['val_specificity']:.4f} | "
                f"Val AUC={0.0 if np.isnan(row['val_auc']) else row['val_auc']:.4f} | "
                f"Val Score={row['val_score']:.4f}"
            )

            if val_score > best_score:
                best_score = float(val_score)
                best_epoch = int(epoch + 1)
                no_improve = 0
                torch.save(model.state_dict(), ckpt_path)
                print(f"  -> saved best checkpoint ({ckpt_path})")
            else:
                no_improve += 1
                if no_improve >= CFG.PATIENCE:
                    print(f"Early stopping at epoch {epoch+1}")
                    history_rows.append(row)
                    break
        else:
            row.update({
                "val_threshold": None,
                "val_macro_f1": None,
                "val_bal_acc": None,
                "val_specificity": None,
                "val_sensitivity": None,
                "val_auc": None,
                "val_pr_auc": None,
                "val_score": None,
            })

        history_rows.append(row)
        clear_memory()

    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=CFG.DEVICE))

    return model, {
        "seed": int(seed),
        "best_epoch": int(best_epoch),
        "best_val_score": float(best_score),
        "history": history_rows
    }


# ============================================================
# AUXILIARY AGGREGATION
# ============================================================

def aggregate_probs_by_patient(probs, labels, pids, slice_idxs=None, top_k=4):
    df = pd.DataFrame({
        "patient_id": pids,
        "label": labels.astype(int),
        "prob_1": probs[:, 1].astype(float)
    })
    if slice_idxs is not None:
        df["slice_idx"] = slice_idxs.astype(int)
    else:
        df["slice_idx"] = np.arange(len(df))

    rows = []
    for pid, g in df.groupby("patient_id", sort=True):
        g = g.sort_values("slice_idx")
        vals = np.sort(g["prob_1"].values)[::-1]
        k = min(int(top_k), len(vals))
        p1 = float(vals[:k].mean()) if k > 0 else 0.5
        rows.append({
            "patient_id": pid,
            "label": int(g["label"].max()),
            "prob_0": float(1.0 - p1),
            "prob_1": float(p1),
        })

    out_df = pd.DataFrame(rows)
    probs_pt = out_df[["prob_0", "prob_1"]].values.astype(np.float32)
    labels_pt = out_df["label"].values.astype(np.int64)
    pids_pt = out_df["patient_id"].values.astype(str)
    return probs_pt, labels_pt, pids_pt, out_df


# ============================================================
# PLOTTING
# ============================================================

def plot_histories(histories):
    if not histories:
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()

    for h in histories:
        df = pd.DataFrame(h["history"])
        seed = h["seed"]

        axes[0].plot(df["epoch"], df["train_total_loss"], marker="o", linewidth=1.7, label=f"Seed {seed}")
        axes[1].plot(df["epoch"], df["val_macro_f1"], marker="o", linewidth=1.7, label=f"Seed {seed}")
        axes[2].plot(df["epoch"], df["val_bal_acc"], marker="o", linewidth=1.7, label=f"Seed {seed}")
        axes[3].plot(df["epoch"], df["val_specificity"], marker="o", linewidth=1.7, label=f"Seed {seed}")

    titles = ["Training Loss", "Validation Macro-F1", "Validation Balanced Accuracy", "Validation Specificity"]
    for ax, title in zip(axes, titles):
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.grid(alpha=0.3)
        ax.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(CFG.WORK_DIR, "phase2_training_history_v3.png"), dpi=CFG.FIG_DPI, bbox_inches="tight")
    plt.close()


# ============================================================
# MAIN
# ============================================================

def run_phase2():
    phase_start = time.time()
    ensure_dir(CFG.WORK_DIR)

    print("\n" + "=" * 72)
    print("PHASE 2 (V6): REVIEWER-DRIVEN SLICE-CENTER TRAINING")
    print("=" * 72)
    print(f"Device: {CFG.DEVICE}")
    print(f"Seeds: {CFG.SEEDS}")
    print(f"Epochs: {CFG.TOTAL_EPOCHS}")
    print("=" * 72 + "\n")

    splits, train_data, val_data, tune_data, test_data = load_all_data()

    bag_weights = {
        0: float(splits["bag_class_weights"]["0"]),
        1: float(splits["bag_class_weights"]["1"]),
    }
    slice_weights = {
        0: float(splits["slice_class_weights"]["0"]),
        1: float(splits["slice_class_weights"]["1"]),
    }

    histories = []
    per_seed_rows = []

    tune_ds = CenterWindowDataset(tune_data, window_size=CFG.TRAIN_WINDOW)
    test_ds = CenterWindowDataset(test_data, window_size=CFG.TRAIN_WINDOW)

    tune_probs_all = []
    test_probs_all = []

    tune_labels_ref = None
    test_labels_ref = None
    tune_pids_ref = None
    test_pids_ref = None
    tune_slice_idxs_ref = None
    test_slice_idxs_ref = None
    tune_roles_ref = None
    test_roles_ref = None
    tune_dists_ref = None
    test_dists_ref = None
    tune_tp_ref = None
    test_tp_ref = None

    for seed in CFG.SEEDS:
        model, history = train_single_seed(seed, train_data, val_data, bag_weights, slice_weights)
        histories.append(history)

        ckpt_path = os.path.join(CFG.WORK_DIR, f"pumamil_best_seed{seed}.pth")
        model.load_state_dict(torch.load(ckpt_path, map_location=CFG.DEVICE))
        model.eval()

        tune_logits_raw, tune_probs_raw, tune_labels, tune_pids, tune_slice_idxs, tune_roles, tune_dists, tune_tp = \
            evaluate_window_dataset(model, tune_ds, batch_size=CFG.EVAL_BATCH_SIZE, temperature=1.0)
        best_T = fit_temperature(tune_logits_raw, tune_labels, CFG.DEVICE)

        tune_logits, tune_probs, tune_labels, tune_pids, tune_slice_idxs, tune_roles, tune_dists, tune_tp = \
            evaluate_window_dataset(model, tune_ds, batch_size=CFG.EVAL_BATCH_SIZE, temperature=best_T)

        test_logits, test_probs, test_labels, test_pids, test_slice_idxs, test_roles, test_dists, test_tp = \
            evaluate_window_dataset(model, test_ds, batch_size=CFG.EVAL_BATCH_SIZE, temperature=best_T)

        best_thresh, best_tune_f1 = find_best_threshold(tune_probs, tune_labels)
        test_metrics, test_preds = metrics_from_probs(test_probs, test_labels, threshold=best_thresh)

        print(f"\nSeed {seed} calibrated results")
        print(f" Temperature: {best_T:.3f}")
        print(f" Threshold: {best_thresh:.2f}")
        print(f" Test Accuracy: {test_metrics['accuracy']:.4f}")
        print(f" Test Macro-F1: {test_metrics['f1']:.4f}")
        print(f" Test Balanced Accuracy: {test_metrics['balanced_accuracy']:.4f}")
        print(f" Test Specificity: {test_metrics['specificity']:.4f}")
        print(f" Test Sensitivity: {test_metrics['sensitivity']:.4f}")
        print(f" Test PR-AUC: {0.0 if np.isnan(test_metrics['pr_auc']) else test_metrics['pr_auc']:.4f}")
        print(f" Test AUC: {0.0 if np.isnan(test_metrics['auc']) else test_metrics['auc']:.4f}")
        print(f" Test CM: {confusion_matrix(test_labels, test_preds).tolist()}")

        per_seed_rows.append({
            "seed": int(seed),
            "task_level": "slice_center",
            "best_val_score": float(history["best_val_score"]),
            "best_epoch": int(history["best_epoch"]),
            "temperature": float(best_T),
            "threshold": float(best_thresh),
            "tune_f1": float(best_tune_f1),
            "test_accuracy": float(test_metrics["accuracy"]),
            "test_precision": float(test_metrics["precision"]),
            "test_recall": float(test_metrics["recall"]),
            "test_f1": float(test_metrics["f1"]),
            "test_auc": float(test_metrics["auc"]) if not np.isnan(test_metrics["auc"]) else float("nan"),
            "test_pr_auc": float(test_metrics["pr_auc"]) if not np.isnan(test_metrics["pr_auc"]) else float("nan"),
            "test_balanced_accuracy": float(test_metrics["balanced_accuracy"]),
            "test_sensitivity": float(test_metrics["sensitivity"]),
            "test_specificity": float(test_metrics["specificity"]),
            "test_ece": float(test_metrics["ece"]),
        })

        tune_probs_all.append(tune_probs)
        test_probs_all.append(test_probs)

        if tune_labels_ref is None:
            tune_labels_ref = tune_labels
            test_labels_ref = test_labels
            tune_pids_ref = tune_pids
            test_pids_ref = test_pids
            tune_slice_idxs_ref = tune_slice_idxs
            test_slice_idxs_ref = test_slice_idxs
            tune_roles_ref = tune_roles
            test_roles_ref = test_roles
            tune_dists_ref = tune_dists
            test_dists_ref = test_dists
            tune_tp_ref = tune_tp
            test_tp_ref = test_tp
        else:
            assert np.array_equal(tune_labels_ref, tune_labels)
            assert np.array_equal(test_labels_ref, test_labels)
            assert np.array_equal(tune_pids_ref, tune_pids)
            assert np.array_equal(test_pids_ref, test_pids)
            assert np.array_equal(tune_slice_idxs_ref, tune_slice_idxs)
            assert np.array_equal(test_slice_idxs_ref, test_slice_idxs)

        del model
        clear_memory()

    metrics_df = pd.DataFrame(per_seed_rows)
    metrics_df.to_csv(os.path.join(CFG.WORK_DIR, "per_seed_test_metrics_v3.csv"), index=False)

    best_seed = int(
        metrics_df.sort_values(
            ["test_f1", "test_balanced_accuracy", "test_specificity", "best_val_score"],
            ascending=False
        ).iloc[0]["seed"]
    )

    best_ckpt_src = os.path.join(CFG.WORK_DIR, f"pumamil_best_seed{best_seed}.pth")
    best_ckpt_dst = os.path.join(CFG.WORK_DIR, "pumamil_best_v3.pth")
    shutil.copy2(best_ckpt_src, best_ckpt_dst)

    avg_tune_probs = np.mean(tune_probs_all, axis=0)
    avg_test_probs = np.mean(test_probs_all, axis=0)

    ens_thresh, ens_tune_f1 = find_best_threshold(avg_tune_probs, tune_labels_ref)
    ens_metrics, ens_preds = metrics_from_probs(avg_test_probs, test_labels_ref, threshold=ens_thresh)

    print("\n" + "=" * 72)
    print("ENSEMBLE RESULTS")
    print("=" * 72)
    print(f"Best seed by slice metrics: {best_seed}")
    print(f"Ensemble threshold: {ens_thresh:.2f}")
    print(f"Ensemble Test Accuracy: {ens_metrics['accuracy']:.4f}")
    print(f"Ensemble Test Macro-F1: {ens_metrics['f1']:.4f}")
    print(f"Ensemble Test Balanced Accuracy: {ens_metrics['balanced_accuracy']:.4f}")
    print(f"Ensemble Test Specificity: {ens_metrics['specificity']:.4f}")
    print(f"Ensemble Test Sensitivity: {ens_metrics['sensitivity']:.4f}")
    print(f"Ensemble Test PR-AUC: {0.0 if np.isnan(ens_metrics['pr_auc']) else ens_metrics['pr_auc']:.4f}")
    print(f"Ensemble Test AUC: {0.0 if np.isnan(ens_metrics['auc']) else ens_metrics['auc']:.4f}")
    print(f"Ensemble CM: {confusion_matrix(test_labels_ref, ens_preds).tolist()}")

    agg = {
        "task_level": "slice_center",
        "primary_task": "tumor_slice_identification",
        "main_metric": "macro_f1",
        "best_seed": int(best_seed),
        "per_seed_mean": {
            "test_accuracy": float(metrics_df["test_accuracy"].mean()),
            "test_precision": float(metrics_df["test_precision"].mean()),
            "test_recall": float(metrics_df["test_recall"].mean()),
            "test_f1": float(metrics_df["test_f1"].mean()),
            "test_auc": nanmean_or_zero(metrics_df["test_auc"].values),
            "test_pr_auc": nanmean_or_zero(metrics_df["test_pr_auc"].values),
            "test_balanced_accuracy": float(metrics_df["test_balanced_accuracy"].mean()),
            "test_sensitivity": float(metrics_df["test_sensitivity"].mean()),
            "test_specificity": float(metrics_df["test_specificity"].mean()),
            "test_ece": float(metrics_df["test_ece"].mean()),
        },
        "per_seed_std": {
            "test_accuracy": float(metrics_df["test_accuracy"].std()),
            "test_precision": float(metrics_df["test_precision"].std()),
            "test_recall": float(metrics_df["test_recall"].std()),
            "test_f1": float(metrics_df["test_f1"].std()),
            "test_auc": nanstd_or_zero(metrics_df["test_auc"].values),
            "test_pr_auc": nanstd_or_zero(metrics_df["test_pr_auc"].values),
            "test_balanced_accuracy": float(metrics_df["test_balanced_accuracy"].std()),
            "test_sensitivity": float(metrics_df["test_sensitivity"].std()),
            "test_specificity": float(metrics_df["test_specificity"].std()),
            "test_ece": float(metrics_df["test_ece"].std()),
        },
        "ensemble": {
            "threshold": float(ens_thresh),
            "tune_f1": float(ens_tune_f1),
            "test_accuracy": float(ens_metrics["accuracy"]),
            "test_precision": float(ens_metrics["precision"]),
            "test_recall": float(ens_metrics["recall"]),
            "test_f1": float(ens_metrics["f1"]),
            "test_auc": float(ens_metrics["auc"]) if not np.isnan(ens_metrics["auc"]) else float("nan"),
            "test_pr_auc": float(ens_metrics["pr_auc"]) if not np.isnan(ens_metrics["pr_auc"]) else float("nan"),
            "test_balanced_accuracy": float(ens_metrics["balanced_accuracy"]),
            "test_sensitivity": float(ens_metrics["sensitivity"]),
            "test_specificity": float(ens_metrics["specificity"]),
            "test_ece": float(ens_metrics["ece"]),
        }
    }

    save_json(agg, os.path.join(CFG.WORK_DIR, "aggregated_test_metrics_v3.json"))
    save_json(histories, os.path.join(CFG.WORK_DIR, "training_histories_v3.json"))

    np.savez_compressed(
        os.path.join(CFG.WORK_DIR, "ensemble_test_probs_v3.npz"),
        probs=avg_test_probs,
        labels=test_labels_ref,
        pids=test_pids_ref,
        slice_idxs=test_slice_idxs_ref,
        slice_role_id=test_roles_ref,
        distance_to_positive=test_dists_ref,
        tumor_pixels=test_tp_ref,
    )

    np.savez_compressed(
        os.path.join(CFG.WORK_DIR, "ensemble_test_slice_probs_v3.npz"),
        probs=avg_test_probs,
        labels=test_labels_ref,
        pids=test_pids_ref,
        slice_idxs=test_slice_idxs_ref,
        slice_role_id=test_roles_ref,
        distance_to_positive=test_dists_ref,
        tumor_pixels=test_tp_ref,
    )

    slice_pred_df = pd.DataFrame({
        "patient_id": test_pids_ref,
        "slice_idx": test_slice_idxs_ref.astype(int),
        "label": test_labels_ref.astype(int),
        "prob_0": avg_test_probs[:, 0].astype(float),
        "prob_1": avg_test_probs[:, 1].astype(float),
        "pred": ens_preds.astype(int),
        "slice_role_id": test_roles_ref.astype(int),
        "distance_to_positive": test_dists_ref.astype(int),
        "tumor_pixels": test_tp_ref.astype(int),
    })
    slice_pred_df.to_csv(os.path.join(CFG.WORK_DIR, "slice_predictions_v3.csv"), index=False)

    patient_probs_aux, patient_labels_aux, patient_ids_aux, patient_df_aux = aggregate_probs_by_patient(
        avg_test_probs, test_labels_ref, test_pids_ref, test_slice_idxs_ref, top_k=CFG.TOP_K
    )
    patient_preds_aux = (patient_probs_aux[:, 1] >= 0.50).astype(int)
    patient_df_aux["pred"] = patient_preds_aux.astype(int)
    patient_df_aux.to_csv(os.path.join(CFG.WORK_DIR, "patient_predictions_v3.csv"), index=False)

    aux_patient_metrics, _ = metrics_from_probs(patient_probs_aux, patient_labels_aux, threshold=0.50)
    save_json(
        {
            "note": "auxiliary compatibility artifact only; not the primary scientific endpoint",
            "accuracy": float(aux_patient_metrics["accuracy"]),
            "macro_f1": float(aux_patient_metrics["f1"]),
            "auc": float(aux_patient_metrics["auc"]) if not np.isnan(aux_patient_metrics["auc"]) else float("nan"),
            "pr_auc": float(aux_patient_metrics["pr_auc"]) if not np.isnan(aux_patient_metrics["pr_auc"]) else float("nan"),
        },
        os.path.join(CFG.WORK_DIR, "aux_patient_metrics_v3.json")
    )

    plot_histories(histories)

    phase_time = (time.time() - phase_start) / 60.0
    hyper = {
        "architecture": "PUMA-MIL-Lite",
        "phase2_version": "v6_reviewer_driven_center_windows",
        "device": CFG.DEVICE,
        "seeds": CFG.SEEDS,
        "epochs": CFG.TOTAL_EPOCHS,
        "d_model": CFG.D_MODEL,
        "max_slices": CFG.MAX_SLICES,
        "train_window": CFG.TRAIN_WINDOW,
        "top_k": CFG.TOP_K,
        "modality_dropout": CFG.MODALITY_DROPOUT,
        "center_gamma_pos": CFG.CENTER_GAMMA_POS,
        "center_gamma_neg": CFG.CENTER_GAMMA_NEG,
        "center_loss_weight": CFG.CENTER_LOSS_WEIGHT,
        "bag_loss_weight": CFG.BAG_LOSS_WEIGHT,
        "smoothness_loss_weight": CFG.SMOOTHNESS_LOSS_WEIGHT,
        "diversity_loss_weight": CFG.DIVERSITY_LOSS_WEIGHT,
        "training_time_min": round(phase_time, 2)
    }
    save_json(hyper, os.path.join(CFG.WORK_DIR, "hyperparameters_v3.json"))

    print("\n" + "=" * 72)
    print("✓ PHASE 2 COMPLETE")
    print("=" * 72)
    print(f"Wall-clock time: {phase_time:.1f} min")
    print("Generated:")
    print(" - pumamil_best_seed{42,123,456}.pth")
    print(" - pumamil_best_v3.pth")
    print(" - per_seed_test_metrics_v3.csv")
    print(" - aggregated_test_metrics_v3.json")
    print(" - training_histories_v3.json")
    print(" - phase2_training_history_v3.png")
    print(" - ensemble_test_probs_v3.npz")
    print(" - ensemble_test_slice_probs_v3.npz")
    print(" - slice_predictions_v3.csv")
    print(" - patient_predictions_v3.csv")
    print(" - aux_patient_metrics_v3.json")
    print(" - hyperparameters_v3.json")
    print("=" * 72)

if __name__ == "__main__":
    run_phase2()


In [ ]:
# ============================================================
# PHASE 3 (V8): FAIR BASELINES & ABLATIONS
# Reviewer-fixed, self-contained, and aligned with current
# Phase 1 V4-IN-PLACE + Phase 2 V6 slice-center setup
# ============================================================

import os
import gc
import json
import time
import math
import copy
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    average_precision_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================

class Config:
    PHASE1_DIR = "/kaggle/working"
    WORK_DIR = "/kaggle/working"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    SEED = 42

    # Must match Phase 1 / Phase 2 outputs
    NUM_CLASSES = 2
    D_IN = 384
    D_MODEL = 128
    MAX_SLICES = 48
    TRAIN_WINDOW = 11
    TOP_K = 4
    PROTO_MOMENTUM = 0.90

    FAST_DEBUG = False

    TOTAL_EPOCHS = 14 if FAST_DEBUG else 32
    LR = 8e-4
    WEIGHT_DECAY = 1e-4
    BATCH_SIZE = 32
    EVAL_BATCH_SIZE = 64
    NUM_WORKERS = 0
    EVAL_EVERY = 1
    PATIENCE = 4 if FAST_DEBUG else 8
    CLIP_GRAD_NORM = 1.0

    MODALITY_DROPOUT = 0.10
    FIG_DPI = 300

    CENTER_GAMMA_POS = 1.5
    CENTER_GAMMA_NEG = 3.0
    CENTER_LOSS_WEIGHT = 1.00
    BAG_LOSS_WEIGHT = 0.20
    SMOOTHNESS_LOSS_WEIGHT = 0.06
    DIVERSITY_LOSS_WEIGHT = 0.01

    SAMPLER_POSITIVE_BOOST = 1.20
    SAMPLER_BOUNDARY_NEG_BOOST = 1.45
    SAMPLER_FAR_NEG_BOOST = 1.20
    SAMPLER_NO_TUMOR_BOOST = 1.10


CFG = Config()

SLICE_ROLE_TO_ID = {
    "far_neg": 0,
    "boundary_neg": 1,
    "positive": 2,
    "mid_neg": 3,
    "no_tumor": 4,
}
ID_TO_SLICE_ROLE = {v: k for k, v in SLICE_ROLE_TO_ID.items()}

ROLE_LOSS_MULTIPLIER = {
    0: 1.25,
    1: 1.60,
    2: 1.00,
    3: 1.00,
    4: 1.15,
}


# ============================================================
# UTILS
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)


def safe_auc(labels, prob1):
    labels = np.asarray(labels)
    prob1 = np.asarray(prob1)
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return float("nan")
    try:
        return float(roc_auc_score(labels, prob1))
    except Exception:
        return float("nan")


def safe_pr_auc(labels, prob1):
    labels = np.asarray(labels)
    prob1 = np.asarray(prob1)
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return float("nan")
    try:
        return float(average_precision_score(labels, prob1))
    except Exception:
        return float("nan")


def calculate_ece(probs, labels, n_bins=15):
    probs = np.asarray(probs)
    labels = np.asarray(labels)
    if len(labels) == 0:
        return 0.0

    conf = np.max(probs, axis=1)
    pred = np.argmax(probs, axis=1)
    acc = (pred == labels).astype(float)

    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        if i == 0:
            m = (conf >= bins[i]) & (conf <= bins[i + 1])
        else:
            m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.sum() > 0:
            ece += (m.sum() / len(labels)) * abs(acc[m].mean() - conf[m].mean())
    return float(ece)


def metrics_from_probs(probs, labels, threshold=0.5):
    probs = np.asarray(probs)
    labels = np.asarray(labels)

    if len(labels) == 0:
        out = {
            "accuracy": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "auc": float("nan"),
            "pr_auc": float("nan"),
            "ece": 0.0,
            "balanced_accuracy": 0.0,
            "specificity": 0.0,
            "sensitivity": 0.0,
        }
        return out, np.zeros((0,), dtype=np.int64)

    preds = (probs[:, 1] >= threshold).astype(int)

    out = {
        "accuracy": float(accuracy_score(labels, preds)),
        "precision": float(precision_score(labels, preds, average="macro", zero_division=0)),
        "recall": float(recall_score(labels, preds, average="macro", zero_division=0)),
        "f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "auc": safe_auc(labels, probs[:, 1]),
        "pr_auc": safe_pr_auc(labels, probs[:, 1]),
        "ece": float(calculate_ece(probs, labels)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, preds)) if len(np.unique(labels)) > 1 else 0.0,
        "specificity": float(recall_score(labels, preds, pos_label=0, zero_division=0)),
        "sensitivity": float(recall_score(labels, preds, pos_label=1, zero_division=0)),
    }
    return out, preds


def find_best_threshold(probs, labels):
    probs = np.asarray(probs)
    labels = np.asarray(labels)

    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return 0.50, 0.0

    best_t = 0.50
    best_f1 = -1.0
    best_bal = -1.0
    best_spec = -1.0

    for t in np.arange(0.10, 0.91, 0.01):
        preds = (probs[:, 1] >= t).astype(int)
        f1 = f1_score(labels, preds, average="macro", zero_division=0)
        bal = balanced_accuracy_score(labels, preds)
        spec = recall_score(labels, preds, pos_label=0, zero_division=0)

        better = (
            (f1 > best_f1) or
            (abs(f1 - best_f1) < 1e-12 and bal > best_bal) or
            (abs(f1 - best_f1) < 1e-12 and abs(bal - best_bal) < 1e-12 and spec > best_spec)
        )
        if better:
            best_t = float(t)
            best_f1 = float(f1)
            best_bal = float(bal)
            best_spec = float(spec)

    return best_t, best_f1


def fit_temperature(logits, labels, device):
    logits = np.asarray(logits)
    labels = np.asarray(labels)

    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return 1.0

    T = nn.Parameter(torch.ones(1, device=device))
    logits_t = torch.tensor(logits, dtype=torch.float32, device=device)
    labels_t = torch.tensor(labels, dtype=torch.long, device=device)
    optimizer = torch.optim.LBFGS([T], lr=0.1, max_iter=50, line_search_fn="strong_wolfe")

    def closure():
        optimizer.zero_grad()
        loss = F.cross_entropy(logits_t / T.clamp_min(0.5), labels_t)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(T.detach().clamp_min(0.5).cpu().item())


def score_for_early_stop(metrics):
    return float(
        0.50 * metrics["f1"] +
        0.30 * metrics["balanced_accuracy"] +
        0.20 * metrics["specificity"]
    )


# ============================================================
# LOAD DATA
# ============================================================

def load_npz(name):
    path = os.path.join(CFG.PHASE1_DIR, f"{name}_features_4mod_v3.npz")
    with np.load(path, allow_pickle=True) as data:
        out = {k: data[k] for k in data.files}
    return out


def _zeros_if_missing(data_dict, key, shape, dtype):
    if key in data_dict:
        return data_dict[key]
    return np.zeros(shape, dtype=dtype)


def _validate_data_dict(data_dict, split_name="unknown"):
    required = [
        "features_t1", "features_t1ce", "features_t2", "features_flair",
        "labels", "patient_ids", "slice_idxs", "modality_mask",
    ]
    missing = [k for k in required if k not in data_dict]
    if missing:
        raise KeyError(f"{split_name} missing keys: {missing}")

    n = len(data_dict["labels"])
    for k in required:
        if len(data_dict[k]) != n:
            raise ValueError(f"{split_name}: key '{k}' length mismatch")


def load_data():
    with open(os.path.join(CFG.PHASE1_DIR, "patient_splits_v3.json"), "r") as f:
        splits = json.load(f)

    tr = load_npz("train")
    vl = load_npz("val")
    tu = load_npz("tune")
    te = load_npz("test")

    for name, d in [("train", tr), ("val", vl), ("tune", tu), ("test", te)]:
        _validate_data_dict(d, name)

    print("=" * 72)
    print("LOADING DATA FOR PHASE 3")
    print("=" * 72)
    print(f"Train slices: {len(tr['labels'])}")
    print(f"Val slices: {len(vl['labels'])}")
    print(f"Tune slices: {len(tu['labels'])}")
    print(f"Test slices: {len(te['labels'])}")
    print("=" * 72)
    return splits, tr, vl, tu, te


# ============================================================
# DATASET
# ============================================================

class CenterWindowDataset(Dataset):
    def __init__(self, data_dict, window_size=11):
        assert window_size % 2 == 1, "window_size must be odd"
        self.window_size = window_size
        self.half = window_size // 2

        patient_ids = np.asarray(data_dict["patient_ids"]).astype(str)
        slice_idxs = np.asarray(data_dict["slice_idxs"]).astype(np.int64)

        f_t1 = np.asarray(data_dict["features_t1"], dtype=np.float32)
        f_t1ce = np.asarray(data_dict["features_t1ce"], dtype=np.float32)
        f_t2 = np.asarray(data_dict["features_t2"], dtype=np.float32)
        f_flair = np.asarray(data_dict["features_flair"], dtype=np.float32)
        labels = np.asarray(data_dict["labels"], dtype=np.int64)
        mod_mask = np.asarray(data_dict["modality_mask"], dtype=np.int64)

        n = len(labels)
        dist = _zeros_if_missing(data_dict, "distance_to_positive", (n,), np.int64).astype(np.int64)
        role_id = _zeros_if_missing(data_dict, "slice_role_id", (n,), np.int64).astype(np.int64)
        tumor_pixels = _zeros_if_missing(data_dict, "tumor_pixels", (n,), np.int64).astype(np.int64)
        has_positive_patient = _zeros_if_missing(data_dict, "has_positive_patient", (n,), np.int64).astype(np.int64)

        self.patient_data = {}
        pid_to_indices = {}
        for i, pid in enumerate(patient_ids):
            pid_to_indices.setdefault(pid, []).append(i)

        for pid in sorted(pid_to_indices.keys()):
            idxs = np.asarray(pid_to_indices[pid], dtype=np.int64)
            order = np.argsort(slice_idxs[idxs], kind="mergesort")
            idxs = idxs[order]
            self.patient_data[pid] = {
                "t1": f_t1[idxs],
                "t1ce": f_t1ce[idxs],
                "t2": f_t2[idxs],
                "flair": f_flair[idxs],
                "labels": labels[idxs],
                "slice_idxs": slice_idxs[idxs],
                "modality_mask": mod_mask[idxs],
                "distance_to_positive": dist[idxs],
                "slice_role_id": role_id[idxs],
                "tumor_pixels": tumor_pixels[idxs],
                "has_positive_patient": has_positive_patient[idxs],
            }

        self.index = []
        for pid in sorted(self.patient_data.keys()):
            y = self.patient_data[pid]["labels"]
            roles = self.patient_data[pid]["slice_role_id"]
            sx = self.patient_data[pid]["slice_idxs"]
            dd = self.patient_data[pid]["distance_to_positive"]
            tp = self.patient_data[pid]["tumor_pixels"]
            bag_label = int(y.max() > 0)
            for local_idx in range(len(y)):
                self.index.append({
                    "pid": pid,
                    "local_idx": int(local_idx),
                    "center_label": int(y[local_idx]),
                    "bag_label": bag_label,
                    "slice_role_id": int(roles[local_idx]),
                    "slice_idx": int(sx[local_idx]),
                    "distance_to_positive": int(dd[local_idx]),
                    "tumor_pixels": int(tp[local_idx]),
                })

    def __len__(self):
        return len(self.index)

    def _build_window(self, arr, center_idx):
        out = []
        mask = []
        for offset in range(-self.half, self.half + 1):
            j = center_idx + offset
            if 0 <= j < len(arr):
                out.append(arr[j])
                mask.append(True)
            else:
                out.append(np.zeros(arr.shape[1], dtype=np.float32))
                mask.append(False)
        return np.stack(out, axis=0), np.array(mask, dtype=bool)

    def __getitem__(self, idx):
        meta = self.index[idx]
        pid = meta["pid"]
        c = meta["local_idx"]
        d = self.patient_data[pid]

        t1, seq_mask = self._build_window(d["t1"], c)
        t1ce, _ = self._build_window(d["t1ce"], c)
        t2, _ = self._build_window(d["t2"], c)
        flair, _ = self._build_window(d["flair"], c)

        valid_mm = d["modality_mask"]
        bag_mod_mask = (valid_mm.sum(axis=0) > 0).astype(np.int64)
        if bag_mod_mask.sum() == 0:
            bag_mod_mask[:] = 1

        return (
            torch.tensor(t1, dtype=torch.float32),
            torch.tensor(t1ce, dtype=torch.float32),
            torch.tensor(t2, dtype=torch.float32),
            torch.tensor(flair, dtype=torch.float32),
            torch.tensor(int(meta["bag_label"]), dtype=torch.long),
            torch.tensor(int(meta["center_label"]), dtype=torch.long),
            torch.tensor(seq_mask, dtype=torch.bool),
            torch.tensor(bag_mod_mask, dtype=torch.bool),
            torch.tensor(self.half, dtype=torch.long),
            torch.tensor(int(meta["slice_role_id"]), dtype=torch.long),
            str(pid),
            torch.tensor(int(meta["slice_idx"]), dtype=torch.long),
            torch.tensor(int(meta["distance_to_positive"]), dtype=torch.long),
            torch.tensor(int(meta["tumor_pixels"]), dtype=torch.long),
        )


def compute_sample_weights(data_dict, split_info):
    ds = CenterWindowDataset(data_dict, window_size=CFG.TRAIN_WINDOW)
    slice_weights = split_info.get("slice_class_weights", {"0": 1.0, "1": 1.0})
    weights = []
    for item in ds.index:
        y = int(item["center_label"])
        role_id = int(item["slice_role_id"])
        w = float(slice_weights.get(str(y), 1.0))
        if y == 1:
            w *= CFG.SAMPLER_POSITIVE_BOOST
        elif role_id == SLICE_ROLE_TO_ID["boundary_neg"]:
            w *= CFG.SAMPLER_BOUNDARY_NEG_BOOST
        elif role_id == SLICE_ROLE_TO_ID["far_neg"]:
            w *= CFG.SAMPLER_FAR_NEG_BOOST
        elif role_id == SLICE_ROLE_TO_ID["no_tumor"]:
            w *= CFG.SAMPLER_NO_TUMOR_BOOST
        weights.append(w)
    return ds, np.asarray(weights, dtype=np.float64)


def make_loader(data_dict, split_info=None, training=False):
    ds = CenterWindowDataset(data_dict, window_size=CFG.TRAIN_WINDOW)
    if training and split_info is not None:
        ds, weights = compute_sample_weights(data_dict, split_info)
        sampler = WeightedRandomSampler(
            weights=torch.as_tensor(weights, dtype=torch.double),
            num_samples=len(weights),
            replacement=True,
        )
        loader = DataLoader(
            ds,
            batch_size=CFG.BATCH_SIZE,
            sampler=sampler,
            num_workers=CFG.NUM_WORKERS,
            pin_memory=torch.cuda.is_available(),
        )
    else:
        loader = DataLoader(
            ds,
            batch_size=CFG.EVAL_BATCH_SIZE if not training else CFG.BATCH_SIZE,
            shuffle=False,
            num_workers=CFG.NUM_WORKERS,
            pin_memory=torch.cuda.is_available(),
        )
    return ds, loader


# ============================================================
# BASELINE FEATURES
# ============================================================

def build_baseline_features(data_dict, feature_mode="center_concat"):
    ds = CenterWindowDataset(data_dict, window_size=CFG.TRAIN_WINDOW)
    X, y, pids, slice_idxs = [], [], [], []

    for i in range(len(ds)):
        (
            t1, t1ce, t2, flair, bag_y, center_y, seq_mask, mod_mask,
            center_pos, role_id, pid, slice_idx, distance_to_positive, tumor_pixels
        ) = ds[i]

        c = int(center_pos.item())
        valid = seq_mask.numpy().astype(bool)

        if feature_mode == "center_concat":
            feat = np.concatenate(
                [t1[c].numpy(), t1ce[c].numpy(), t2[c].numpy(), flair[c].numpy()],
                axis=0,
            )
        elif feature_mode == "window_mean_concat":
            feat = np.concatenate(
                [
                    t1.numpy()[valid].mean(axis=0),
                    t1ce.numpy()[valid].mean(axis=0),
                    t2.numpy()[valid].mean(axis=0),
                    flair.numpy()[valid].mean(axis=0),
                ],
                axis=0,
            )
        else:
            raise ValueError(f"Unknown feature_mode: {feature_mode}")

        X.append(feat.astype(np.float32))
        y.append(int(center_y.item()))
        pids.append(pid)
        slice_idxs.append(int(slice_idx.item()))

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.int64),
        np.asarray(pids).astype(str),
        np.asarray(slice_idxs, dtype=np.int64),
    )


def run_baselines(tr, tu, te):
    print(f"\n{'=' * 72}")
    print("BASELINES")
    print(f"{'=' * 72}")

    baseline_specs = [
        (
            "LogReg-CenterConcat",
            "center_concat",
            Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=CFG.SEED,
                    solver="lbfgs",
                )),
            ]),
        ),
        (
            "LogReg-WindowMean",
            "window_mean_concat",
            Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=CFG.SEED,
                    solver="lbfgs",
                )),
            ]),
        ),
        (
            "MLP-WindowMean",
            "window_mean_concat",
            Pipeline([
                ("scaler", StandardScaler()),
                ("clf", MLPClassifier(
                    hidden_layer_sizes=(256, 128),
                    activation="relu",
                    alpha=1e-4,
                    batch_size=256,
                    learning_rate_init=1e-3,
                    max_iter=200,
                    early_stopping=True,
                    validation_fraction=0.15,
                    n_iter_no_change=12,
                    random_state=CFG.SEED,
                )),
            ]),
        ),
    ]

    results = {}
    pred_store = {}

    for name, mode, model in baseline_specs:
        print(f"\nRunning baseline: {name}")
        X_tr, y_tr, _, _ = build_baseline_features(tr, feature_mode=mode)
        X_tu, y_tu, p_tu, s_tu = build_baseline_features(tu, feature_mode=mode)
        X_te, y_te, p_te, s_te = build_baseline_features(te, feature_mode=mode)

        model.fit(X_tr, y_tr)
        tune_probs = model.predict_proba(X_tu).astype(np.float32)
        test_probs = model.predict_proba(X_te).astype(np.float32)

        thr, tune_f1 = find_best_threshold(tune_probs, y_tu)
        test_metrics, test_preds = metrics_from_probs(test_probs, y_te, threshold=thr)

        results[name] = {
            "family": "baseline",
            "feature_mode": mode,
            "temperature": 1.0,
            "threshold": float(thr),
            "tune_f1": float(tune_f1),
            **{k: float(v) if not np.isnan(v) else float("nan") for k, v in test_metrics.items()},
        }
        pred_store[name] = {
            "patient_id": p_te.tolist(),
            "slice_idx": s_te.astype(int).tolist(),
            "label": y_te.astype(int).tolist(),
            "pred": test_preds.astype(int).tolist(),
            "prob_1": test_probs[:, 1].astype(float).tolist(),
        }
        print(
            f" Test: F1={test_metrics['f1']:.4f}, "
            f"BalAcc={test_metrics['balanced_accuracy']:.4f}, "
            f"Spec={test_metrics['specificity']:.4f}, "
            f"AUC={0.0 if np.isnan(test_metrics['auc']) else test_metrics['auc']:.4f}"
        )

    return results, pred_store


# ============================================================
# MODEL
# ============================================================

def masked_topk_mean(seq_feats, scores, seq_mask, top_k):
    bsz, _, dim = seq_feats.shape
    out = []
    device = seq_feats.device
    dtype = seq_feats.dtype
    for b in range(bsz):
        valid_idx = torch.where(seq_mask[b])[0]
        if valid_idx.numel() == 0:
            out.append(torch.zeros(dim, device=device, dtype=dtype))
            continue
        vals = scores[b, valid_idx]
        k = min(int(top_k), int(valid_idx.numel()))
        chosen_local = torch.topk(vals, k=k, dim=0).indices
        chosen_idx = valid_idx[chosen_local]
        out.append(seq_feats[b, chosen_idx].mean(dim=0))
    return torch.stack(out, dim=0)


class ReliabilityFusion(nn.Module):
    def __init__(self, d_in=384, d_model=128):
        super().__init__()
        self.proj = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_in, d_model),
                nn.LayerNorm(d_model),
                nn.GELU(),
            ) for _ in range(4)
        ])
        self.rel = nn.ModuleList([nn.Linear(d_model, 1) for _ in range(4)])
        self.out = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
        )

    def forward(self, mods, mod_mask):
        proj_feats = []
        rel_logits = []
        for i in range(4):
            p = self.proj[i](mods[i])
            proj_feats.append(p)
            rel_logits.append(self.rel[i](p))

        feat_stack = torch.stack(proj_feats, dim=2)
        rel_stack = torch.cat(rel_logits, dim=-1)
        rel_stack = rel_stack.masked_fill(~mod_mask[:, None, :], -1e4)

        weights = torch.softmax(rel_stack, dim=-1)
        fused = (feat_stack * weights.unsqueeze(-1)).sum(dim=2)

        valid = mod_mask[:, None, :, None].float()
        mean_feat = (feat_stack * valid).sum(dim=2) / valid.sum(dim=2).clamp_min(1.0)
        out = self.out(torch.cat([fused, mean_feat], dim=-1))
        return out, weights


class EqualWeightFusion(nn.Module):
    def __init__(self, d_in=384, d_model=128):
        super().__init__()
        self.proj = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_in, d_model),
                nn.LayerNorm(d_model),
                nn.GELU(),
            ) for _ in range(4)
        ])
        self.out = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
        )

    def forward(self, mods, mod_mask):
        proj_feats = [self.proj[i](mods[i]) for i in range(4)]
        feat_stack = torch.stack(proj_feats, dim=2)
        weights = mod_mask[:, None, :].float()
        weights = weights / weights.sum(dim=-1, keepdim=True).clamp_min(1.0)
        weights = weights.expand(feat_stack.size(0), feat_stack.size(1), 4)
        fused = (feat_stack * weights.unsqueeze(-1)).sum(dim=2)
        mean_feat = fused
        out = self.out(torch.cat([fused, mean_feat], dim=-1))
        return out, weights


class HybridMILPool(nn.Module):
    def __init__(self, d_model=128, top_k=4):
        super().__init__()
        self.top_k = top_k
        self.attn = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.Tanh(),
            nn.Linear(d_model // 2, 1),
        )

    def forward(self, seq_feats, slice_logits, seq_mask):
        probs = torch.softmax(slice_logits, dim=-1)
        entropy = -(probs * torch.log(probs.clamp_min(1e-8))).sum(dim=-1) / math.log(2.0)
        reliability = 1.0 - entropy

        attn_logits = self.attn(seq_feats).squeeze(-1)
        attn_logits = attn_logits + torch.log(reliability.clamp_min(1e-6))
        attn_logits = attn_logits.masked_fill(~seq_mask, -1e9)

        attn_w = torch.softmax(attn_logits, dim=1)
        attn_pool = (seq_feats * attn_w.unsqueeze(-1)).sum(dim=1)

        pos_feat = masked_topk_mean(seq_feats, probs[..., 1], seq_mask, self.top_k)
        neg_feat = masked_topk_mean(seq_feats, probs[..., 0], seq_mask, self.top_k)
        bag_embed = torch.cat([attn_pool, pos_feat, neg_feat], dim=-1)
        return bag_embed, attn_w, reliability


class AttentionOnlyPool(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.Tanh(),
            nn.Linear(d_model // 2, 1),
        )

    def forward(self, seq_feats, slice_logits, seq_mask):
        probs = torch.softmax(slice_logits, dim=-1)
        entropy = -(probs * torch.log(probs.clamp_min(1e-8))).sum(dim=-1) / math.log(2.0)
        reliability = 1.0 - entropy

        attn_logits = self.attn(seq_feats).squeeze(-1)
        attn_logits = attn_logits + torch.log(reliability.clamp_min(1e-6))
        attn_logits = attn_logits.masked_fill(~seq_mask, -1e9)

        attn_w = torch.softmax(attn_logits, dim=1)
        attn_pool = (seq_feats * attn_w.unsqueeze(-1)).sum(dim=1)
        zeros = torch.zeros_like(attn_pool)
        bag_embed = torch.cat([attn_pool, zeros, zeros], dim=-1)
        return bag_embed, attn_w, reliability


class PUMAMILLite(nn.Module):
    def __init__(
        self,
        d_in=384,
        d_model=128,
        max_slices=48,
        top_k=4,
        proto_momentum=0.9,
        use_prototypes=True,
        allow_modality_dropout=True,
        fusion_type="reliability",
        pool_type="hybrid",
    ):
        super().__init__()
        self.d_model = d_model
        self.max_slices = max_slices
        self.top_k = top_k
        self.proto_momentum = proto_momentum
        self.use_prototypes = use_prototypes
        self.allow_modality_dropout = allow_modality_dropout

        if fusion_type == "reliability":
            self.fusion = ReliabilityFusion(d_in=d_in, d_model=d_model)
        elif fusion_type == "equal":
            self.fusion = EqualWeightFusion(d_in=d_in, d_model=d_model)
        else:
            raise ValueError(f"Unknown fusion_type: {fusion_type}")

        self.pos_embed = nn.Parameter(torch.zeros(1, max_slices, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dim_feedforward=256,
            dropout=0.10,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.slice_head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 2))

        if pool_type == "hybrid":
            self.pool = HybridMILPool(d_model=d_model, top_k=top_k)
        elif pool_type == "attention_only":
            self.pool = AttentionOnlyPool(d_model=d_model)
        else:
            raise ValueError(f"Unknown pool_type: {pool_type}")

        self.register_buffer("prototypes", torch.zeros(2, d_model * 3))
        self.register_buffer("proto_counts", torch.zeros(2))

        self.final_head = nn.Sequential(
            nn.LayerNorm(d_model * 3 + 2),
            nn.Linear(d_model * 3 + 2, d_model),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(d_model, 2),
        )
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def modality_dropout(self, mod_mask):
        if (not self.training) or (not self.allow_modality_dropout) or CFG.MODALITY_DROPOUT <= 0:
            return mod_mask
        keep = mod_mask.clone()
        rand = torch.rand_like(keep.float())
        keep = keep & (rand > CFG.MODALITY_DROPOUT)
        for i in range(keep.size(0)):
            if keep[i].sum() == 0:
                valid = torch.where(mod_mask[i])[0]
                if valid.numel() > 0:
                    j = valid[torch.randint(0, valid.numel(), (1,), device=valid.device)]
                    keep[i, j] = True
                else:
                    keep[i] = True
        return keep

    def update_prototypes(self, bag_embed, bag_labels):
        if not self.use_prototypes:
            return
        with torch.no_grad():
            for c in [0, 1]:
                mask = bag_labels == c
                if mask.any():
                    batch_proto = bag_embed[mask].mean(dim=0)
                    if self.proto_counts[c] == 0:
                        self.prototypes[c] = batch_proto
                    else:
                        self.prototypes[c] = (
                            self.proto_momentum * self.prototypes[c] +
                            (1.0 - self.proto_momentum) * batch_proto
                        )
                    self.proto_counts[c] += mask.sum()

    def prototype_sims(self, bag_embed):
        if not self.use_prototypes:
            return torch.zeros(bag_embed.size(0), 2, device=bag_embed.device)
        sims = []
        for c in [0, 1]:
            if self.proto_counts[c] <= 0 or torch.norm(self.prototypes[c]) < 1e-8:
                sims.append(torch.zeros(bag_embed.size(0), 1, device=bag_embed.device))
            else:
                s = F.cosine_similarity(bag_embed, self.prototypes[c].unsqueeze(0), dim=-1).unsqueeze(-1)
                sims.append(s)
        return torch.cat(sims, dim=-1)

    def forward(self, t1, t1ce, t2, flair, seq_mask, mod_mask, return_aux=False):
        assert t1.size(1) <= self.max_slices, "Input sequence longer than model max_slices"
        mod_mask_used = self.modality_dropout(mod_mask)
        fused, mod_weights = self.fusion([t1, t1ce, t2, flair], mod_mask_used)

        x = fused + self.pos_embed[:, :t1.size(1), :]
        x = self.encoder(x, src_key_padding_mask=~seq_mask)

        slice_logits = self.slice_head(x)
        bag_embed, attn_w, reliability = self.pool(x, slice_logits, seq_mask)
        proto_s = self.prototype_sims(bag_embed)
        bag_logits = self.final_head(torch.cat([bag_embed, proto_s], dim=-1))

        if return_aux:
            return bag_logits, slice_logits, bag_embed, mod_weights, attn_w, reliability
        return bag_logits


# ============================================================
# LOSSES / EVAL
# ============================================================

def weighted_center_focal_loss(center_logits, center_labels, role_ids, use_role_aware=True):
    log_probs = F.log_softmax(center_logits, dim=-1)
    probs = log_probs.exp()
    pt = probs.gather(1, center_labels.unsqueeze(1)).squeeze(1).clamp_min(1e-8)
    ce = F.nll_loss(log_probs, center_labels, reduction="none")

    gamma = torch.where(
        center_labels == 1,
        torch.full_like(center_labels, CFG.CENTER_GAMMA_POS, dtype=torch.float32),
        torch.full_like(center_labels, CFG.CENTER_GAMMA_NEG, dtype=torch.float32),
    )
    focal = ((1.0 - pt) ** gamma) * ce

    if use_role_aware:
        role_mult = torch.tensor(
            [ROLE_LOSS_MULTIPLIER.get(int(r), 1.0) for r in role_ids.detach().cpu().numpy()],
            dtype=torch.float32,
            device=center_logits.device,
        )
        focal = focal * role_mult

    return focal.mean()


def bag_class_weights_from_split(split_info, device):
    bag_w = split_info.get("bag_class_weights", {"0": 1.0, "1": 1.0})
    return torch.tensor([float(bag_w.get("0", 1.0)), float(bag_w.get("1", 1.0))], dtype=torch.float32, device=device)


def smoothness_loss(slice_logits, seq_mask):
    probs = torch.softmax(slice_logits, dim=-1)[..., 1]
    diffs = torch.abs(probs[:, 1:] - probs[:, :-1])
    valid_pairs = seq_mask[:, 1:] & seq_mask[:, :-1]
    if valid_pairs.any():
        return diffs[valid_pairs].mean()
    return slice_logits.new_tensor(0.0)


def modality_diversity_loss(mod_weights):
    if mod_weights.numel() == 0:
        return mod_weights.new_tensor(0.0)
    concentration = (mod_weights ** 2).sum(dim=-1)
    target = 1.0 / max(mod_weights.size(-1), 1)
    return torch.clamp(concentration - target, min=0.0).mean()


def evaluate_loader(model, loader, device, threshold=0.5, temperature=1.0):
    model.eval()
    all_logits = []
    all_probs = []
    all_labels = []
    all_pids = []
    all_slice_idxs = []

    with torch.no_grad():
        for batch in loader:
            t1, t1ce, t2, flair, bag_y, center_y, seq_mask, mod_mask, center_pos, role_id, pid, slice_idx, dist, tumor_pixels = batch
            t1 = t1.to(device)
            t1ce = t1ce.to(device)
            t2 = t2.to(device)
            flair = flair.to(device)
            seq_mask = seq_mask.to(device)
            mod_mask = mod_mask.to(device)
            center_pos = center_pos.to(device)

            bag_logits, slice_logits, bag_embed, mod_weights, attn_w, reliability = model(
                t1, t1ce, t2, flair, seq_mask, mod_mask, return_aux=True
            )
            center_logits = slice_logits[torch.arange(slice_logits.size(0), device=device), center_pos]
            center_logits = center_logits / max(float(temperature), 1e-6)
            probs = torch.softmax(center_logits, dim=-1)

            all_logits.append(center_logits.detach().cpu().numpy())
            all_probs.append(probs.detach().cpu().numpy())
            all_labels.append(center_y.numpy())
            all_pids.extend(list(pid))
            all_slice_idxs.append(slice_idx.numpy())

    logits = np.concatenate(all_logits, axis=0) if all_logits else np.zeros((0, 2), dtype=np.float32)
    probs = np.concatenate(all_probs, axis=0) if all_probs else np.zeros((0, 2), dtype=np.float32)
    labels = np.concatenate(all_labels, axis=0).astype(np.int64) if all_labels else np.zeros((0,), dtype=np.int64)
    slice_idxs = np.concatenate(all_slice_idxs, axis=0).astype(np.int64) if all_slice_idxs else np.zeros((0,), dtype=np.int64)
    metrics, preds = metrics_from_probs(probs, labels, threshold=threshold)
    return {
        "logits": logits,
        "probs": probs,
        "labels": labels,
        "preds": preds,
        "pids": np.asarray(all_pids).astype(str),
        "slice_idxs": slice_idxs,
        "metrics": metrics,
    }


def evaluate_with_calibration(model, tune_data, test_data):
    _, tune_loader = make_loader(tune_data, training=False)
    _, test_loader = make_loader(test_data, training=False)

    tune_raw = evaluate_loader(model, tune_loader, CFG.DEVICE, threshold=0.5, temperature=1.0)
    temperature = fit_temperature(tune_raw["logits"], tune_raw["labels"], CFG.DEVICE)

    tune_eval = evaluate_loader(model, tune_loader, CFG.DEVICE, threshold=0.5, temperature=temperature)
    threshold, tune_f1 = find_best_threshold(tune_eval["probs"], tune_eval["labels"])
    test_eval = evaluate_loader(model, test_loader, CFG.DEVICE, threshold=threshold, temperature=temperature)

    return {
        "temperature": float(temperature),
        "threshold": float(threshold),
        "tune_f1": float(tune_f1),
        "test_metrics": test_eval["metrics"],
        "test_probs": test_eval["probs"],
        "test_preds": test_eval["preds"],
        "test_labels": test_eval["labels"],
        "test_pids": test_eval["pids"],
        "test_slice_idxs": test_eval["slice_idxs"],
    }


# ============================================================
# TRAINING
# ============================================================

def build_model(spec):
    model = PUMAMILLite(
        d_in=CFG.D_IN,
        d_model=CFG.D_MODEL,
        max_slices=CFG.MAX_SLICES,
        top_k=CFG.TOP_K,
        proto_momentum=CFG.PROTO_MOMENTUM,
        use_prototypes=spec["use_prototypes"],
        allow_modality_dropout=spec["allow_modality_dropout"],
        fusion_type=spec["fusion_type"],
        pool_type=spec["pool_type"],
    )
    return model.to(CFG.DEVICE)


def train_variant(variant_name, train_data, val_data, split_info, spec, seed=42):
    set_seed(seed)
    clear_memory()

    model = build_model(spec)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    bag_weights = bag_class_weights_from_split(split_info, CFG.DEVICE)

    _, train_loader = make_loader(train_data, split_info=split_info, training=True)
    _, val_loader = make_loader(val_data, training=False)

    best_state = copy.deepcopy(model.state_dict())
    best_score = -1e18
    best_epoch = 0
    bad_epochs = 0
    history = []

    for epoch in range(1, CFG.TOTAL_EPOCHS + 1):
        model.train()
        loss_meter = []
        center_loss_meter = []
        bag_loss_meter = []
        smooth_loss_meter = []
        div_loss_meter = []

        for batch in train_loader:
            t1, t1ce, t2, flair, bag_y, center_y, seq_mask, mod_mask, center_pos, role_id, pid, slice_idx, dist, tumor_pixels = batch
            t1 = t1.to(CFG.DEVICE)
            t1ce = t1ce.to(CFG.DEVICE)
            t2 = t2.to(CFG.DEVICE)
            flair = flair.to(CFG.DEVICE)
            bag_y = bag_y.to(CFG.DEVICE)
            center_y = center_y.to(CFG.DEVICE)
            seq_mask = seq_mask.to(CFG.DEVICE)
            mod_mask = mod_mask.to(CFG.DEVICE)
            center_pos = center_pos.to(CFG.DEVICE)
            role_id = role_id.to(CFG.DEVICE)

            optimizer.zero_grad(set_to_none=True)
            bag_logits, slice_logits, bag_embed, mod_weights, attn_w, reliability = model(
                t1, t1ce, t2, flair, seq_mask, mod_mask, return_aux=True
            )
            center_logits = slice_logits[torch.arange(slice_logits.size(0), device=CFG.DEVICE), center_pos]

            center_loss = weighted_center_focal_loss(
                center_logits=center_logits,
                center_labels=center_y,
                role_ids=role_id,
                use_role_aware=spec["use_role_aware"],
            )

            if spec["use_bag_loss"]:
                bag_loss = F.cross_entropy(bag_logits, bag_y, weight=bag_weights)
            else:
                bag_loss = center_logits.new_tensor(0.0)

            smooth_loss_val = smoothness_loss(slice_logits, seq_mask)
            if spec["use_div_loss"]:
                div_loss = modality_diversity_loss(mod_weights)
            else:
                div_loss = center_logits.new_tensor(0.0)

            total_loss = (
                CFG.CENTER_LOSS_WEIGHT * center_loss +
                CFG.BAG_LOSS_WEIGHT * bag_loss +
                CFG.SMOOTHNESS_LOSS_WEIGHT * smooth_loss_val +
                CFG.DIVERSITY_LOSS_WEIGHT * div_loss
            )
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.CLIP_GRAD_NORM)
            optimizer.step()
            model.update_prototypes(bag_embed.detach(), bag_y)

            loss_meter.append(float(total_loss.detach().cpu().item()))
            center_loss_meter.append(float(center_loss.detach().cpu().item()))
            bag_loss_meter.append(float(bag_loss.detach().cpu().item()))
            smooth_loss_meter.append(float(smooth_loss_val.detach().cpu().item()))
            div_loss_meter.append(float(div_loss.detach().cpu().item()))

        val_eval = evaluate_loader(model, val_loader, CFG.DEVICE, threshold=0.5, temperature=1.0)
        val_metrics = val_eval["metrics"]
        val_score = score_for_early_stop(val_metrics)

        row = {
            "epoch": int(epoch),
            "train_total_loss": float(np.mean(loss_meter)) if loss_meter else 0.0,
            "train_center_loss": float(np.mean(center_loss_meter)) if center_loss_meter else 0.0,
            "train_bag_loss": float(np.mean(bag_loss_meter)) if bag_loss_meter else 0.0,
            "train_smoothness_loss": float(np.mean(smooth_loss_meter)) if smooth_loss_meter else 0.0,
            "train_diversity_loss": float(np.mean(div_loss_meter)) if div_loss_meter else 0.0,
            "val_accuracy": float(val_metrics["accuracy"]),
            "val_macro_f1": float(val_metrics["f1"]),
            "val_bal_acc": float(val_metrics["balanced_accuracy"]),
            "val_specificity": float(val_metrics["specificity"]),
            "val_auc": float(val_metrics["auc"]) if not np.isnan(val_metrics["auc"]) else float("nan"),
            "val_score": float(val_score),
        }
        history.append(row)

        print(
            f"Epoch {epoch:02d}/{CFG.TOTAL_EPOCHS} | "
            f"Loss={row['train_total_loss']:.4f} | "
            f"ValF1={row['val_macro_f1']:.4f} | "
            f"ValBal={row['val_bal_acc']:.4f} | "
            f"ValSpec={row['val_specificity']:.4f}"
        )

        if val_score > best_score + 1e-8:
            best_score = val_score
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= CFG.PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    meta_history = {
        "variant_name": variant_name,
        "best_val_score": float(best_score),
        "best_epoch": int(best_epoch),
        "history": history,
    }
    return model, meta_history


# ============================================================
# ABLATIONS
# ============================================================

def run_ablations(splits, tr, vl, tu, te):
    print(f"\n{'=' * 72}")
    print("FAIR CONTROL + ABLATIONS")
    print(f"{'=' * 72}")

    ablation_specs = [
        {
            "name": "FullControl-MatchedBudget",
            "family": "full_control",
            "fusion_type": "reliability",
            "pool_type": "hybrid",
            "use_role_aware": True,
            "use_bag_loss": True,
            "use_div_loss": True,
            "allow_modality_dropout": True,
            "use_prototypes": True,
        },
        {
            "name": "EqualWeightFusion",
            "family": "ablation",
            "fusion_type": "equal",
            "pool_type": "hybrid",
            "use_role_aware": True,
            "use_bag_loss": True,
            "use_div_loss": True,
            "allow_modality_dropout": True,
            "use_prototypes": True,
        },
        {
            "name": "AttentionOnlyPool",
            "family": "ablation",
            "fusion_type": "reliability",
            "pool_type": "attention_only",
            "use_role_aware": True,
            "use_bag_loss": True,
            "use_div_loss": True,
            "allow_modality_dropout": True,
            "use_prototypes": True,
        },
        {
            "name": "NoRoleAwareLoss",
            "family": "ablation",
            "fusion_type": "reliability",
            "pool_type": "hybrid",
            "use_role_aware": False,
            "use_bag_loss": True,
            "use_div_loss": True,
            "allow_modality_dropout": True,
            "use_prototypes": True,
        },
        {
            "name": "NoBagLoss",
            "family": "ablation",
            "fusion_type": "reliability",
            "pool_type": "hybrid",
            "use_role_aware": True,
            "use_bag_loss": False,
            "use_div_loss": True,
            "allow_modality_dropout": True,
            "use_prototypes": True,
        },
        {
            "name": "NoModalityDropout",
            "family": "ablation",
            "fusion_type": "reliability",
            "pool_type": "hybrid",
            "use_role_aware": True,
            "use_bag_loss": True,
            "use_div_loss": True,
            "allow_modality_dropout": False,
            "use_prototypes": True,
        },
        {
            "name": "NoPrototypes",
            "family": "ablation",
            "fusion_type": "reliability",
            "pool_type": "hybrid",
            "use_role_aware": True,
            "use_bag_loss": True,
            "use_div_loss": True,
            "allow_modality_dropout": True,
            "use_prototypes": False,
        },
    ]

    results = {}
    pred_store = {}
    histories = {}

    for idx, spec in enumerate(ablation_specs, start=1):
        print(f"\n[{idx}/{len(ablation_specs)}] {spec['name']}")
        model, history = train_variant(
            variant_name=spec["name"],
            train_data=tr,
            val_data=vl,
            split_info=splits,
            spec=spec,
            seed=CFG.SEED,
        )
        histories[spec["name"]] = history

        res = evaluate_with_calibration(model, tu, te)
        m = res["test_metrics"]
        results[spec["name"]] = {
            "family": spec["family"],
            "temperature": float(res["temperature"]),
            "threshold": float(res["threshold"]),
            "tune_f1": float(res["tune_f1"]),
            "best_val_score": float(history["best_val_score"]),
            "best_epoch": int(history["best_epoch"]),
            **{k: float(v) if not np.isnan(v) else float("nan") for k, v in m.items()},
        }
        pred_store[spec["name"]] = {
            "patient_id": res["test_pids"].tolist(),
            "slice_idx": res["test_slice_idxs"].astype(int).tolist(),
            "label": res["test_labels"].astype(int).tolist(),
            "pred": res["test_preds"].astype(int).tolist(),
            "prob_1": res["test_probs"][:, 1].astype(float).tolist(),
        }
        print(
            f" Test: "
            f"F1={m['f1']:.4f}, "
            f"BalAcc={m['balanced_accuracy']:.4f}, "
            f"Spec={m['specificity']:.4f}, "
            f"AUC={0.0 if np.isnan(m['auc']) else m['auc']:.4f}"
        )
        del model
        clear_memory()

    return results, pred_store, histories


# ============================================================
# PLOTS
# ============================================================

def plot_results(baseline_results, ablation_results):
    rows = []
    for name, m in baseline_results.items():
        rows.append({
            "group": "Baseline",
            "method": name,
            "f1": m["f1"],
            "balanced_accuracy": m["balanced_accuracy"],
            "specificity": m["specificity"],
        })
    for name, m in ablation_results.items():
        rows.append({
            "group": "FullControl" if name == "FullControl-MatchedBudget" else "Ablation",
            "method": name,
            "f1": m["f1"],
            "balanced_accuracy": m["balanced_accuracy"],
            "specificity": m["specificity"],
        })

    df = pd.DataFrame(rows)
    order = df.sort_values(["group", "f1"], ascending=[True, False])["method"].tolist()
    color_map = {
        "Baseline": "#1E88E5",
        "FullControl": "#43A047",
        "Ablation": "#FB8C00",
    }

    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    fig.suptitle("Phase 3: Fair Baselines and Ablations", fontsize=15, fontweight="bold")

    for ax, (col, title) in zip(
        axes,
        [("f1", "Macro-F1"), ("balanced_accuracy", "Balanced Accuracy"), ("specificity", "Specificity")],
    ):
        plot_df = df.set_index("method").loc[order].reset_index()
        colors = [color_map[g] for g in plot_df["group"]]
        ax.barh(plot_df["method"], plot_df[col], color=colors, alpha=0.88)
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel(title)
        ax.grid(axis="x", alpha=0.3)

    plt.tight_layout()
    plt.savefig(
        os.path.join(CFG.WORK_DIR, "phase3_fair_results_v8.png"),
        dpi=CFG.FIG_DPI,
        bbox_inches="tight",
    )
    plt.close()
    print("✓ Plot saved")


# ============================================================
# MAIN
# ============================================================

def run_phase3():
    start = time.time()
    ensure_dir(CFG.WORK_DIR)
    set_seed(CFG.SEED)

    print("\n" + "=" * 72)
    print("PHASE 3 (V8): FAIR BASELINES & ABLATIONS")
    print("=" * 72)
    print(f"Device: {CFG.DEVICE}")
    print(f"FAST_DEBUG: {CFG.FAST_DEBUG}")
    print(f"Epochs: {CFG.TOTAL_EPOCHS}")
    print("=" * 72)

    splits, tr, vl, tu, te = load_data()

    baseline_results, baseline_predictions = run_baselines(tr, tu, te)
    ablation_results, ablation_predictions, histories = run_ablations(splits, tr, vl, tu, te)

    all_results = {
        "baselines": baseline_results,
        "ablations": ablation_results,
    }

    protocol = {
        "phase3_version": "v8_reviewer_fixed_fair_self_contained",
        "alignment": {
            "phase1_artifacts": [
                "train_features_4mod_v3.npz",
                "val_features_4mod_v3.npz",
                "tune_features_4mod_v3.npz",
                "test_features_4mod_v3.npz",
                "patient_splits_v3.json",
            ],
            "scientific_endpoint": "slice_center",
            "tune_time_temperature_scaling": True,
            "tune_time_threshold_selection": True,
            "fair_control_rule": "Full control is retrained under the same Phase 3 budget as all ablations",
            "external_checkpoint_used_for_ablation_ranking": False,
        },
        "config": {
            "seed": CFG.SEED,
            "epochs": CFG.TOTAL_EPOCHS,
            "patience": CFG.PATIENCE,
            "d_in": CFG.D_IN,
            "d_model": CFG.D_MODEL,
            "train_window": CFG.TRAIN_WINDOW,
            "top_k": CFG.TOP_K,
            "modality_dropout": CFG.MODALITY_DROPOUT,
        },
    }

    save_json(protocol, os.path.join(CFG.WORK_DIR, "phase3_protocol_v8.json"))
    save_json(all_results, os.path.join(CFG.WORK_DIR, "phase3_results_v8.json"))
    save_json(baseline_predictions, os.path.join(CFG.WORK_DIR, "phase3_baseline_predictions_v8.json"))
    save_json(ablation_predictions, os.path.join(CFG.WORK_DIR, "phase3_ablation_predictions_v8.json"))
    save_json(histories, os.path.join(CFG.WORK_DIR, "phase3_training_histories_v8.json"))

    rows = []
    for group, res in all_results.items():
        for method, metrics in res.items():
            row = {"group": group, "method": method}
            row.update(metrics)
            rows.append(row)
    pd.DataFrame(rows).to_csv(os.path.join(CFG.WORK_DIR, "phase3_results_v8.csv"), index=False)

    plot_results(baseline_results, ablation_results)

    elapsed = (time.time() - start) / 60.0
    print(f"\n{'=' * 72}")
    print(f"✓ PHASE 3 COMPLETE ({elapsed:.1f} min)")
    print(f"{'=' * 72}")
    print("Generated:")
    print(" - phase3_protocol_v8.json")
    print(" - phase3_results_v8.json")
    print(" - phase3_results_v8.csv")
    print(" - phase3_baseline_predictions_v8.json")
    print(" - phase3_ablation_predictions_v8.json")
    print(" - phase3_training_histories_v8.json")
    print(" - phase3_fair_results_v8.png")


if __name__ == "__main__":
    run_phase3()


In [ ]:
import os
import shutil
from pathlib import Path

SRC = Path("/kaggle/input/datasets/absanim/phase-1-output")   # change only if Kaggle changes the folder name
DST = Path("/kaggle/working")

if not SRC.exists():
    print("Available input folders:")
    for p in Path("/kaggle/input").iterdir():
        print(" -", p.name)
    raise FileNotFoundError(f"Not found: {SRC}")

copied = 0
for item in SRC.iterdir():
    dst_path = DST / item.name

    if item.is_file():
        shutil.copy2(item, dst_path)
        copied += 1
    elif item.is_dir():
        if dst_path.exists():
            shutil.rmtree(dst_path)
        shutil.copytree(item, dst_path)
        copied += 1

print(f"Copied {copied} items from {SRC} to {DST}")
print("\nNow in /kaggle/working:")
for x in sorted(os.listdir(DST)):
    print(" -", x)


In [ ]:
# ============================================================
# PHASE 3B: COMPETITIVE DEEP-LEARNING BASELINES
# ABMIL [Ilse 2018], TransMIL [Shao 2021], CLAM [Lu 2021],
# ViT-Slice-MLP (single-slice ViT classifier)
# ============================================================
# Fairness guarantees:
#   - Same precomputed ViT features (frozen vit_small_patch16_224)
#   - Same patient-level stratified splits (train/val/tune/test)
#   - Same CenterWindowDataset with W=11, center-slice labels
#   - CLASS-BALANCED sampling (no role-aware boosts -- those are
#     PUMA-MIL's contribution, giving them to baselines would be
#     an unearned advantage)
#   - Same evaluation: temperature scaling on tune, threshold
#     tuning, metrics_from_probs on test, ECE with 5 bins
#   - Same model selection: maximum validation macro-F1
#   - Same optimizer (AdamW), lr=8e-4, early stopping patience 8
#   - Same training budget (32 epochs max)
#   - 3-SEED ENSEMBLE {42, 123, 456} with probability averaging
#     (identical to PUMA-MIL ensembling protocol)
# ============================================================

import os
import gc
import json
import time
import math
import copy
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    average_precision_score,
    confusion_matrix,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG  (identical to Phase 3)
# ============================================================

class Config:
    PHASE1_DIR = "/kaggle/working"
    WORK_DIR = "/kaggle/working"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    SEEDS = [42, 123, 456]   # same 3 seeds as PUMA-MIL ensemble

    NUM_CLASSES = 2
    D_IN = 384          # ViT-Small CLS token dimension
    D_MODEL = 128       # projection dimension (same as PUMA-MIL)
    MAX_SLICES = 48
    TRAIN_WINDOW = 11
    TOP_K = 4

    FAST_DEBUG = False

    TOTAL_EPOCHS = 14 if FAST_DEBUG else 32
    LR = 8e-4
    WEIGHT_DECAY = 1e-4
    BATCH_SIZE = 32
    EVAL_BATCH_SIZE = 64
    NUM_WORKERS = 0
    PATIENCE = 4 if FAST_DEBUG else 8
    CLIP_GRAD_NORM = 1.0

    FIG_DPI = 300

    # NOTE: Role-aware sampler boosts are PUMA-MIL's contribution.
    # Baselines use plain class-balanced sampling only.


CFG = Config()

SLICE_ROLE_TO_ID = {
    "far_neg": 0,
    "boundary_neg": 1,
    "positive": 2,
    "mid_neg": 3,
    "no_tumor": 4,
}


# ============================================================
# UTILITIES  (copied from Phase 3 for self-containment)
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)


def safe_auc(labels, prob1):
    labels, prob1 = np.asarray(labels), np.asarray(prob1)
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return float("nan")
    try:
        return float(roc_auc_score(labels, prob1))
    except Exception:
        return float("nan")


def safe_pr_auc(labels, prob1):
    labels, prob1 = np.asarray(labels), np.asarray(prob1)
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return float("nan")
    try:
        return float(average_precision_score(labels, prob1))
    except Exception:
        return float("nan")


def calculate_ece(probs, labels, n_bins=5):
    probs, labels = np.asarray(probs), np.asarray(labels)
    if len(labels) == 0:
        return 0.0
    conf = np.max(probs, axis=1)
    pred = np.argmax(probs, axis=1)
    acc = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        m = (conf >= bins[i]) & (conf <= bins[i + 1]) if i == 0 else (conf > bins[i]) & (conf <= bins[i + 1])
        if m.sum() > 0:
            ece += (m.sum() / len(labels)) * abs(acc[m].mean() - conf[m].mean())
    return float(ece)


def metrics_from_probs(probs, labels, threshold=0.5):
    probs, labels = np.asarray(probs), np.asarray(labels)
    if len(labels) == 0:
        return {k: 0.0 for k in ["accuracy","precision","recall","f1","auc","pr_auc","ece","balanced_accuracy","specificity","sensitivity"]}, np.zeros(0, dtype=np.int64)
    preds = (probs[:, 1] >= threshold).astype(int)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "precision": float(precision_score(labels, preds, average="macro", zero_division=0)),
        "recall": float(recall_score(labels, preds, average="macro", zero_division=0)),
        "f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "auc": safe_auc(labels, probs[:, 1]),
        "pr_auc": safe_pr_auc(labels, probs[:, 1]),
        "ece": float(calculate_ece(probs, labels)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, preds)) if len(np.unique(labels)) > 1 else 0.0,
        "specificity": float(recall_score(labels, preds, pos_label=0, zero_division=0)),
        "sensitivity": float(recall_score(labels, preds, pos_label=1, zero_division=0)),
    }, preds


def find_best_threshold(probs, labels):
    probs, labels = np.asarray(probs), np.asarray(labels)
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return 0.50, 0.0
    best_t, best_f1 = 0.50, -1.0
    for t in np.arange(0.10, 0.91, 0.01):
        preds = (probs[:, 1] >= t).astype(int)
        f1 = f1_score(labels, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_t, best_f1 = float(t), float(f1)
    return best_t, best_f1


def fit_temperature(logits, labels, device):
    logits, labels = np.asarray(logits), np.asarray(labels)
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return 1.0
    T = nn.Parameter(torch.ones(1, device=device))
    logits_t = torch.tensor(logits, dtype=torch.float32, device=device)
    labels_t = torch.tensor(labels, dtype=torch.long, device=device)
    optimizer = torch.optim.LBFGS([T], lr=0.1, max_iter=50, line_search_fn="strong_wolfe")
    def closure():
        optimizer.zero_grad()
        loss = F.cross_entropy(logits_t / T.clamp_min(0.5), labels_t)
        loss.backward()
        return loss
    optimizer.step(closure)
    return float(T.detach().clamp_min(0.5).cpu().item())


# ============================================================
# DATA LOADING  (identical to Phase 3)
# ============================================================

def load_npz(name):
    path = os.path.join(CFG.PHASE1_DIR, f"{name}_features_4mod_v3.npz")
    with np.load(path, allow_pickle=True) as data:
        return {k: data[k] for k in data.files}


def _zeros_if_missing(data_dict, key, shape, dtype):
    return data_dict[key] if key in data_dict else np.zeros(shape, dtype=dtype)


class CenterWindowDataset(Dataset):
    """Identical to Phase 3 CenterWindowDataset for fairness."""
    def __init__(self, data_dict, window_size=11):
        assert window_size % 2 == 1
        self.window_size = window_size
        self.half = window_size // 2

        patient_ids = np.asarray(data_dict["patient_ids"]).astype(str)
        slice_idxs = np.asarray(data_dict["slice_idxs"]).astype(np.int64)
        f_t1 = np.asarray(data_dict["features_t1"], dtype=np.float32)
        f_t1ce = np.asarray(data_dict["features_t1ce"], dtype=np.float32)
        f_t2 = np.asarray(data_dict["features_t2"], dtype=np.float32)
        f_flair = np.asarray(data_dict["features_flair"], dtype=np.float32)
        labels = np.asarray(data_dict["labels"], dtype=np.int64)
        mod_mask = np.asarray(data_dict["modality_mask"], dtype=np.int64)
        n = len(labels)
        dist = _zeros_if_missing(data_dict, "distance_to_positive", (n,), np.int64).astype(np.int64)
        role_id = _zeros_if_missing(data_dict, "slice_role_id", (n,), np.int64).astype(np.int64)

        self.patient_data = {}
        pid_to_idx = {}
        for i, pid in enumerate(patient_ids):
            pid_to_idx.setdefault(pid, []).append(i)
        for pid in sorted(pid_to_idx):
            idxs = np.asarray(pid_to_idx[pid], dtype=np.int64)
            order = np.argsort(slice_idxs[idxs], kind="mergesort")
            idxs = idxs[order]
            self.patient_data[pid] = {
                "t1": f_t1[idxs], "t1ce": f_t1ce[idxs],
                "t2": f_t2[idxs], "flair": f_flair[idxs],
                "labels": labels[idxs], "slice_idxs": slice_idxs[idxs],
                "modality_mask": mod_mask[idxs],
                "distance_to_positive": dist[idxs],
                "slice_role_id": role_id[idxs],
            }

        self.index = []
        for pid in sorted(self.patient_data):
            y = self.patient_data[pid]["labels"]
            roles = self.patient_data[pid]["slice_role_id"]
            sx = self.patient_data[pid]["slice_idxs"]
            bag_label = int(y.max() > 0)
            for local_idx in range(len(y)):
                self.index.append({
                    "pid": pid, "local_idx": int(local_idx),
                    "center_label": int(y[local_idx]),
                    "bag_label": bag_label,
                    "slice_role_id": int(roles[local_idx]),
                    "slice_idx": int(sx[local_idx]),
                })

    def __len__(self):
        return len(self.index)

    def _build_window(self, arr, center_idx):
        out, mask = [], []
        for offset in range(-self.half, self.half + 1):
            j = center_idx + offset
            if 0 <= j < len(arr):
                out.append(arr[j]); mask.append(True)
            else:
                out.append(np.zeros(arr.shape[1], dtype=np.float32)); mask.append(False)
        return np.stack(out, axis=0), np.array(mask, dtype=bool)

    def __getitem__(self, idx):
        meta = self.index[idx]
        pid, c = meta["pid"], meta["local_idx"]
        d = self.patient_data[pid]
        t1, seq_mask = self._build_window(d["t1"], c)
        t1ce, _ = self._build_window(d["t1ce"], c)
        t2, _ = self._build_window(d["t2"], c)
        flair, _ = self._build_window(d["flair"], c)
        valid_mm = d["modality_mask"]
        bag_mod_mask = (valid_mm.sum(axis=0) > 0).astype(np.int64)
        if bag_mod_mask.sum() == 0:
            bag_mod_mask[:] = 1
        return (
            torch.tensor(t1, dtype=torch.float32),
            torch.tensor(t1ce, dtype=torch.float32),
            torch.tensor(t2, dtype=torch.float32),
            torch.tensor(flair, dtype=torch.float32),
            torch.tensor(int(meta["bag_label"]), dtype=torch.long),
            torch.tensor(int(meta["center_label"]), dtype=torch.long),
            torch.tensor(seq_mask, dtype=torch.bool),
            torch.tensor(bag_mod_mask, dtype=torch.bool),
            torch.tensor(self.half, dtype=torch.long),
            torch.tensor(int(meta["slice_role_id"]), dtype=torch.long),
            str(pid),
            torch.tensor(int(meta["slice_idx"]), dtype=torch.long),
        )


def compute_class_balanced_weights(ds):
    """Plain class-balanced sampling: only correct for class imbalance.
    No role-aware boosts (those are PUMA-MIL's contribution).
    """
    labels = np.array([int(item["center_label"]) for item in ds.index])
    n0, n1 = (labels == 0).sum(), (labels == 1).sum()
    n_total = len(labels)
    w0 = n_total / (2.0 * max(n0, 1))
    w1 = n_total / (2.0 * max(n1, 1))
    weights = np.where(labels == 1, w1, w0).astype(np.float64)
    return weights


def make_loader(data_dict, training=False):
    ds = CenterWindowDataset(data_dict, window_size=CFG.TRAIN_WINDOW)
    if training:
        weights = compute_class_balanced_weights(ds)
        sampler = WeightedRandomSampler(
            weights=torch.as_tensor(weights, dtype=torch.double),
            num_samples=len(weights), replacement=True,
        )
        loader = DataLoader(ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                            num_workers=CFG.NUM_WORKERS, pin_memory=torch.cuda.is_available())
    else:
        loader = DataLoader(ds, batch_size=CFG.EVAL_BATCH_SIZE, shuffle=False,
                            num_workers=CFG.NUM_WORKERS, pin_memory=torch.cuda.is_available())
    return ds, loader


def load_data():
    with open(os.path.join(CFG.PHASE1_DIR, "patient_splits_v3.json"), "r") as f:
        splits = json.load(f)
    tr = load_npz("train"); vl = load_npz("val")
    tu = load_npz("tune"); te = load_npz("test")
    for name, d in [("train", tr), ("val", vl), ("tune", tu), ("test", te)]:
        required = ["features_t1","features_t1ce","features_t2","features_flair","labels","patient_ids","slice_idxs","modality_mask"]
        missing = [k for k in required if k not in d]
        if missing:
            raise KeyError(f"{name} missing: {missing}")
    print(f"Train: {len(tr['labels'])}, Val: {len(vl['labels'])}, "
          f"Tune: {len(tu['labels'])}, Test: {len(te['labels'])}")
    return splits, tr, vl, tu, te


# ============================================================
# SIMPLE FUSION: concat 4 modalities -> project to D_MODEL
# All competitive baselines use the same input representation
# ============================================================

class SimpleModalityFusion(nn.Module):
    """Fuse 4 modality features by concatenation + projection.
    Input:  4 tensors of shape (B, W, 384) -> Output: (B, W, D_MODEL)
    """
    def __init__(self, d_in=384, d_model=128):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(d_in * 4, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
        )

    def forward(self, t1, t1ce, t2, flair, seq_mask):
        # Concatenate all 4 modalities along feature dim
        x = torch.cat([t1, t1ce, t2, flair], dim=-1)  # (B, W, 1536)
        return self.proj(x)  # (B, W, d_model)


# ============================================================
# MODEL 1: ABMIL  (Ilse et al., ICML 2018)
# Attention-Based MIL with gated attention mechanism
# Applied to the 11-slice window: center-slice classification
# ============================================================

class ABMIL(nn.Module):
    """Attention-Based MIL (Ilse et al. 2018).
    Uses gated attention on the 11-slice window, then classifies
    the attention-pooled bag representation.
    """
    def __init__(self, d_in=384, d_model=128, d_attn=64):
        super().__init__()
        self.fusion = SimpleModalityFusion(d_in, d_model)
        # Gated attention mechanism (Eq. 9 in Ilse et al.)
        self.attn_V = nn.Sequential(nn.Linear(d_model, d_attn), nn.Tanh())
        self.attn_U = nn.Sequential(nn.Linear(d_model, d_attn), nn.Sigmoid())
        self.attn_w = nn.Linear(d_attn, 1)
        # Classification head
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(d_model // 2, 2),
        )

    def forward(self, t1, t1ce, t2, flair, seq_mask, **kwargs):
        h = self.fusion(t1, t1ce, t2, flair, seq_mask)  # (B, W, d_model)
        # Gated attention
        a_V = self.attn_V(h)   # (B, W, d_attn)
        a_U = self.attn_U(h)   # (B, W, d_attn)
        a = self.attn_w(a_V * a_U).squeeze(-1)  # (B, W)
        a = a.masked_fill(~seq_mask, -1e9)
        a = torch.softmax(a, dim=1)  # (B, W)
        # Attention pooling
        bag = (h * a.unsqueeze(-1)).sum(dim=1)  # (B, d_model)
        logits = self.classifier(bag)  # (B, 2)
        return logits


# ============================================================
# MODEL 2: TransMIL  (Shao et al., NeurIPS 2021)
# Transformer-based MIL with PPEG and pyramid positions
# Simplified to match our feature setting (precomputed ViT features)
# ============================================================

class PPEG(nn.Module):
    """Pyramid Position Encoding Generator (from TransMIL).
    Adapted: since our instances are 1D sequence (slices), we use
    1D convolutions instead of 2D.
    """
    def __init__(self, d_model=128):
        super().__init__()
        self.proj = nn.Linear(d_model, d_model)
        self.conv1 = nn.Conv1d(d_model, d_model, 3, padding=1, groups=d_model)
        self.conv2 = nn.Conv1d(d_model, d_model, 5, padding=2, groups=d_model)
        self.conv3 = nn.Conv1d(d_model, d_model, 7, padding=3, groups=d_model)

    def forward(self, x):
        # x: (B, W, d_model)
        cls_token = x[:, :1, :]
        feat = x[:, 1:, :]  # skip CLS for position encoding
        xt = feat.transpose(1, 2)  # (B, d_model, W-1)
        x1 = self.conv1(xt).transpose(1, 2)
        x2 = self.conv2(xt).transpose(1, 2)
        x3 = self.conv3(xt).transpose(1, 2)
        feat = feat + x1 + x2 + x3
        return torch.cat([cls_token, feat], dim=1)


class TransMIL(nn.Module):
    """TransMIL (Shao et al., NeurIPS 2021).
    CLS token + Transformer encoder + PPEG for positional info.
    Applied on the 11-slice window with 4-modality fusion.
    """
    def __init__(self, d_in=384, d_model=128, n_heads=4, n_layers=2):
        super().__init__()
        self.fusion = SimpleModalityFusion(d_in, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.ppeg = PPEG(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 2,
            dropout=0.10, batch_first=True, activation="gelu",
        )
        self.encoder1 = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(d_model // 2, 2),
        )

    def forward(self, t1, t1ce, t2, flair, seq_mask, **kwargs):
        h = self.fusion(t1, t1ce, t2, flair, seq_mask)  # (B, W, d_model)
        B = h.size(0)
        # Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)
        h = torch.cat([cls, h], dim=1)  # (B, W+1, d_model)
        # Extend mask for CLS
        cls_mask = torch.ones(B, 1, dtype=torch.bool, device=h.device)
        full_mask = torch.cat([cls_mask, seq_mask], dim=1)
        # Transformer + PPEG
        h = self.encoder1(h, src_key_padding_mask=~full_mask)
        h = self.ppeg(h)
        h = self.encoder1(h, src_key_padding_mask=~full_mask)
        h = self.norm(h)
        # Use CLS token for classification
        logits = self.classifier(h[:, 0, :])
        return logits


# ============================================================
# MODEL 3: CLAM  (Lu et al., Nature BME 2021)
# Clustering-constrained Attention MIL
# Single-branch version (CLAM-SB) adapted for our setting
# ============================================================

class CLAM_SB(nn.Module):
    """CLAM Single-Branch (Lu et al. 2021).
    Gated attention + instance-level clustering loss (auxiliary).
    We expose the instance loss to the training loop.
    """
    def __init__(self, d_in=384, d_model=128, d_attn=64, n_clusters=8):
        super().__init__()
        self.fusion = SimpleModalityFusion(d_in, d_model)
        # Attention (gated)
        self.attn_V = nn.Sequential(nn.Linear(d_model, d_attn), nn.Tanh())
        self.attn_U = nn.Sequential(nn.Linear(d_model, d_attn), nn.Sigmoid())
        self.attn_w = nn.Linear(d_attn, 1)
        # Instance classifier for top-k (clustering constraint)
        self.inst_classifier = nn.Sequential(
            nn.Linear(d_model, 2),
        )
        self.n_clusters = n_clusters
        # Bag classifier
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(d_model // 2, 2),
        )

    def forward(self, t1, t1ce, t2, flair, seq_mask, return_inst_loss=False, center_label=None, **kwargs):
        h = self.fusion(t1, t1ce, t2, flair, seq_mask)  # (B, W, d_model)
        # Gated attention
        a_V = self.attn_V(h)
        a_U = self.attn_U(h)
        a = self.attn_w(a_V * a_U).squeeze(-1)
        a = a.masked_fill(~seq_mask, -1e9)
        a_weights = torch.softmax(a, dim=1)
        # Attention pooling
        bag = (h * a_weights.unsqueeze(-1)).sum(dim=1)
        logits = self.classifier(bag)

        inst_loss = torch.tensor(0.0, device=h.device)
        if return_inst_loss and center_label is not None:
            # Instance-level clustering constraint:
            # For positive bags, top-attended instances should be positive
            # For negative bags, top-attended instances should be negative
            for b_idx in range(h.size(0)):
                valid = seq_mask[b_idx]
                if valid.sum() == 0:
                    continue
                valid_h = h[b_idx, valid]      # (n_valid, d_model)
                valid_a = a_weights[b_idx, valid]  # (n_valid,)
                k = min(self.n_clusters, valid_h.size(0))
                topk_idx = torch.topk(valid_a, k).indices
                inst_logits = self.inst_classifier(valid_h[topk_idx])
                inst_labels = torch.full((k,), int(center_label[b_idx].item()),
                                         dtype=torch.long, device=h.device)
                inst_loss = inst_loss + F.cross_entropy(inst_logits, inst_labels)
            inst_loss = inst_loss / max(h.size(0), 1)

        if return_inst_loss:
            return logits, inst_loss
        return logits


# ============================================================
# MODEL 4: ViT-Slice-MLP (single-slice deep classifier)
# Strong single-slice baseline using the same ViT features
# ============================================================

class ViTSliceMLP(nn.Module):
    """Deep MLP classifier on center-slice ViT features.
    Uses only the center slice (no window context) with
    4-modality concatenation and a 3-layer MLP.
    This represents a 'best single-slice ViT classifier'.
    """
    def __init__(self, d_in=384, d_hidden=256):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(d_in * 4, d_hidden),
            nn.LayerNorm(d_hidden),
            nn.GELU(),
            nn.Dropout(0.30),
            nn.Linear(d_hidden, d_hidden // 2),
            nn.LayerNorm(d_hidden // 2),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(d_hidden // 2, 2),
        )

    def forward(self, t1, t1ce, t2, flair, seq_mask, center_pos=None, **kwargs):
        # Use only center slice
        if center_pos is None:
            c = t1.size(1) // 2
        else:
            c = center_pos[0].item()  # same for all in batch
        x = torch.cat([t1[:, c], t1ce[:, c], t2[:, c], flair[:, c]], dim=-1)
        return self.classifier(x)


# ============================================================
# TRAINING AND EVALUATION LOOP
# ============================================================

def evaluate_model(model, loader, device, threshold=0.5, temperature=1.0, model_type="bag"):
    model.eval()
    all_logits, all_probs, all_labels, all_pids, all_slice_idxs = [], [], [], [], []

    with torch.no_grad():
        for batch in loader:
            t1, t1ce, t2, flair = batch[0].to(device), batch[1].to(device), batch[2].to(device), batch[3].to(device)
            center_y = batch[5]
            seq_mask = batch[6].to(device)
            center_pos = batch[8].to(device)
            pid = batch[10]
            slice_idx = batch[11]

            logits = model(t1, t1ce, t2, flair, seq_mask, center_pos=center_pos)
            logits = logits / max(float(temperature), 1e-6)
            probs = torch.softmax(logits, dim=-1)

            all_logits.append(logits.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
            all_labels.append(center_y.numpy())
            all_pids.extend(list(pid))
            all_slice_idxs.append(slice_idx.numpy())

    logits = np.concatenate(all_logits, axis=0)
    probs = np.concatenate(all_probs, axis=0)
    labels = np.concatenate(all_labels, axis=0).astype(np.int64)
    slice_idxs = np.concatenate(all_slice_idxs, axis=0).astype(np.int64)
    metrics, preds = metrics_from_probs(probs, labels, threshold=threshold)

    return {
        "logits": logits, "probs": probs, "labels": labels, "preds": preds,
        "pids": np.asarray(all_pids).astype(str), "slice_idxs": slice_idxs,
        "metrics": metrics,
    }


def train_single_seed(
    name, model_cls, model_kwargs, train_data, val_data,
    seed=42, use_inst_loss=False
):
    """Train one seed of a competitive baseline.
    Model selection uses pure validation macro-F1 (same as PUMA-MIL).
    """
    set_seed(seed)
    clear_memory()
    model = model_cls(**model_kwargs).to(CFG.DEVICE)

    param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Seed {seed} | Params: {param_count:,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)

    _, train_loader = make_loader(train_data, training=True)
    _, val_loader = make_loader(val_data, training=False)

    # Compute class weights for CE loss
    train_labels = np.asarray(train_data["labels"])
    n0, n1 = (train_labels == 0).sum(), (train_labels == 1).sum()
    ce_weight = torch.tensor([len(train_labels) / (2.0 * max(n0, 1)),
                              len(train_labels) / (2.0 * max(n1, 1))],
                             dtype=torch.float32, device=CFG.DEVICE)

    best_state = copy.deepcopy(model.state_dict())
    best_f1 = -1.0     # FIX #2: pure macro-F1 for model selection
    best_epoch = 0
    bad_epochs = 0
    history = []

    t_start = time.time()

    for epoch in range(1, CFG.TOTAL_EPOCHS + 1):
        model.train()
        loss_meter = []

        for batch in train_loader:
            t1 = batch[0].to(CFG.DEVICE)
            t1ce = batch[1].to(CFG.DEVICE)
            t2 = batch[2].to(CFG.DEVICE)
            flair = batch[3].to(CFG.DEVICE)
            center_y = batch[5].to(CFG.DEVICE)
            seq_mask = batch[6].to(CFG.DEVICE)
            center_pos = batch[8].to(CFG.DEVICE)

            optimizer.zero_grad(set_to_none=True)

            if use_inst_loss:
                logits, inst_loss = model(
                    t1, t1ce, t2, flair, seq_mask,
                    return_inst_loss=True, center_label=center_y, center_pos=center_pos,
                )
                ce_loss = F.cross_entropy(logits, center_y, weight=ce_weight)
                loss = ce_loss + 0.3 * inst_loss
            else:
                logits = model(t1, t1ce, t2, flair, seq_mask, center_pos=center_pos)
                loss = F.cross_entropy(logits, center_y, weight=ce_weight)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.CLIP_GRAD_NORM)
            optimizer.step()
            loss_meter.append(loss.item())

        # Validation — FIX #2: select by pure macro-F1
        val_result = evaluate_model(model, val_loader, CFG.DEVICE)
        vm = val_result["metrics"]
        val_f1 = vm["f1"]

        history.append({
            "epoch": epoch,
            "train_loss": float(np.mean(loss_meter)),
            "val_f1": vm["f1"], "val_auc": vm["auc"],
            "val_acc": vm["accuracy"], "val_spec": vm["specificity"],
            "val_sens": vm["sensitivity"],
        })

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch % 4 == 0 or epoch == 1:
            print(f"    Ep {epoch:3d}  loss={np.mean(loss_meter):.4f}  "
                  f"val_F1={vm['f1']:.4f}  val_AUC={vm['auc']:.4f}  "
                  f"{'*BEST*' if bad_epochs == 0 else ''}")

        if bad_epochs >= CFG.PATIENCE:
            print(f"    Early stop at epoch {epoch}")
            break

    elapsed = time.time() - t_start
    print(f"    Best epoch: {best_epoch}  ({elapsed:.1f}s)")

    model.load_state_dict(best_state)
    model.eval()
    return model, param_count, elapsed, history


def train_competitive_baseline(
    name, model_cls, model_kwargs, train_data, val_data, tune_data, test_data,
    use_inst_loss=False
):
    """Train a 3-seed ensemble of a competitive baseline.
    FIX #1: Identical ensembling protocol to PUMA-MIL.
    """
    print(f"\n{'='*72}")
    print(f"TRAINING 3-SEED ENSEMBLE: {name}")
    print(f"{'='*72}")

    seed_models = []
    all_histories = {}
    total_time = 0.0
    param_count = 0

    for seed in CFG.SEEDS:
        print(f"\n  --- Seed {seed} ---")
        model, pc, elapsed, hist = train_single_seed(
            name, model_cls, model_kwargs,
            train_data, val_data, seed=seed,
            use_inst_loss=use_inst_loss,
        )
        seed_models.append(model)
        all_histories[f"seed_{seed}"] = hist
        total_time += elapsed
        param_count = pc

    # ---- Per-seed temperature scaling + threshold on tune ----
    _, tune_loader = make_loader(tune_data, training=False)
    _, test_loader = make_loader(test_data, training=False)

    seed_temperatures = []
    seed_thresholds = []
    seed_test_probs = []
    seed_test_metrics = []

    for i, (seed, model) in enumerate(zip(CFG.SEEDS, seed_models)):
        # Temperature scaling
        tune_raw = evaluate_model(model, tune_loader, CFG.DEVICE, threshold=0.5, temperature=1.0)
        temp = fit_temperature(tune_raw["logits"], tune_raw["labels"], CFG.DEVICE)
        seed_temperatures.append(temp)

        # Threshold tuning with temperature
        tune_cal = evaluate_model(model, tune_loader, CFG.DEVICE, threshold=0.5, temperature=temp)
        thr, tune_f1 = find_best_threshold(tune_cal["probs"], tune_cal["labels"])
        seed_thresholds.append(thr)

        # Per-seed test
        test_res = evaluate_model(model, test_loader, CFG.DEVICE, threshold=thr, temperature=temp)
        seed_test_probs.append(test_res["probs"])
        seed_test_metrics.append(test_res["metrics"])
        tm = test_res["metrics"]
        print(f"  Seed {seed}: F1={tm['f1']:.4f}  AUC={tm['auc']:.4f}  "
              f"Spec={tm['specificity']:.4f}  Sens={tm['sensitivity']:.4f}")

    # ---- Ensemble: average softmax probabilities ----
    ensemble_probs = np.mean(seed_test_probs, axis=0)
    # Tune ensemble threshold on tune set
    tune_ensemble_probs = []
    for model, temp in zip(seed_models, seed_temperatures):
        tune_res = evaluate_model(model, tune_loader, CFG.DEVICE, threshold=0.5, temperature=temp)
        tune_ensemble_probs.append(tune_res["probs"])
    tune_ens_probs = np.mean(tune_ensemble_probs, axis=0)
    tune_labels = tune_res["labels"]
    ens_thr, ens_tune_f1 = find_best_threshold(tune_ens_probs, tune_labels)

    # Final ensemble test metrics
    test_labels = test_res["labels"]  # same for all seeds
    ens_metrics, ens_preds = metrics_from_probs(ensemble_probs, test_labels, threshold=ens_thr)

    print(f"\n  ENSEMBLE RESULTS ({name}):")
    print(f"  Threshold={ens_thr:.2f}")
    print(f"  Acc={ens_metrics['accuracy']:.4f}  F1={ens_metrics['f1']:.4f}  "
          f"AUC={ens_metrics['auc']:.4f}  BalAcc={ens_metrics['balanced_accuracy']:.4f}")
    print(f"  Sens={ens_metrics['sensitivity']:.4f}  Spec={ens_metrics['specificity']:.4f}  "
          f"ECE={ens_metrics['ece']:.4f}")

    return {
        "name": name,
        "temperatures": seed_temperatures,
        "thresholds": seed_thresholds,
        "ensemble_threshold": float(ens_thr),
        "ensemble_tune_f1": float(ens_tune_f1),
        "trainable_params": param_count,
        "training_time_s": total_time,
        "test_metrics": ens_metrics,
        "per_seed_metrics": seed_test_metrics,
        "test_preds": ens_preds,
        "test_labels": test_labels,
        "test_probs": ensemble_probs,
        "test_pids": test_res["pids"],
        "test_slice_idxs": test_res["slice_idxs"],
        "histories": all_histories,
    }


# ============================================================
# PLOTTING
# ============================================================

def plot_confusion_matrix(labels, preds, name, save_path):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(["Non-Tumor","Tumor"])
    ax.set_yticklabels(["Non-Tumor","Tumor"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i,j]}", ha="center", va="center",
                    color="white" if cm[i,j] > cm.max()/2 else "black", fontsize=14)
    ax.set_title(f"CM: {name}")
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(save_path, dpi=CFG.FIG_DPI, bbox_inches="tight")
    plt.close()


def plot_training_curves(history, name, save_path):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    epochs = [h["epoch"] for h in history]
    axes[0].plot(epochs, [h["train_loss"] for h in history], 'b-')
    axes[0].set_title(f"{name}: Training Loss"); axes[0].set_xlabel("Epoch")
    axes[1].plot(epochs, [h["val_f1"] for h in history], 'g-', label="F1")
    axes[1].plot(epochs, [h["val_acc"] for h in history], 'b--', label="Acc")
    axes[1].legend(); axes[1].set_title(f"{name}: Validation")
    axes[2].plot(epochs, [h["val_sens"] for h in history], 'r-', label="Sens")
    axes[2].plot(epochs, [h["val_spec"] for h in history], 'b-', label="Spec")
    axes[2].legend(); axes[2].set_title(f"{name}: Sens/Spec")
    plt.tight_layout()
    plt.savefig(save_path, dpi=CFG.FIG_DPI, bbox_inches="tight")
    plt.close()


def plot_comparison_chart(all_results, save_path):
    names = [r["name"] for r in all_results]
    metrics_list = ["f1", "auc", "balanced_accuracy", "specificity", "sensitivity"]
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(names))
    width = 0.15
    for i, m in enumerate(metrics_list):
        vals = [r["test_metrics"][m] for r in all_results]
        ax.bar(x + i * width, vals, width, label=m.upper())
    ax.set_xticks(x + 2 * width)
    ax.set_xticklabels(names, rotation=30, ha="right")
    ax.set_ylim(0.80, 1.0)
    ax.legend(fontsize=8)
    ax.set_title("Competitive Baselines Comparison")
    plt.tight_layout()
    plt.savefig(save_path, dpi=CFG.FIG_DPI, bbox_inches="tight")
    plt.close()


# ============================================================
# MAIN
# ============================================================

def main():
    set_seed(CFG.SEEDS[0])
    splits, tr, vl, tu, te = load_data()

    all_results = []

    # Define all baselines: (name, model_class, model_kwargs, use_inst_loss)
    baseline_specs = [
        ("ABMIL",         ABMIL,       {"d_in": CFG.D_IN, "d_model": CFG.D_MODEL},   False),
        ("TransMIL",      TransMIL,    {"d_in": CFG.D_IN, "d_model": CFG.D_MODEL},   False),
        ("CLAM-SB",       CLAM_SB,     {"d_in": CFG.D_IN, "d_model": CFG.D_MODEL},   True),
        ("ViT-Slice-MLP", ViTSliceMLP, {"d_in": CFG.D_IN},                           False),
    ]

    for bname, model_cls, model_kwargs, use_inst in baseline_specs:
        r = train_competitive_baseline(
            bname, model_cls, model_kwargs,
            tr, vl, tu, te,
            use_inst_loss=use_inst,
        )
        all_results.append(r)
        plot_confusion_matrix(r["test_labels"], r["test_preds"], bname,
                             os.path.join(CFG.WORK_DIR, f"phase3b_cm_{bname}.png"))
        # Plot first seed's history
        first_hist = list(r["histories"].values())[0]
        plot_training_curves(first_hist, bname,
                            os.path.join(CFG.WORK_DIR, f"phase3b_curves_{bname}.png"))
        clear_memory()

    # ---- Summary ----
    print(f"\n{'='*72}")
    print("COMPETITIVE BASELINES — 3-SEED ENSEMBLE SUMMARY")
    print(f"{'='*72}")
    print(f"{'Method':<20} {'Acc':>7} {'F1':>7} {'AUC':>7} {'BalAcc':>7} {'Spec':>7} {'Sens':>7} {'ECE':>7}")
    print("-" * 79)
    for r in all_results:
        m = r["test_metrics"]
        print(f"{r['name']:<20} {m['accuracy']:>7.4f} {m['f1']:>7.4f} {m['auc']:>7.4f} "
              f"{m['balanced_accuracy']:>7.4f} {m['specificity']:>7.4f} {m['sensitivity']:>7.4f} "
              f"{m['ece']:>7.4f}")
    print()
    print("Per-seed breakdown:")
    for r in all_results:
        for i, (seed, sm) in enumerate(zip(CFG.SEEDS, r["per_seed_metrics"])):
            print(f"  {r['name']} seed {seed}: F1={sm['f1']:.4f} AUC={sm['auc']:.4f} "
                  f"Spec={sm['specificity']:.4f} Sens={sm['sensitivity']:.4f}")

    # ---- Save ----
    plot_comparison_chart(all_results,
                         os.path.join(CFG.WORK_DIR, "phase3b_comparison_chart.png"))

    # Save predictions for McNemar's test
    pred_store = {}
    for r in all_results:
        pred_store[r["name"]] = {
            "patient_id": r["test_pids"].tolist(),
            "slice_idx": r["test_slice_idxs"].astype(int).tolist(),
            "label": r["test_labels"].astype(int).tolist(),
            "pred": r["test_preds"].astype(int).tolist(),
            "prob_1": r["test_probs"][:, 1].astype(float).tolist(),
        }
    save_json(pred_store, os.path.join(CFG.WORK_DIR, "phase3b_competitive_predictions.json"))

    # Save summary metrics
    summary = {}
    for r in all_results:
        summary[r["name"]] = {
            "family": "competitive_baseline",
            "ensemble": "3-seed average",
            "seeds": CFG.SEEDS,
            "temperatures": r["temperatures"],
            "ensemble_threshold": r["ensemble_threshold"],
            "ensemble_tune_f1": r["ensemble_tune_f1"],
            "trainable_params": r["trainable_params"],
            "training_time_s": r["training_time_s"],
            "per_seed_metrics": r["per_seed_metrics"],
            **r["test_metrics"],
        }
    save_json(summary, os.path.join(CFG.WORK_DIR, "phase3b_competitive_results.json"))

    # Save histories
    histories = {r["name"]: r["histories"] for r in all_results}
    save_json(histories, os.path.join(CFG.WORK_DIR, "phase3b_training_histories.json"))

    print(f"\n{'='*72}")
    print("ALL COMPETITIVE BASELINES COMPLETE (3-SEED ENSEMBLE)")
    print(f"Outputs saved to {CFG.WORK_DIR}/phase3b_*")
    print(f"{'='*72}")


if __name__ == "__main__":
    main()


In [ ]:
# ============================================================
# PHASE 3C: HYPERPARAMETER SENSITIVITY ANALYSIS
# Varies: W (window size), K (top-K), n_layers (Transformer depth),
#         loss weights — single-seed (42) for efficiency
# Note: tau (tumor threshold) is fixed at 100 because it affects
#       Phase 1 slice labeling and requires full data re-extraction.
#       Justified in paper as ~100mm^2 minimum clinically relevant area.
# Designed for Kaggle P100, ~3 hours total runtime
# ============================================================

import os, gc, json, time, math, copy, random, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, balanced_accuracy_score, average_precision_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================
class Config:
    PHASE1_DIR = "/kaggle/working"
    WORK_DIR   = "/kaggle/working"
    DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

    SEED       = 42          # single seed for sensitivity
    NUM_CLASSES = 2
    D_IN       = 384
    D_MODEL    = 128
    MAX_SLICES = 48

    # ----- defaults (these are what we vary) -----
    TRAIN_WINDOW          = 11
    TOP_K                 = 4
    CENTER_LOSS_WEIGHT    = 1.00
    BAG_LOSS_WEIGHT       = 0.20
    SMOOTHNESS_LOSS_WEIGHT = 0.06
    DIVERSITY_LOSS_WEIGHT  = 0.01

    TOTAL_EPOCHS   = 32
    LR             = 8e-4
    WEIGHT_DECAY   = 1e-4
    BATCH_SIZE     = 32
    EVAL_BATCH_SIZE = 64
    NUM_WORKERS    = 0
    PATIENCE       = 8
    CLIP_GRAD_NORM = 1.0

    MODALITY_DROPOUT    = 0.10
    PROTO_MOMENTUM      = 0.90
    CENTER_GAMMA_POS    = 1.5
    CENTER_GAMMA_NEG    = 3.0
    FIG_DPI             = 300

    SAMPLER_POSITIVE_BOOST     = 1.20
    SAMPLER_BOUNDARY_NEG_BOOST = 1.45
    SAMPLER_FAR_NEG_BOOST      = 1.20
    SAMPLER_NO_TUMOR_BOOST     = 1.10


CFG = Config()

SLICE_ROLE_TO_ID = {
    "far_neg": 0, "boundary_neg": 1, "positive": 2,
    "mid_neg": 3, "no_tumor": 4,
}
ROLE_LOSS_MULTIPLIER = {0: 1.25, 1: 1.60, 2: 1.00, 3: 1.00, 4: 1.15}


# ============================================================
# UTILS (compact)
# ============================================================
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def clear_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def save_json(obj, path):
    with open(path, "w") as f: json.dump(obj, f, indent=2, default=str)

def safe_auc(labels, prob1):
    labels, prob1 = np.asarray(labels), np.asarray(prob1)
    if len(labels) == 0 or len(np.unique(labels)) < 2: return float("nan")
    try: return float(roc_auc_score(labels, prob1))
    except: return float("nan")

def safe_pr_auc(labels, prob1):
    labels, prob1 = np.asarray(labels), np.asarray(prob1)
    if len(labels) == 0 or len(np.unique(labels)) < 2: return float("nan")
    try: return float(average_precision_score(labels, prob1))
    except: return float("nan")

def calculate_ece(probs, labels, n_bins=5):
    probs, labels = np.asarray(probs), np.asarray(labels)
    if len(labels) == 0: return 0.0
    conf = np.max(probs, axis=1); pred = np.argmax(probs, axis=1)
    acc = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for i in range(n_bins):
        m = (conf >= bins[i]) & (conf <= bins[i+1]) if i == 0 else (conf > bins[i]) & (conf <= bins[i+1])
        if m.sum() > 0:
            ece += (m.sum()/len(labels)) * abs(acc[m].mean() - conf[m].mean())
    return float(ece)

def metrics_from_probs(probs, labels, threshold=0.5):
    probs, labels = np.asarray(probs), np.asarray(labels)
    if len(labels) == 0:
        return {"accuracy":0,"precision":0,"recall":0,"f1":0,"auc":0,"pr_auc":0,
                "ece":0,"balanced_accuracy":0,"specificity":0,"sensitivity":0}, np.array([])
    preds = (probs[:, 1] >= threshold).astype(int)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "precision": float(precision_score(labels, preds, average="macro", zero_division=0)),
        "recall": float(recall_score(labels, preds, average="macro", zero_division=0)),
        "f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "auc": safe_auc(labels, probs[:, 1]),
        "pr_auc": safe_pr_auc(labels, probs[:, 1]),
        "ece": float(calculate_ece(probs, labels)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, preds)) if len(np.unique(labels))>1 else 0.0,
        "specificity": float(recall_score(labels, preds, pos_label=0, zero_division=0)),
        "sensitivity": float(recall_score(labels, preds, pos_label=1, zero_division=0)),
    }, preds

def find_best_threshold(probs, labels):
    probs, labels = np.asarray(probs), np.asarray(labels)
    if len(labels)==0 or len(np.unique(labels))<2: return 0.50, 0.0
    best_t, best_f1 = 0.50, -1.0
    for t in np.arange(0.10, 0.91, 0.01):
        preds = (probs[:,1]>=t).astype(int)
        f = f1_score(labels, preds, average="macro", zero_division=0)
        if f > best_f1: best_t, best_f1 = float(t), float(f)
    return best_t, best_f1

def fit_temperature(logits, labels, device):
    logits, labels = np.asarray(logits), np.asarray(labels)
    if len(labels)==0 or len(np.unique(labels))<2: return 1.0
    T = nn.Parameter(torch.ones(1, device=device))
    lt = torch.tensor(logits, dtype=torch.float32, device=device)
    lb = torch.tensor(labels, dtype=torch.long, device=device)
    opt = torch.optim.LBFGS([T], lr=0.1, max_iter=50, line_search_fn="strong_wolfe")
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(lt/T.clamp_min(0.5), lb); loss.backward(); return loss
    opt.step(closure)
    return float(T.detach().clamp_min(0.5).cpu().item())


# ============================================================
# DATA LOADING
# ============================================================
def load_npz(name):
    path = os.path.join(CFG.PHASE1_DIR, f"{name}_features_4mod_v3.npz")
    with np.load(path, allow_pickle=True) as data:
        return {k: data[k] for k in data.files}

def load_data():
    with open(os.path.join(CFG.PHASE1_DIR, "patient_splits_v3.json")) as f:
        splits = json.load(f)
    tr = load_npz("train"); vl = load_npz("val"); tu = load_npz("tune"); te = load_npz("test")
    print(f"Loaded: train={len(tr['labels'])}, val={len(vl['labels'])}, tune={len(tu['labels'])}, test={len(te['labels'])}")
    return splits, tr, vl, tu, te


# ============================================================
# DATASET (parameterized window_size)
# ============================================================
class CenterWindowDataset(Dataset):
    def __init__(self, data_dict, window_size=11):
        assert window_size % 2 == 1
        self.window_size = window_size
        self.half = window_size // 2

        patient_ids = np.asarray(data_dict["patient_ids"]).astype(str)
        slice_idxs  = np.asarray(data_dict["slice_idxs"]).astype(np.int64)
        f_t1    = np.asarray(data_dict["features_t1"], dtype=np.float32)
        f_t1ce  = np.asarray(data_dict["features_t1ce"], dtype=np.float32)
        f_t2    = np.asarray(data_dict["features_t2"], dtype=np.float32)
        f_flair = np.asarray(data_dict["features_flair"], dtype=np.float32)
        labels  = np.asarray(data_dict["labels"], dtype=np.int64)
        mod_mask = np.asarray(data_dict["modality_mask"], dtype=np.int64)
        n = len(labels)

        dist  = data_dict.get("distance_to_positive", np.zeros(n, dtype=np.int64)).astype(np.int64)
        role  = data_dict.get("slice_role_id", np.zeros(n, dtype=np.int64)).astype(np.int64)
        tpix  = data_dict.get("tumor_pixels", np.zeros(n, dtype=np.int64)).astype(np.int64)

        self.patient_data = {}
        pid_map = {}
        for i, pid in enumerate(patient_ids):
            pid_map.setdefault(pid, []).append(i)
        for pid in sorted(pid_map):
            idxs = np.asarray(pid_map[pid], dtype=np.int64)
            order = np.argsort(slice_idxs[idxs], kind="mergesort")
            idxs = idxs[order]
            self.patient_data[pid] = {
                "t1": f_t1[idxs], "t1ce": f_t1ce[idxs], "t2": f_t2[idxs], "flair": f_flair[idxs],
                "labels": labels[idxs], "slice_idxs": slice_idxs[idxs],
                "modality_mask": mod_mask[idxs], "distance_to_positive": dist[idxs],
                "slice_role_id": role[idxs], "tumor_pixels": tpix[idxs],
            }

        self.index = []
        for pid in sorted(self.patient_data):
            d = self.patient_data[pid]
            bag_label = int(d["labels"].max() > 0)
            for li in range(len(d["labels"])):
                self.index.append({
                    "pid": pid, "local_idx": int(li),
                    "center_label": int(d["labels"][li]), "bag_label": bag_label,
                    "slice_role_id": int(d["slice_role_id"][li]),
                    "slice_idx": int(d["slice_idxs"][li]),
                })

    def __len__(self): return len(self.index)

    def _build_window(self, arr, center_idx):
        out, mask = [], []
        for offset in range(-self.half, self.half + 1):
            j = center_idx + offset
            if 0 <= j < len(arr):
                out.append(arr[j]); mask.append(True)
            else:
                out.append(np.zeros(arr.shape[1], dtype=np.float32)); mask.append(False)
        return np.stack(out), np.array(mask, dtype=bool)

    def __getitem__(self, idx):
        m = self.index[idx]; d = self.patient_data[m["pid"]]; c = m["local_idx"]
        t1, seq_mask = self._build_window(d["t1"], c)
        t1ce, _ = self._build_window(d["t1ce"], c)
        t2, _   = self._build_window(d["t2"], c)
        flair, _ = self._build_window(d["flair"], c)
        bmm = (d["modality_mask"].sum(axis=0) > 0).astype(np.int64)
        if bmm.sum() == 0: bmm[:] = 1
        return (
            torch.tensor(t1,   dtype=torch.float32),
            torch.tensor(t1ce, dtype=torch.float32),
            torch.tensor(t2,   dtype=torch.float32),
            torch.tensor(flair,dtype=torch.float32),
            torch.tensor(m["bag_label"],    dtype=torch.long),
            torch.tensor(m["center_label"], dtype=torch.long),
            torch.tensor(seq_mask,  dtype=torch.bool),
            torch.tensor(bmm,      dtype=torch.bool),
            torch.tensor(self.half, dtype=torch.long),
            torch.tensor(m["slice_role_id"], dtype=torch.long),
        )


def make_loader(data_dict, split_info=None, training=False, window_size=11):
    ds = CenterWindowDataset(data_dict, window_size=window_size)
    if training and split_info is not None:
        weights = []
        scw = split_info.get("slice_class_weights", {"0": 1.0, "1": 1.0})
        for item in ds.index:
            w = float(scw.get(str(item["center_label"]), 1.0))
            if item["center_label"] == 1: w *= CFG.SAMPLER_POSITIVE_BOOST
            elif item["slice_role_id"] == 1: w *= CFG.SAMPLER_BOUNDARY_NEG_BOOST
            elif item["slice_role_id"] == 0: w *= CFG.SAMPLER_FAR_NEG_BOOST
            elif item["slice_role_id"] == 4: w *= CFG.SAMPLER_NO_TUMOR_BOOST
            weights.append(w)
        sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), len(weights), replacement=True)
        return ds, DataLoader(ds, batch_size=CFG.BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=True)
    return ds, DataLoader(ds, batch_size=CFG.EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)


# ============================================================
# MODEL (identical to Phase 2/3 PUMAMILLite)
# ============================================================
def masked_topk_mean(seq_feats, scores, seq_mask, top_k):
    bsz, _, dim = seq_feats.shape; out = []; dev = seq_feats.device; dt = seq_feats.dtype
    for b in range(bsz):
        vi = torch.where(seq_mask[b])[0]
        if vi.numel()==0: out.append(torch.zeros(dim, device=dev, dtype=dt)); continue
        k = min(int(top_k), int(vi.numel()))
        ci = vi[torch.topk(scores[b, vi], k=k).indices]
        out.append(seq_feats[b, ci].mean(dim=0))
    return torch.stack(out)

class ReliabilityFusion(nn.Module):
    def __init__(self, d_in=384, d_model=128):
        super().__init__()
        self.proj = nn.ModuleList([nn.Sequential(nn.Linear(d_in, d_model), nn.LayerNorm(d_model), nn.GELU()) for _ in range(4)])
        self.rel  = nn.ModuleList([nn.Linear(d_model, 1) for _ in range(4)])
        self.out  = nn.Sequential(nn.Linear(d_model*2, d_model), nn.LayerNorm(d_model), nn.GELU())
    def forward(self, mods, mod_mask):
        pf, rl = [], []
        for i in range(4):
            p = self.proj[i](mods[i]); pf.append(p); rl.append(self.rel[i](p))
        fs = torch.stack(pf, dim=2); rs = torch.cat(rl, dim=-1)
        rs = rs.masked_fill(~mod_mask[:, None, :], -1e4)
        w = torch.softmax(rs, dim=-1); fused = (fs * w.unsqueeze(-1)).sum(dim=2)
        valid = mod_mask[:, None, :, None].float()
        mf = (fs * valid).sum(dim=2) / valid.sum(dim=2).clamp_min(1.0)
        return self.out(torch.cat([fused, mf], dim=-1)), w

class HybridMILPool(nn.Module):
    def __init__(self, d_model=128, top_k=4):
        super().__init__()
        self.top_k = top_k
        self.attn = nn.Sequential(nn.Linear(d_model, d_model//2), nn.Tanh(), nn.Linear(d_model//2, 1))
    def forward(self, seq_feats, slice_logits, seq_mask):
        probs = torch.softmax(slice_logits, dim=-1)
        entropy = -(probs * torch.log(probs.clamp_min(1e-8))).sum(dim=-1)/math.log(2.0)
        rel = 1.0 - entropy
        al = self.attn(seq_feats).squeeze(-1) + torch.log(rel.clamp_min(1e-6))
        al = al.masked_fill(~seq_mask, -1e9)
        aw = torch.softmax(al, dim=1); ap = (seq_feats * aw.unsqueeze(-1)).sum(dim=1)
        pf = masked_topk_mean(seq_feats, probs[...,1], seq_mask, self.top_k)
        nf = masked_topk_mean(seq_feats, probs[...,0], seq_mask, self.top_k)
        return torch.cat([ap, pf, nf], dim=-1), aw, rel

class PUMAMILLite(nn.Module):
    def __init__(self, d_in=384, d_model=128, max_slices=48, top_k=4, proto_momentum=0.9, n_layers=2):
        super().__init__()
        self.d_model = d_model; self.max_slices = max_slices; self.top_k = top_k
        self.fusion = ReliabilityFusion(d_in, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, max_slices, d_model))
        el = nn.TransformerEncoderLayer(d_model=d_model, nhead=4, dim_feedforward=256,
                                        dropout=0.10, batch_first=True, activation="gelu")
        self.encoder = nn.TransformerEncoder(el, num_layers=n_layers)
        self.slice_head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 2))
        self.pool = HybridMILPool(d_model, top_k)
        self.register_buffer("prototypes", torch.zeros(2, d_model*3))
        self.register_buffer("proto_counts", torch.zeros(2))
        self.final_head = nn.Sequential(
            nn.LayerNorm(d_model*3+2), nn.Linear(d_model*3+2, d_model),
            nn.GELU(), nn.Dropout(0.25), nn.Linear(d_model, 2))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.proto_momentum = proto_momentum

    def modality_dropout(self, mod_mask):
        if not self.training or CFG.MODALITY_DROPOUT <= 0: return mod_mask
        keep = mod_mask.clone(); rand = torch.rand_like(keep.float())
        keep = keep & (rand > CFG.MODALITY_DROPOUT)
        for i in range(keep.size(0)):
            if keep[i].sum()==0:
                v=torch.where(mod_mask[i])[0]
                if v.numel()>0: keep[i, v[torch.randint(0,v.numel(),(1,),device=v.device)]] = True
                else: keep[i] = True
        return keep

    def update_prototypes(self, bag_embed, bag_labels):
        with torch.no_grad():
            for c in [0,1]:
                m = bag_labels == c
                if m.any():
                    bp = bag_embed[m].mean(dim=0)
                    if self.proto_counts[c]==0: self.prototypes[c] = bp
                    else: self.prototypes[c] = self.proto_momentum*self.prototypes[c]+(1-self.proto_momentum)*bp
                    self.proto_counts[c] += m.sum()

    def prototype_sims(self, bag_embed):
        sims = []
        for c in [0,1]:
            if self.proto_counts[c]<=0 or torch.norm(self.prototypes[c])<1e-8:
                sims.append(torch.zeros(bag_embed.size(0),1,device=bag_embed.device))
            else:
                sims.append(F.cosine_similarity(bag_embed, self.prototypes[c].unsqueeze(0), dim=-1).unsqueeze(-1))
        return torch.cat(sims, dim=-1)

    def forward(self, t1, t1ce, t2, flair, seq_mask, mod_mask, return_aux=False):
        mm = self.modality_dropout(mod_mask)
        fused, mw = self.fusion([t1, t1ce, t2, flair], mm)
        x = fused + self.pos_embed[:, :t1.size(1), :]
        x = self.encoder(x, src_key_padding_mask=~seq_mask)
        sl = self.slice_head(x)
        be, aw, rel = self.pool(x, sl, seq_mask)
        ps = self.prototype_sims(be)
        bl = self.final_head(torch.cat([be, ps], dim=-1))
        if return_aux: return bl, sl, be, mw, aw, rel
        return bl


# ============================================================
# LOSSES
# ============================================================
def weighted_center_focal_loss(center_logits, center_labels, role_ids):
    lp = F.log_softmax(center_logits, dim=-1)
    pt = lp.exp().gather(1, center_labels.unsqueeze(1)).squeeze(1).clamp_min(1e-8)
    ce = F.nll_loss(lp, center_labels, reduction="none")
    gamma = torch.where(center_labels==1,
        torch.full_like(center_labels, CFG.CENTER_GAMMA_POS, dtype=torch.float32),
        torch.full_like(center_labels, CFG.CENTER_GAMMA_NEG, dtype=torch.float32))
    focal = ((1.0-pt)**gamma)*ce
    rm = torch.tensor([ROLE_LOSS_MULTIPLIER.get(int(r),1.0) for r in role_ids.cpu().numpy()],
                      dtype=torch.float32, device=center_logits.device)
    return (focal*rm).mean()

def smoothness_loss(slice_logits, seq_mask):
    p = torch.softmax(slice_logits, dim=-1)[...,1]
    d = torch.abs(p[:,1:]-p[:,:-1]); v = seq_mask[:,1:]&seq_mask[:,:-1]
    return d[v].mean() if v.any() else slice_logits.new_tensor(0.0)

def diversity_loss(mod_weights):
    """Returns negative entropy — minimize this to maximize modality diversity."""
    w = mod_weights.mean(dim=(0,1))
    return (w * torch.log(w.clamp_min(1e-8))).sum()  # negative value; minimizing = maximizing entropy


# ============================================================
# SINGLE TRAINING RUN
# ============================================================
def train_one_config(tr, vl, tu, te, split_info, config_name,
                     window_size=11, top_k=4, n_layers=2,
                     lw_center=1.0, lw_bag=0.2, lw_smooth=0.06, lw_div=0.01):
    """Train PUMA-MIL with one hyperparameter configuration, single seed."""
    print(f"\n{'='*60}")
    print(f"CONFIG: {config_name}")
    print(f"  W={window_size}, K={top_k}, n_layers={n_layers}, losses=({lw_center},{lw_bag},{lw_smooth},{lw_div})")
    print(f"{'='*60}")

    set_seed(CFG.SEED)
    device = CFG.DEVICE

    # Build data loaders with the specified window
    _, train_loader = make_loader(tr, split_info, training=True,  window_size=window_size)
    _, val_loader   = make_loader(vl, None,       training=False, window_size=window_size)
    _, tune_loader  = make_loader(tu, None,       training=False, window_size=window_size)
    _, test_loader  = make_loader(te, None,       training=False, window_size=window_size)

    # Build model with specified top_k and n_layers
    model = PUMAMILLite(d_in=CFG.D_IN, d_model=CFG.D_MODEL, max_slices=CFG.MAX_SLICES,
                        top_k=top_k, proto_momentum=CFG.PROTO_MOMENTUM, n_layers=n_layers).to(device)

    bag_w = split_info.get("bag_class_weights", {"0":1.0,"1":1.0})
    bag_cw = torch.tensor([float(bag_w.get("0",1.0)), float(bag_w.get("1",1.0))],
                          dtype=torch.float32, device=device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    best_f1, best_state, patience_ctr = -1.0, None, 0
    center_half = window_size // 2

    t0 = time.time()

    for epoch in range(CFG.TOTAL_EPOCHS):
        model.train()
        for batch in train_loader:
            t1,t1ce,t2,flair,bag_y,center_y,seq_mask,mod_mask,_,role_ids = [
                b.to(device) if isinstance(b, torch.Tensor) else b for b in batch]

            bag_logits, slice_logits, bag_embed, mod_w, _, _ = model(
                t1, t1ce, t2, flair, seq_mask, mod_mask, return_aux=True)

            center_logits = slice_logits[:, center_half, :]
            l_center = weighted_center_focal_loss(center_logits, center_y, role_ids)
            l_bag    = F.cross_entropy(bag_logits, bag_y, weight=bag_cw)
            l_smooth = smoothness_loss(slice_logits, seq_mask)
            l_div    = diversity_loss(mod_w)  # already negative; minimizing maximizes diversity

            loss = lw_center*l_center + lw_bag*l_bag + lw_smooth*l_smooth + lw_div*l_div

            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.CLIP_GRAD_NORM)
            optimizer.step()
            model.update_prototypes(bag_embed.detach(), bag_y)

        # Validate
        model.eval()
        all_logits, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                t1,t1ce,t2,flair,bag_y,center_y,seq_mask,mod_mask,_,role_ids = [
                    b.to(device) if isinstance(b, torch.Tensor) else b for b in batch]
                _, sl, _, _, _, _ = model(t1, t1ce, t2, flair, seq_mask, mod_mask, return_aux=True)
                all_logits.append(sl[:, center_half, :].cpu().numpy())
                all_labels.append(center_y.cpu().numpy())

        val_logits = np.concatenate(all_logits)
        val_labels = np.concatenate(all_labels)
        val_probs  = torch.softmax(torch.tensor(val_logits), dim=-1).numpy()
        vm, _ = metrics_from_probs(val_probs, val_labels)
        val_f1 = vm["f1"]

        if val_f1 > best_f1:
            best_f1 = val_f1; best_state = copy.deepcopy(model.state_dict())
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= CFG.PATIENCE:
                print(f"  Early stop at epoch {epoch+1}, best val F1={best_f1:.4f}")
                break

    train_time = time.time() - t0
    print(f"  Training time: {train_time:.1f}s, best val F1={best_f1:.4f}")

    # Load best model
    model.load_state_dict(best_state)
    model.eval()

    # Temperature scaling on tune set
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in tune_loader:
            t1,t1ce,t2,flair,bag_y,center_y,seq_mask,mod_mask,_,role_ids = [
                b.to(device) if isinstance(b, torch.Tensor) else b for b in batch]
            _, sl, _, _, _, _ = model(t1, t1ce, t2, flair, seq_mask, mod_mask, return_aux=True)
            all_logits.append(sl[:, center_half, :].cpu().numpy())
            all_labels.append(center_y.cpu().numpy())
    tune_logits = np.concatenate(all_logits); tune_labels = np.concatenate(all_labels)
    temperature = fit_temperature(tune_logits, tune_labels, device)
    tune_probs = torch.softmax(torch.tensor(tune_logits)/temperature, dim=-1).numpy()
    threshold, tune_f1 = find_best_threshold(tune_probs, tune_labels)

    # Test set evaluation
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            t1,t1ce,t2,flair,bag_y,center_y,seq_mask,mod_mask,_,role_ids = [
                b.to(device) if isinstance(b, torch.Tensor) else b for b in batch]
            _, sl, _, _, _, _ = model(t1, t1ce, t2, flair, seq_mask, mod_mask, return_aux=True)
            all_logits.append(sl[:, center_half, :].cpu().numpy())
            all_labels.append(center_y.cpu().numpy())
    test_logits = np.concatenate(all_logits); test_labels = np.concatenate(all_labels)
    test_probs = torch.softmax(torch.tensor(test_logits)/temperature, dim=-1).numpy()
    test_metrics, _ = metrics_from_probs(test_probs, test_labels, threshold=threshold)

    result = {
        "config": config_name,
        "window_size": window_size, "top_k": top_k, "n_layers": n_layers,
        "loss_weights": {"center": lw_center, "bag": lw_bag, "smooth": lw_smooth, "div": lw_div},
        "temperature": temperature, "threshold": threshold,
        "best_val_f1": best_f1, "tune_f1": tune_f1,
        "train_time_s": train_time,
        **test_metrics,
    }

    print(f"  Test: F1={test_metrics['f1']:.4f}, AUC={test_metrics['auc']:.4f}, "
          f"Spec={test_metrics['specificity']:.4f}, Sens={test_metrics['sensitivity']:.4f}")

    del model, optimizer; clear_memory()
    return result


# ============================================================
# MAIN: SENSITIVITY EXPERIMENTS
# ============================================================
def main():
    splits, tr, vl, tu, te = load_data()
    split_info = splits.get("train_split_info", splits.get("train", {}))

    all_results = []

    # ---------------------------------------------------------
    # 1. WINDOW SIZE SENSITIVITY: W ∈ {5, 7, 9, 11, 13, 15}
    # ---------------------------------------------------------
    print("\n" + "=" * 72)
    print("SENSITIVITY ANALYSIS 1: WINDOW SIZE (W)")
    print("=" * 72)
    for W in [5, 7, 9, 11, 13, 15]:
        r = train_one_config(tr, vl, tu, te, split_info,
                             config_name=f"W={W}", window_size=W)
        r["experiment"] = "window_size"
        all_results.append(r)

    # ---------------------------------------------------------
    # 2. TOP-K SENSITIVITY: K ∈ {2, 3, 4, 6, 8}
    # ---------------------------------------------------------
    print("\n" + "=" * 72)
    print("SENSITIVITY ANALYSIS 2: TOP-K")
    print("=" * 72)
    for K in [2, 3, 4, 6, 8]:
        r = train_one_config(tr, vl, tu, te, split_info,
                             config_name=f"K={K}", top_k=K)
        r["experiment"] = "top_k"
        all_results.append(r)

    # ---------------------------------------------------------
    # 3. TRANSFORMER DEPTH: n_layers ∈ {1, 2, 3}
    # ---------------------------------------------------------
    print("\n" + "=" * 72)
    print("SENSITIVITY ANALYSIS 3: TRANSFORMER DEPTH")
    print("=" * 72)
    for nl in [1, 2, 3]:
        r = train_one_config(tr, vl, tu, te, split_info,
                             config_name=f"layers={nl}", n_layers=nl)
        r["experiment"] = "n_layers"
        all_results.append(r)

    # ---------------------------------------------------------
    # 4. LOSS WEIGHT SENSITIVITY
    # ---------------------------------------------------------
    print("\n" + "=" * 72)
    print("SENSITIVITY ANALYSIS 4: LOSS WEIGHTS")
    print("=" * 72)
    loss_configs = [
        ("center-only",     1.0, 0.0,  0.0,  0.0),
        ("center+bag",      1.0, 0.2,  0.0,  0.0),
        ("default",         1.0, 0.2,  0.06, 0.01),
        ("2x-bag",          1.0, 0.4,  0.06, 0.01),
        ("2x-smooth",       1.0, 0.2,  0.12, 0.01),
        ("equal-aux",       1.0, 0.1,  0.1,  0.1),
    ]
    for name, lc, lb, ls, ld in loss_configs:
        r = train_one_config(tr, vl, tu, te, split_info,
                             config_name=f"loss-{name}",
                             lw_center=lc, lw_bag=lb, lw_smooth=ls, lw_div=ld)
        r["experiment"] = "loss_weights"
        all_results.append(r)

    # ---------------------------------------------------------
    # SAVE RESULTS
    # ---------------------------------------------------------
    save_json(all_results, os.path.join(CFG.WORK_DIR, "phase3c_sensitivity_results.json"))

    # ---------------------------------------------------------
    # SUMMARY TABLE
    # ---------------------------------------------------------
    print("\n" + "=" * 72)
    print("SENSITIVITY ANALYSIS — SUMMARY TABLE")
    print("=" * 72)
    print(f"  {'Config':<22} {'Val-F1':>7} {'Test-F1':>8} {'AUC':>7} {'Spec':>7} {'Sens':>7} {'ECE':>7}")
    print(f"  {'-'*68}")
    for r in all_results:
        print(f"  {r['config']:<22} {r['best_val_f1']:.4f} {r['f1']:.4f}  "
              f"{r['auc']:.4f} {r['specificity']:.4f} {r['sensitivity']:.4f} {r['ece']:.4f}")

    # ---------------------------------------------------------
    # PLOT: one figure per experiment
    # ---------------------------------------------------------
    for exp_name in ["window_size", "top_k", "n_layers", "loss_weights"]:
        subset = [r for r in all_results if r["experiment"] == exp_name]
        if not subset: continue

        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        names = [r["config"] for r in subset]
        x = np.arange(len(names))

        for ax, metric, label in zip(axes, ["f1", "auc", "specificity"],
                                     ["Macro-F1", "AUC", "Specificity"]):
            vals = [r[metric] for r in subset]
            bars = ax.bar(x, vals, color="#4ECDC4", edgecolor="#2C3E50", alpha=0.85)
            ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha="right", fontsize=8)
            ax.set_ylabel(label); ax.set_title(f"{label} by {exp_name}")
            # highlight default config in red
            for i, r in enumerate(subset):
                if "default" in r["config"] or r["config"] in ["W=11", "K=4", "layers=2"]:
                    bars[i].set_color("#E74C3C")
            ax.set_ylim(min(vals)-0.02, min(max(vals)+0.005, 1.0))

        fig.suptitle(f"Sensitivity: {exp_name}", fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig(os.path.join(CFG.WORK_DIR, f"phase3c_sensitivity_{exp_name}.png"), dpi=CFG.FIG_DPI, bbox_inches="tight")
        plt.close()
        print(f"  Saved: phase3c_sensitivity_{exp_name}.png")

    print("\n✅ Phase 3C sensitivity analysis complete.")


if __name__ == "__main__":
    main()


In [ ]:
# ============================================================
# PATCH CELL: PHASE 2 & 3 — MISSING OUTPUTS (NO RETRAINING)
# ============================================================
# Run this AFTER Phase 1, 2, and 3 have already completed.
# This cell generates everything that was missing:
#
#  FROM PHASE 2 SAVED ARTIFACTS:
#   - Separate loss + val-metric curve per seed (from training_histories_v3.json)
#   - Separate confusion matrix per seed      (inference from saved checkpoints)
#   - Ensemble confusion matrix
#
#  FROM PHASE 3 SAVED ARTIFACTS:
#   - Separate confusion matrix per baseline  (from phase3_baseline_predictions_v8.json)
#   - Separate confusion matrix per ablation  (from phase3_ablation_predictions_v8.json)
#   - Separate loss + val-metric curve per ablation variant
#     (from phase3_training_histories_v8.json)
#
# Nothing is retrained. Inference for per-seed CMs uses the saved
# pumamil_best_seed{N}.pth checkpoints and is fast (~seconds per seed).
# ============================================================

import os, gc, json, math, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, balanced_accuracy_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ── CONFIG ────────────────────────────────────────────────────────────────────
WORK_DIR  = "/kaggle/working"
SEEDS     = [42, 123, 777]          # must match Phase 2 CFG.SEEDS
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
FIG_DPI   = 300

# Phase 2 model hyper-params (must match Phase 2 CFG exactly)
D_IN            = 384
D_MODEL         = 128
MAX_SLICES      = 48
TOP_K           = 4
PROTO_MOMENTUM  = 0.90
TRAIN_WINDOW    = 11
EVAL_BATCH_SIZE = 64

# Paths written by Phase 2
P2_HISTORIES_JSON = os.path.join(WORK_DIR, "training_histories_v3.json")
P2_PER_SEED_CSV   = os.path.join(WORK_DIR, "per_seed_test_metrics_v3.csv")
P2_SLICE_CSV      = os.path.join(WORK_DIR, "slice_predictions_v3.csv")

# Paths written by Phase 3
P3_BASELINE_PRED_JSON  = os.path.join(WORK_DIR, "phase3_baseline_predictions_v8.json")
P3_ABLATION_PRED_JSON  = os.path.join(WORK_DIR, "phase3_ablation_predictions_v8.json")
P3_HISTORIES_JSON      = os.path.join(WORK_DIR, "phase3_training_histories_v8.json")

# Feature NPZs (Phase 1 outputs used by Phase 2)
# Phase 2 saves "test_features_4mod_v3.npz"; some Kaggle copies drop underscores
_TEST_NPZ_CANDIDATES = [
    os.path.join(WORK_DIR, "test_features_4mod_v3.npz"),
    os.path.join(WORK_DIR, "testfeatures4modv3.npz"),
]
TEST_NPZ = next((p for p in _TEST_NPZ_CANDIDATES if os.path.exists(p)), _TEST_NPZ_CANDIDATES[0])

# ── HELPERS ───────────────────────────────────────────────────────────────────
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _safe_cm(labels, preds):
    """Return 2×2 confusion matrix even when only one class is present."""
    labels = np.asarray(labels, dtype=int)
    preds  = np.asarray(preds,  dtype=int)
    cm = np.zeros((2, 2), dtype=int)
    for t, p in zip(labels, preds):
        t = int(np.clip(t, 0, 1))
        p = int(np.clip(p, 0, 1))
        cm[t, p] += 1
    return cm

def plot_cm(labels, preds, title, out_png):
    cm = _safe_cm(labels, preds)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Non-tumor", "Tumor"],
        yticklabels=["Non-tumor", "Tumor"],
        ax=ax
    )
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("True",      fontsize=11)
    ax.set_title(title,        fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_png, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()
    print(f"  ✓ saved: {os.path.basename(out_png)}")


# ─────────────────────────────────────────────────────────────────────────────
#  SECTION 1 — PHASE 2: PER-SEED TRAINING CURVES (pure plotting)
# ─────────────────────────────────────────────────────────────────────────────
def plot_phase2_per_seed_curves():
    print("\n" + "=" * 60)
    print("SECTION 1 — Phase 2 per-seed training curves")
    print("=" * 60)

    if not os.path.exists(P2_HISTORIES_JSON):
        print(f"  [SKIP] {P2_HISTORIES_JSON} not found")
        return

    with open(P2_HISTORIES_JSON) as f:
        histories = json.load(f)

    if not histories:
        print("  [SKIP] histories JSON is empty")
        return

    for h in histories:
        seed = h["seed"]
        df   = pd.DataFrame(h["history"])

        # Some rows (non-eval epochs) have None for val columns → drop
        val_df  = df.dropna(subset=["val_macro_f1"]).copy()

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f"PUMA-MIL-Lite Training Curves — Seed {seed}",
                     fontsize=13, fontweight="bold")

        # ── panel 1: loss breakdown ──
        ax = axes[0]
        ax.plot(df["epoch"], df["train_total_loss"],      "b-o",  lw=2.0, ms=5, label="Total Loss")
        ax.plot(df["epoch"], df["train_center_loss"],     "r--^", lw=1.5, ms=4, label="Center Loss")
        ax.plot(df["epoch"], df["train_bag_loss"],        "g--s", lw=1.5, ms=4, label="Bag Loss")
        ax.plot(df["epoch"], df["train_smoothness_loss"], "m--D", lw=1.2, ms=3, label="Smooth Loss")
        ax.plot(df["epoch"], df["train_diversity_loss"],  "c--x", lw=1.2, ms=3, label="Div Loss")
        ax.set_title("Training Losses", fontsize=11)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

        # ── panel 2: val accuracy-like metrics ──
        ax = axes[1]
        if len(val_df):
            ax.plot(val_df["epoch"], val_df["val_macro_f1"], "g-^",  lw=2.0, ms=6, label="Val Macro-F1")
            ax.plot(val_df["epoch"], val_df["val_bal_acc"],  "b-o",  lw=2.0, ms=6, label="Val Bal-Acc")
        ax.set_title("Validation Accuracy Metrics", fontsize=11)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Score")
        ax.set_ylim(0, 1); ax.legend(fontsize=9); ax.grid(alpha=0.3)

        # ── panel 3: val specificity / sensitivity / AUC ──
        ax = axes[2]
        if len(val_df):
            ax.plot(val_df["epoch"], val_df["val_specificity"],  "r-s", lw=2.0, ms=6, label="Val Specificity")
            ax.plot(val_df["epoch"], val_df["val_sensitivity"],  "m-D", lw=2.0, ms=6, label="Val Sensitivity")
            if "val_auc" in val_df.columns:
                valid_auc = val_df.dropna(subset=["val_auc"])
                if len(valid_auc):
                    ax.plot(valid_auc["epoch"], valid_auc["val_auc"], "c-o", lw=1.8, ms=5, label="Val AUC")
        ax.set_title("Validation Specificity / Sensitivity / AUC", fontsize=11)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Score")
        ax.set_ylim(0, 1); ax.legend(fontsize=9); ax.grid(alpha=0.3)

        plt.tight_layout()
        out = os.path.join(WORK_DIR, f"patch_phase2_curves_seed{seed}.png")
        plt.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
        plt.close()
        print(f"  ✓ saved: {os.path.basename(out)}")

    # Also save a combined overview (all seeds on one figure)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()
    for h in histories:
        df = pd.DataFrame(h["history"])
        vdf = df.dropna(subset=["val_macro_f1"])
        seed = h["seed"]
        axes[0].plot(df["epoch"],  df["train_total_loss"],  "-o", lw=1.5, label=f"Seed {seed}")
        if len(vdf):
            axes[1].plot(vdf["epoch"], vdf["val_macro_f1"],  "-^", lw=1.5, label=f"Seed {seed}")
            axes[2].plot(vdf["epoch"], vdf["val_bal_acc"],   "-s", lw=1.5, label=f"Seed {seed}")
            axes[3].plot(vdf["epoch"], vdf["val_specificity"],"-D", lw=1.5, label=f"Seed {seed}")
    for ax, ttl in zip(axes, ["Train Total Loss","Val Macro-F1","Val Bal-Acc","Val Specificity"]):
        ax.set_title(ttl); ax.set_xlabel("Epoch"); ax.grid(alpha=0.3); ax.legend()
    plt.suptitle("PUMA-MIL-Lite — All Seeds Overview", fontsize=13, fontweight="bold")
    plt.tight_layout()
    out = os.path.join(WORK_DIR, "patch_phase2_curves_all_seeds.png")
    plt.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()
    print(f"  ✓ saved: {os.path.basename(out)}")


# ─────────────────────────────────────────────────────────────────────────────
#  SECTION 2 — PHASE 2: PER-SEED CONFUSION MATRICES (fast inference only)
# ─────────────────────────────────────────────────────────────────────────────

# ── minimal copy of the Phase 2 model & dataset (no training) ───────────────
def masked_topk_mean(seq_feats, scores, seq_mask, top_k):
    bsz, _, dim = seq_feats.shape
    out = []
    device, dtype = seq_feats.device, seq_feats.dtype
    for b in range(bsz):
        valid_idx = torch.where(seq_mask[b])[0]
        if valid_idx.numel() == 0:
            out.append(torch.zeros(dim, device=device, dtype=dtype))
            continue
        vals = scores[b, valid_idx]
        k = min(int(top_k), int(valid_idx.numel()))
        chosen = valid_idx[torch.topk(vals, k=k, dim=0).indices]
        out.append(seq_feats[b, chosen].mean(dim=0))
    return torch.stack(out, dim=0)

class ReliabilityFusion(nn.Module):
    def __init__(self, d_in=D_IN, d_model=D_MODEL):
        super().__init__()
        self.proj = nn.ModuleList([
            nn.Sequential(nn.Linear(d_in, d_model), nn.LayerNorm(d_model), nn.GELU())
            for _ in range(4)
        ])
        self.rel = nn.ModuleList([nn.Linear(d_model, 1) for _ in range(4)])
        self.out = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.LayerNorm(d_model), nn.GELU()
        )

    def forward(self, mods, mod_mask):
        proj_feats, rel_logits = [], []
        for i in range(4):
            p = self.proj[i](mods[i]); proj_feats.append(p); rel_logits.append(self.rel[i](p))
        feat_stack = torch.stack(proj_feats, dim=2)
        rel_stack  = torch.cat(rel_logits, dim=-1).masked_fill(~mod_mask[:, None, :], -1e4)
        weights = torch.softmax(rel_stack, dim=-1)
        fused   = (feat_stack * weights.unsqueeze(-1)).sum(dim=2)
        valid   = mod_mask[:, None, :, None].float()
        mean_f  = (feat_stack * valid).sum(dim=2) / valid.sum(dim=2).clamp_min(1.0)
        return self.out(torch.cat([fused, mean_f], dim=-1)), weights

class HybridMILPool(nn.Module):
    def __init__(self, d_model=D_MODEL, top_k=TOP_K):
        super().__init__()
        self.top_k = top_k
        self.attn  = nn.Sequential(nn.Linear(d_model, d_model//2), nn.Tanh(), nn.Linear(d_model//2, 1))

    def forward(self, seq_feats, slice_logits, seq_mask):
        probs      = torch.softmax(slice_logits, dim=-1)
        entropy    = -(probs * torch.log(probs.clamp_min(1e-8))).sum(dim=-1) / math.log(2.0)
        reliability = 1.0 - entropy
        attn_logits = self.attn(seq_feats).squeeze(-1) + torch.log(reliability.clamp_min(1e-6))
        attn_logits = attn_logits.masked_fill(~seq_mask, -1e9)
        attn_w  = torch.softmax(attn_logits, dim=1)
        attn_p  = (seq_feats * attn_w.unsqueeze(-1)).sum(dim=1)
        pos_f   = masked_topk_mean(seq_feats, probs[..., 1], seq_mask, self.top_k)
        neg_f   = masked_topk_mean(seq_feats, probs[..., 0], seq_mask, self.top_k)
        return torch.cat([attn_p, pos_f, neg_f], dim=-1), attn_w, reliability

class PUMAMILLite(nn.Module):
    def __init__(self, d_in=D_IN, d_model=D_MODEL, max_slices=MAX_SLICES,
                 top_k=TOP_K, proto_momentum=PROTO_MOMENTUM):
        super().__init__()
        self.d_model = d_model; self.max_slices = max_slices
        self.top_k = top_k; self.proto_momentum = proto_momentum
        self.fusion    = ReliabilityFusion(d_in, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, max_slices, d_model))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=4, dim_feedforward=256,
            dropout=0.10, batch_first=True, activation="gelu")
        self.encoder    = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.slice_head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 2))
        self.pool       = HybridMILPool(d_model, top_k)
        self.register_buffer("prototypes",   torch.zeros(2, d_model * 3))
        self.register_buffer("proto_counts", torch.zeros(2))
        self.final_head = nn.Sequential(
            nn.LayerNorm(d_model * 3 + 2), nn.Linear(d_model * 3 + 2, d_model),
            nn.GELU(), nn.Dropout(0.25), nn.Linear(d_model, 2))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def prototype_sims(self, bag_embed):
        sims = []
        for c in [0, 1]:
            if self.proto_counts[c] <= 0 or torch.norm(self.prototypes[c]) < 1e-8:
                sims.append(torch.zeros(bag_embed.size(0), 1, device=bag_embed.device))
            else:
                s = F.cosine_similarity(bag_embed, self.prototypes[c].unsqueeze(0), dim=-1).unsqueeze(-1)
                sims.append(s)
        return torch.cat(sims, dim=-1)

    def forward(self, t1, t1ce, t2, flair, seq_mask, mod_mask, return_aux=False):
        fused, mod_weights = self.fusion([t1, t1ce, t2, flair], mod_mask)
        x = fused + self.pos_embed[:, :t1.size(1), :]
        x = self.encoder(x, src_key_padding_mask=~seq_mask)
        slice_logits = self.slice_head(x)
        bag_embed, attn_w, reliability = self.pool(x, slice_logits, seq_mask)
        proto_s    = self.prototype_sims(bag_embed)
        bag_logits = self.final_head(torch.cat([bag_embed, proto_s], dim=-1))
        if return_aux:
            return bag_logits, slice_logits, bag_embed, mod_weights, attn_w, reliability
        return bag_logits


def _zeros_if_missing(data_dict, key, shape, dtype):
    if key in data_dict:
        return data_dict[key]
    return np.zeros(shape, dtype=dtype)


class CenterWindowDataset(Dataset):
    """
    Exact copy of Phase 2's CenterWindowDataset.
    NPZ keys: features_t1, features_t1ce, features_t2, features_flair,
              labels, patient_ids, slice_idxs, modality_mask,
              distance_to_positive, slice_role_id, tumor_pixels
    center_pos is always self.half (symmetric window).
    """
    def __init__(self, data_dict, window_size=TRAIN_WINDOW):
        assert window_size % 2 == 1, "window_size must be odd"
        self.window_size = window_size
        self.half = window_size // 2

        patient_ids = np.asarray(data_dict["patient_ids"]).astype(str)
        slice_idxs  = np.asarray(data_dict["slice_idxs"]).astype(np.int64)

        f_t1    = np.asarray(data_dict["features_t1"],    dtype=np.float32)
        f_t1ce  = np.asarray(data_dict["features_t1ce"],  dtype=np.float32)
        f_t2    = np.asarray(data_dict["features_t2"],    dtype=np.float32)
        f_flair = np.asarray(data_dict["features_flair"], dtype=np.float32)
        labels  = np.asarray(data_dict["labels"],         dtype=np.int64)
        mod_mask = np.asarray(data_dict["modality_mask"], dtype=np.int64)

        n = len(labels)
        dist     = _zeros_if_missing(data_dict, "distance_to_positive", (n,), np.int64).astype(np.int64)
        role_id  = _zeros_if_missing(data_dict, "slice_role_id",        (n,), np.int64).astype(np.int64)
        tpix     = _zeros_if_missing(data_dict, "tumor_pixels",         (n,), np.int64).astype(np.int64)
        has_pos  = _zeros_if_missing(data_dict, "has_positive_patient", (n,), np.int64).astype(np.int64)

        # Group by patient, sort by slice index
        self.patient_data = {}
        pid_to_indices    = {}
        for i, pid in enumerate(patient_ids):
            pid_to_indices.setdefault(pid, []).append(i)

        for pid in sorted(pid_to_indices.keys()):
            idxs  = np.asarray(pid_to_indices[pid], dtype=np.int64)
            order = np.argsort(slice_idxs[idxs], kind="mergesort")
            idxs  = idxs[order]
            self.patient_data[pid] = {
                "t1":                   f_t1[idxs],
                "t1ce":                 f_t1ce[idxs],
                "t2":                   f_t2[idxs],
                "flair":                f_flair[idxs],
                "labels":               labels[idxs],
                "slice_idxs":           slice_idxs[idxs],
                "modality_mask":        mod_mask[idxs],
                "distance_to_positive": dist[idxs],
                "slice_role_id":        role_id[idxs],
                "tumor_pixels":         tpix[idxs],
                "has_positive_patient": has_pos[idxs],
            }

        self.index = []
        for pid in sorted(self.patient_data.keys()):
            d         = self.patient_data[pid]
            y         = d["labels"]
            roles     = d["slice_role_id"]
            sx        = d["slice_idxs"]
            dd        = d["distance_to_positive"]
            tp        = d["tumor_pixels"]
            bag_label = int(y.max() > 0)
            for local_idx in range(len(y)):
                self.index.append({
                    "pid":                  pid,
                    "local_idx":            int(local_idx),
                    "center_label":         int(y[local_idx]),
                    "bag_label":            bag_label,
                    "slice_role_id":        int(roles[local_idx]),
                    "slice_idx":            int(sx[local_idx]),
                    "distance_to_positive": int(dd[local_idx]),
                    "tumor_pixels":         int(tp[local_idx]),
                })

    def __len__(self):
        return len(self.index)

    def _build_window(self, arr, center_idx):
        """Build a fixed-size window centred on center_idx, zero-padding boundaries."""
        out  = []
        mask = []
        for offset in range(-self.half, self.half + 1):
            j = center_idx + offset
            if 0 <= j < len(arr):
                out.append(arr[j])
                mask.append(True)
            else:
                out.append(np.zeros(arr.shape[1], dtype=np.float32))
                mask.append(False)
        return np.stack(out, axis=0), np.array(mask, dtype=bool)

    def __getitem__(self, idx):
        meta = self.index[idx]
        pid  = meta["pid"]
        c    = meta["local_idx"]
        d    = self.patient_data[pid]

        t1,    seq_mask = self._build_window(d["t1"],    c)
        t1ce,  _        = self._build_window(d["t1ce"],  c)
        t2,    _        = self._build_window(d["t2"],    c)
        flair, _        = self._build_window(d["flair"], c)

        valid_mm    = d["modality_mask"]
        bag_mod_mask = (valid_mm.sum(axis=0) > 0).astype(np.int64)
        if bag_mod_mask.sum() == 0:
            bag_mod_mask[:] = 1

        return (
            torch.tensor(t1,    dtype=torch.float32),
            torch.tensor(t1ce,  dtype=torch.float32),
            torch.tensor(t2,    dtype=torch.float32),
            torch.tensor(flair, dtype=torch.float32),
            torch.tensor(meta["bag_label"],    dtype=torch.long),
            torch.tensor(meta["center_label"], dtype=torch.long),
            torch.tensor(seq_mask,   dtype=torch.bool),
            torch.tensor(bag_mod_mask, dtype=torch.bool),
            torch.tensor(self.half,  dtype=torch.long),   # center is always middle of window
            torch.tensor(meta["slice_role_id"],        dtype=torch.long),
            str(pid),
            torch.tensor(meta["slice_idx"],            dtype=torch.long),
            torch.tensor(meta["distance_to_positive"], dtype=torch.long),
            torch.tensor(meta["tumor_pixels"],         dtype=torch.long),
        )


def _infer_seed(seed, test_npz_path):
    """Load saved checkpoint for `seed`, run inference on test set, return labels & preds."""
    ckpt_candidates = [
        os.path.join(WORK_DIR, f"pumamil_best_seed{seed}.pth"),
        os.path.join(WORK_DIR, f"pumamil_best_v3.pth"),     # fallback
    ]
    ckpt = next((p for p in ckpt_candidates if os.path.exists(p)), None)
    if ckpt is None:
        print(f"  [SKIP] no checkpoint found for seed {seed}")
        return None, None

    data = np.load(test_npz_path, allow_pickle=True)
    data_dict = {k: data[k] for k in data.files}

    model = PUMAMILLite().to(DEVICE)

    state = torch.load(ckpt, map_location=DEVICE)
    # unwrap DDP / DataParallel prefix if present
    if any(k.startswith("module.") for k in state):
        state = {k.replace("module.", ""): v for k, v in state.items()}
    model.load_state_dict(state, strict=False)
    model.eval()

    ds     = CenterWindowDataset(data_dict, window_size=TRAIN_WINDOW)
    loader = DataLoader(ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0)

    all_labels, all_preds = [], []

    with torch.no_grad():
        for batch in loader:
            (t1, t1ce, t2, flair, bag_y, center_y,
             seq_mask, mod_mask, center_pos,
             role_id, pid, slice_idx, dist, tpix) = batch

            t1      = t1.to(DEVICE);      t1ce  = t1ce.to(DEVICE)
            t2      = t2.to(DEVICE);      flair = flair.to(DEVICE)
            seq_mask = seq_mask.to(DEVICE); mod_mask = mod_mask.to(DEVICE)
            center_pos = center_pos.to(DEVICE)

            _, slice_logits, _, _, _, _ = model(
                t1, t1ce, t2, flair, seq_mask, mod_mask, return_aux=True
            )
            cl = slice_logits[
                torch.arange(slice_logits.size(0), device=DEVICE), center_pos
            ]
            preds = cl.argmax(dim=-1).cpu().numpy()
            all_labels.append(center_y.numpy())
            all_preds.append(preds)

    del model; clear_mem()
    return (np.concatenate(all_labels), np.concatenate(all_preds))


def plot_phase2_per_seed_cms():
    print("\n" + "=" * 60)
    print("SECTION 2 — Phase 2 per-seed confusion matrices")
    print("=" * 60)

    if not os.path.exists(TEST_NPZ):
        print(f"  [SKIP] test NPZ not found: {TEST_NPZ}")
        return

    for seed in SEEDS:
        labels, preds = _infer_seed(seed, TEST_NPZ)
        if labels is None:
            continue
        out = os.path.join(WORK_DIR, f"patch_phase2_cm_seed{seed}.png")
        plot_cm(labels, preds, f"PUMA-MIL-Lite — Seed {seed}\nTest Set Confusion Matrix", out)

    # Also plot ensemble CM from saved slice predictions CSV
    if os.path.exists(P2_SLICE_CSV):
        df = pd.read_csv(P2_SLICE_CSV)
        if {"label", "prob_1"}.issubset(df.columns):
            threshold = 0.5
            if os.path.exists(os.path.join(WORK_DIR, "aggregated_test_metrics_v3.json")):
                with open(os.path.join(WORK_DIR, "aggregated_test_metrics_v3.json")) as f:
                    agg = json.load(f)
                threshold = float(agg.get("ensemble_threshold", 0.5))
            labels = df["label"].astype(int).values
            preds  = (df["prob_1"].astype(float).values >= threshold).astype(int)
            out    = os.path.join(WORK_DIR, "patch_phase2_cm_ensemble.png")
            plot_cm(labels, preds, f"PUMA-MIL-Lite Ensemble\nTest Set Confusion Matrix (thresh={threshold:.2f})", out)


# ─────────────────────────────────────────────────────────────────────────────
#  SECTION 3 — PHASE 3: PER-BASELINE & PER-ABLATION CONFUSION MATRICES
# ─────────────────────────────────────────────────────────────────────────────
def plot_phase3_cms():
    print("\n" + "=" * 60)
    print("SECTION 3 — Phase 3 per-baseline & per-ablation CMs")
    print("=" * 60)

    for fname, tag in [
        (P3_BASELINE_PRED_JSON, "baseline"),
        (P3_ABLATION_PRED_JSON, "ablation"),
    ]:
        if not os.path.exists(fname):
            print(f"  [SKIP] {fname} not found")
            continue

        with open(fname) as f:
            pred_store = json.load(f)

        for name, store in pred_store.items():
            labels = np.asarray(store["label"],  dtype=int)
            preds  = np.asarray(store["pred"],   dtype=int)
            safe   = name.replace(" ", "_").replace("-", "_").replace("/", "_")
            out    = os.path.join(WORK_DIR, f"patch_phase3_cm_{tag}_{safe}.png")
            plot_cm(labels, preds, f"Phase 3 {tag.title()}: {name}\nTest Set Confusion Matrix", out)


# ─────────────────────────────────────────────────────────────────────────────
#  SECTION 4 — PHASE 3: PER-ABLATION TRAINING CURVES
# ─────────────────────────────────────────────────────────────────────────────
def plot_phase3_ablation_curves():
    print("\n" + "=" * 60)
    print("SECTION 4 — Phase 3 per-ablation training curves")
    print("=" * 60)

    if not os.path.exists(P3_HISTORIES_JSON):
        print(f"  [SKIP] {P3_HISTORIES_JSON} not found")
        return

    with open(P3_HISTORIES_JSON) as f:
        histories = json.load(f)       # dict: {variant_name: {seed, history: [rows]}}

    if not histories:
        print("  [SKIP] ablation histories JSON is empty")
        return

    for name, h in histories.items():
        if not isinstance(h, dict) or "history" not in h:
            continue
        df  = pd.DataFrame(h["history"])
        vdf = df.dropna(subset=["val_macro_f1"]) if "val_macro_f1" in df.columns else pd.DataFrame()

        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle(f"Phase 3 Ablation: {name}", fontsize=12, fontweight="bold")

        ax = axes[0]
        if "train_total_loss" in df.columns:
            ax.plot(df["epoch"], df["train_total_loss"], "b-o", lw=2.0, ms=5, label="Total Loss")
        for col, lbl, sty in [
            ("train_center_loss",     "Center",    "r--^"),
            ("train_bag_loss",        "Bag",       "g--s"),
            ("train_smoothness_loss", "Smooth",    "m--D"),
            ("train_diversity_loss",  "Diversity", "c--x"),
        ]:
            if col in df.columns:
                ax.plot(df["epoch"], df[col], sty, lw=1.2, ms=3, label=lbl)
        ax.set_title("Training Losses"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

        ax = axes[1]
        if len(vdf):
            if "val_macro_f1"   in vdf.columns: ax.plot(vdf["epoch"], vdf["val_macro_f1"],   "g-^", lw=2.0, ms=6, label="Val Macro-F1")
            if "val_bal_acc"    in vdf.columns: ax.plot(vdf["epoch"], vdf["val_bal_acc"],    "b-o", lw=2.0, ms=6, label="Val Bal-Acc")
            if "val_specificity" in vdf.columns: ax.plot(vdf["epoch"], vdf["val_specificity"],"r-s", lw=1.8, ms=5, label="Val Specificity")
        ax.set_title("Validation Metrics"); ax.set_xlabel("Epoch"); ax.set_ylabel("Score")
        ax.set_ylim(0, 1); ax.legend(fontsize=9); ax.grid(alpha=0.3)

        plt.tight_layout()
        safe = name.replace(" ", "_").replace("-", "_").replace("/", "_")
        out  = os.path.join(WORK_DIR, f"patch_phase3_curves_{safe}.png")
        plt.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
        plt.close()
        print(f"  ✓ saved: {os.path.basename(out)}")


# ─────────────────────────────────────────────────────────────────────────────
#  RUN ALL
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    ensure_dir(WORK_DIR)

    plot_phase2_per_seed_curves()   # Section 1 — Phase 2 training curves
    plot_phase2_per_seed_cms()      # Section 2 — Phase 2 per-seed CMs
    plot_phase3_cms()               # Section 3 — Phase 3 baseline + ablation CMs
    plot_phase3_ablation_curves()   # Section 4 — Phase 3 ablation training curves

    print("\n" + "=" * 60)
    print("✅  PATCH COMPLETE — all missing outputs generated.")
    print("=" * 60)

In [ ]:
# ============================================================
# PHASE 4A (V3 — CORRECTED): LOCKED EVALUATION & STATISTICS
# Fixes applied vs original Phase 4a:
#
#  FIX 1 — plot_subgroup_balacc():
#    • NaN balanced_accuracy (single-class groups) now filled with 0.0
#    • xlim set to (0, 1.05) so bars always render
#    • Value labels printed on each bar
#    • Reference line at 0.5 (random chance)
#    • Figure height scales with number of roles
#
#  FIX 2 — per-baseline & per-ablation confusion matrix PNGs:
#    • phase4a_cm_baseline_<name>.png saved for every Phase 3 baseline
#    • phase4a_cm_ablation_<name>.png saved for every Phase 3 ablation
#    • Uses aligned labels so CMs are always index-matched to Phase 2 test set
#
#  FIX 3 — plot_confusion_matrix() accepts dynamic title argument
#
#  Everything else is identical to the original Phase 4a.
# ============================================================

import os
import json
import math
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    log_loss,
    brier_score_loss,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


# ============================================================
# CONFIG
# ============================================================
class Config:
    WORK_DIR   = "/kaggle/working"
    PHASE2_DIR = "/kaggle/working"
    PHASE3_DIR = "/kaggle/working"

    BOOTSTRAP_N    = 2000
    BOOTSTRAP_SEED = 42
    RELIABILITY_BINS = 10
    FIG_DPI        = 300

    ROLE_MAP = {
        0: "far_neg",
        1: "boundary_neg",
        2: "positive",
        3: "mid_neg",
        4: "no_tumor",
    }


CFG = Config()


# ============================================================
# PATHS
# ============================================================
PHASE2_AGG_JSON       = os.path.join(CFG.PHASE2_DIR, "aggregated_test_metrics_v3.json")
PHASE2_PER_SEED_CSV   = os.path.join(CFG.PHASE2_DIR, "per_seed_test_metrics_v3.csv")
PHASE2_SLICE_CSV      = os.path.join(CFG.PHASE2_DIR, "slice_predictions_v3.csv")
PHASE2_PATIENT_CSV    = os.path.join(CFG.PHASE2_DIR, "patient_predictions_v3.csv")
PHASE2_AUX_PATIENT_JSON = os.path.join(CFG.PHASE2_DIR, "aux_patient_metrics_v3.json")

PHASE3_RESULTS_JSON       = os.path.join(CFG.PHASE3_DIR, "phase3_results_v8.json")
PHASE3_BASELINE_PRED_JSON = os.path.join(CFG.PHASE3_DIR, "phase3_baseline_predictions_v8.json")
PHASE3_ABLATION_PRED_JSON = os.path.join(CFG.PHASE3_DIR, "phase3_ablation_predictions_v8.json")

OUT_MAIN_CSV        = os.path.join(CFG.WORK_DIR, "phase4a_main_metrics_with_ci.csv")
OUT_SEED_CSV        = os.path.join(CFG.WORK_DIR, "phase4a_seed_summary.csv")
OUT_SUBGROUP_CSV    = os.path.join(CFG.WORK_DIR, "phase4a_subgroup_by_role.csv")
OUT_STATS_CSV       = os.path.join(CFG.WORK_DIR, "phase4a_mcnemar_vs_phase3.csv")
OUT_FAILURE_FP_CSV  = os.path.join(CFG.WORK_DIR, "phase4a_false_positives.csv")
OUT_FAILURE_FN_CSV  = os.path.join(CFG.WORK_DIR, "phase4a_false_negatives.csv")
OUT_RELIABILITY_CSV = os.path.join(CFG.WORK_DIR, "phase4a_reliability_bins.csv")
OUT_SELECTIVE_CSV   = os.path.join(CFG.WORK_DIR, "phase4a_selective_prediction.csv")
OUT_JSON            = os.path.join(CFG.WORK_DIR, "phase4a_comprehensive_results.json")

OUT_CM_PNG          = os.path.join(CFG.WORK_DIR, "phase4a_confusion_matrix.png")
OUT_ROC_PNG         = os.path.join(CFG.WORK_DIR, "phase4a_roc_curve.png")
OUT_PR_PNG          = os.path.join(CFG.WORK_DIR, "phase4a_pr_curve.png")
OUT_RELIABILITY_PNG = os.path.join(CFG.WORK_DIR, "phase4a_reliability_diagram.png")
OUT_SELECTIVE_PNG   = os.path.join(CFG.WORK_DIR, "phase4a_selective_prediction.png")
OUT_SEED_PNG        = os.path.join(CFG.WORK_DIR, "phase4a_seed_vs_ensemble.png")
OUT_SUBGROUP_PNG    = os.path.join(CFG.WORK_DIR, "phase4a_subgroup_balacc.png")


# ============================================================
# UTILS
# ============================================================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def load_json(path):
    with open(path, "r") as f:
        return json.load(f)


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=_json_default)


def _json_default(x):
    if isinstance(x, (np.integer,)):  return int(x)
    if isinstance(x, (np.floating,)): return float(x)
    if isinstance(x, (np.ndarray,)):  return x.tolist()
    return str(x)


def safe_auc(y_true, prob1):
    y_true = np.asarray(y_true); prob1 = np.asarray(prob1)
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return float("nan")
    try:
        return float(roc_auc_score(y_true, prob1))
    except Exception:
        return float("nan")


def safe_pr_auc(y_true, prob1):
    y_true = np.asarray(y_true); prob1 = np.asarray(prob1)
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return float("nan")
    try:
        return float(average_precision_score(y_true, prob1))
    except Exception:
        return float("nan")


def calculate_ece(probs, labels, n_bins=15):
    probs  = np.asarray(probs,  dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    if len(labels) == 0:
        return 0.0
    conf = np.max(probs, axis=1)
    pred = np.argmax(probs, axis=1)
    acc  = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (conf >= bins[i]) & (conf <= bins[i+1]) if i == 0 else (conf > bins[i]) & (conf <= bins[i+1])
        if mask.sum() > 0:
            ece += (mask.sum() / len(labels)) * abs(acc[mask].mean() - conf[mask].mean())
    return float(ece)


def probs_from_prob1(prob1):
    prob1 = np.clip(np.asarray(prob1, dtype=np.float64), 1e-7, 1 - 1e-7)
    return np.column_stack([1.0 - prob1, prob1])


def compute_metrics(labels, prob1, threshold):
    labels = np.asarray(labels, dtype=np.int64)
    prob1  = np.asarray(prob1,  dtype=np.float64)
    probs  = probs_from_prob1(prob1)
    preds  = (prob1 >= threshold).astype(np.int64)

    out = {
        "n":          int(len(labels)),
        "prevalence": float(labels.mean()) if len(labels) > 0 else float("nan"),
        "accuracy":   float(accuracy_score(labels, preds)) if len(labels) > 0 else float("nan"),
        "precision":  float(precision_score(labels, preds, average="macro", zero_division=0)),
        "recall":     float(recall_score(labels,    preds, average="macro", zero_division=0)),
        "f1":         float(f1_score(labels,        preds, average="macro", zero_division=0)),
        "auc":        safe_auc(labels, prob1),
        "pr_auc":     safe_pr_auc(labels, prob1),
        "ece":        float(calculate_ece(probs, labels)),
        "brier":      float(brier_score_loss(labels, prob1)) if len(labels) > 0 else float("nan"),
        "nll":        float(log_loss(labels, probs, labels=[0, 1])) if len(labels) > 0 else float("nan"),
        "balanced_accuracy": float(balanced_accuracy_score(labels, preds)) if len(np.unique(labels)) > 1 else float("nan"),
        "specificity": float(recall_score(labels, preds, pos_label=0, zero_division=0)),
        "sensitivity": float(recall_score(labels, preds, pos_label=1, zero_division=0)),
    }
    return out, preds, probs


def metric_value(labels, prob1, threshold, metric_name):
    m, _, _ = compute_metrics(labels, prob1, threshold)
    return m[metric_name]


def bootstrap_ci(labels, prob1, threshold, metric_name, n_boot=2000, seed=42):
    labels = np.asarray(labels, dtype=np.int64)
    prob1  = np.asarray(prob1,  dtype=np.float64)
    rng    = np.random.default_rng(seed)
    vals   = []

    for _ in range(n_boot):
        idx = rng.integers(0, len(labels), len(labels))
        yb, pb = labels[idx], prob1[idx]
        if metric_name in {"auc", "pr_auc", "balanced_accuracy"} and len(np.unique(yb)) < 2:
            continue
        try:
            vals.append(metric_value(yb, pb, threshold, metric_name))
        except Exception:
            continue

    vals = np.asarray(vals, dtype=np.float64)
    if vals.size == 0:
        return {"mean": float("nan"), "lower": float("nan"), "upper": float("nan"), "std": float("nan")}
    return {
        "mean":  float(np.mean(vals)),
        "lower": float(np.percentile(vals, 2.5)),
        "upper": float(np.percentile(vals, 97.5)),
        "std":   float(np.std(vals)),
    }


def build_reliability_table(probs, labels, n_bins=10):
    conf = np.max(probs, axis=1)
    pred = np.argmax(probs, axis=1)
    corr = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    rows = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (conf >= lo) & (conf <= hi) if i == 0 else (conf > lo) & (conf <= hi)
        n = int(mask.sum())
        if n == 0:
            continue
        rows.append({
            "bin_left": float(lo), "bin_right": float(hi), "n": n,
            "mean_confidence": float(conf[mask].mean()),
            "accuracy":        float(corr[mask].mean()),
            "gap":             float(abs(corr[mask].mean() - conf[mask].mean())),
        })
    return pd.DataFrame(rows)


def selective_curve(labels, prob1, threshold, coverages=None):
    if coverages is None:
        coverages = np.arange(0.50, 1.01, 0.05)
    labels     = np.asarray(labels, dtype=np.int64)
    prob1      = np.asarray(prob1,  dtype=np.float64)
    probs      = probs_from_prob1(prob1)
    confidence = np.max(probs, axis=1)
    order      = np.argsort(1.0 - confidence)
    rows = []
    for cov in coverages:
        k   = max(1, int(math.ceil(len(labels) * cov)))
        idx = order[:k]
        m, _, _ = compute_metrics(labels[idx], prob1[idx], threshold)
        rows.append({
            "coverage":         float(cov),
            "n_retained":       int(k),
            "accuracy":         float(m["accuracy"]),
            "macro_f1":         float(m["f1"]),
            "balanced_accuracy": float(m["balanced_accuracy"]) if not np.isnan(m["balanced_accuracy"]) else float("nan"),
        })
    return pd.DataFrame(rows)


def holm_bonferroni(pvals):
    pvals = np.asarray(pvals, dtype=np.float64)
    m     = len(pvals)
    order = np.argsort(pvals)
    adj   = np.empty(m, dtype=np.float64)
    running_max = 0.0
    for rank, idx in enumerate(order):
        v = (m - rank) * pvals[idx]
        running_max = max(running_max, v)
        adj[idx] = min(running_max, 1.0)
    return adj


def exact_mcnemar_pvalue(correct_a, correct_b):
    b = int(np.sum(correct_a & (~correct_b)))
    c = int(np.sum((~correct_a) & correct_b))
    n = b + c
    if n == 0:
        return b, c, 1.0
    try:
        from scipy.stats import binomtest
        p = float(binomtest(min(b, c), n=n, p=0.5, alternative="two-sided").pvalue)
    except Exception:
        p = 1.0
    return b, c, p


# ============================================================
# LOADERS
# ============================================================
def load_phase2_outputs():
    agg         = load_json(PHASE2_AGG_JSON)
    per_seed_df = pd.read_csv(PHASE2_PER_SEED_CSV)
    slice_df    = pd.read_csv(PHASE2_SLICE_CSV)

    patient_df   = pd.read_csv(PHASE2_PATIENT_CSV) if os.path.exists(PHASE2_PATIENT_CSV) else None
    aux_patient  = load_json(PHASE2_AUX_PATIENT_JSON) if os.path.exists(PHASE2_AUX_PATIENT_JSON) else None

    required_cols = {"patient_id", "slice_idx", "label", "prob_1", "pred"}
    missing = required_cols - set(slice_df.columns)
    if missing:
        raise ValueError(f"slice_predictions_v3.csv missing columns: {missing}")

    slice_df["patient_id"] = slice_df["patient_id"].astype(str)
    slice_df["slice_idx"]  = slice_df["slice_idx"].astype(int)
    slice_df["label"]      = slice_df["label"].astype(int)
    slice_df["pred"]       = slice_df["pred"].astype(int)
    slice_df["prob_1"]     = slice_df["prob_1"].astype(float)
    if "slice_role_id" in slice_df.columns:
        slice_df["slice_role_id"] = slice_df["slice_role_id"].astype(int)
        slice_df["slice_role"]    = slice_df["slice_role_id"].map(CFG.ROLE_MAP).fillna("unknown")

    threshold = float(agg.get("ensemble", {}).get("threshold", 0.50))
    return agg, per_seed_df, slice_df, patient_df, aux_patient, threshold


def load_phase3_outputs():
    results      = load_json(PHASE3_RESULTS_JSON)
    baseline_pred = load_json(PHASE3_BASELINE_PRED_JSON)
    ablation_pred = load_json(PHASE3_ABLATION_PRED_JSON)
    return results, baseline_pred, ablation_pred


def prediction_store_to_df(store_entry):
    return pd.DataFrame({
        "patient_id":  pd.Series(store_entry["patient_id"], dtype=str),
        "slice_idx":   pd.Series(store_entry["slice_idx"],  dtype=int),
        "label_cmp":   pd.Series(store_entry["label"],      dtype=int),
        "pred_cmp":    pd.Series(store_entry["pred"],       dtype=int),
        "prob_1_cmp":  pd.Series(store_entry["prob_1"],     dtype=float),
    })


def align_method_to_reference(ref_df, store_entry):
    cmp_df = prediction_store_to_df(store_entry)
    merged = ref_df[["patient_id", "slice_idx", "label"]].merge(
        cmp_df, on=["patient_id", "slice_idx"], how="left"
    )
    if merged["pred_cmp"].isna().any() or merged["prob_1_cmp"].isna().any():
        raise ValueError("Comparator predictions could not be fully aligned to Phase 2 reference.")
    if not np.array_equal(merged["label"].astype(int).values, merged["label_cmp"].astype(int).values):
        raise ValueError("Label mismatch after aligning comparator predictions.")
    return merged


# ============================================================
# PLOTS  — all functions
# ============================================================

# ── FIX 1: accepts dynamic title; safe 2×2 CM ──────────────
def plot_confusion_matrix(labels, preds, out_png, title="Phase 2 Ensemble — Confusion Matrix"):
    """Save a seaborn confusion matrix to out_png."""
    labels = np.asarray(labels, dtype=int)
    preds  = np.asarray(preds,  dtype=int)

    # Build safe 2×2 CM even if only one class is present
    cm = np.zeros((2, 2), dtype=int)
    for t, p in zip(labels, preds):
        cm[int(np.clip(t, 0, 1)), int(np.clip(p, 0, 1))] += 1

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Non-tumor", "Tumor"],
        yticklabels=["Non-tumor", "Tumor"],
        ax=ax
    )
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("True",      fontsize=11)
    ax.set_title(title,        fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_png, dpi=CFG.FIG_DPI, bbox_inches="tight")
    plt.close()


def plot_roc_curve(labels, prob1, comparator_curves, out_png):
    plt.figure(figsize=(7, 7))
    fpr, tpr, _ = roc_curve(labels, prob1)
    auc_val = roc_auc_score(labels, prob1)
    plt.plot(fpr, tpr, linewidth=2.8, label=f"Phase 2 Ensemble (AUC={auc_val:.3f})")

    for name, p in comparator_curves.items():
        if len(np.unique(labels)) < 2:
            continue
        fpr2, tpr2, _ = roc_curve(labels, p)
        auc2 = roc_auc_score(labels, p)
        plt.plot(fpr2, tpr2, linewidth=1.7, linestyle="--", label=f"{name} (AUC={auc2:.3f})")

    plt.plot([0, 1], [0, 1], "k--", linewidth=1)
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC Curve"); plt.legend(fontsize=9); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_png, dpi=CFG.FIG_DPI, bbox_inches="tight")
    plt.close()


def plot_pr_curve(labels, prob1, comparator_curves, out_png):
    plt.figure(figsize=(7, 7))
    p, r, _ = precision_recall_curve(labels, prob1)
    ap = average_precision_score(labels, prob1)
    plt.plot(r, p, linewidth=2.8, label=f"Phase 2 Ensemble (PR-AUC={ap:.3f})")

    for name, probs_c in comparator_curves.items():
        p2, r2, _ = precision_recall_curve(labels, probs_c)
        ap2 = average_precision_score(labels, probs_c)
        plt.plot(r2, p2, linewidth=1.7, linestyle="--", label=f"{name} (PR-AUC={ap2:.3f})")

    baseline = labels.mean()
    plt.axhline(baseline, color="gray", linestyle=":", linewidth=1.3, label=f"Prevalence={baseline:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title("Precision-Recall Curve"); plt.legend(fontsize=9); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_png, dpi=CFG.FIG_DPI, bbox_inches="tight")
    plt.close()


def plot_reliability_diagram(rel_df, out_png):
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], "k--", linewidth=1.5, label="Perfect calibration")
    plt.plot(rel_df["mean_confidence"].values, rel_df["accuracy"].values,
             "o-", linewidth=2, markersize=7, label="Phase 2 Ensemble")
    plt.xlabel("Confidence"); plt.ylabel("Accuracy")
    plt.title("Reliability Diagram"); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_png, dpi=CFG.FIG_DPI, bbox_inches="tight")
    plt.close()


def plot_selective_prediction(main_curve, comparator_curve, out_png):
    plt.figure(figsize=(8, 6))
    plt.plot(main_curve["coverage"], main_curve["accuracy"],
             "o-", linewidth=2.4, label="Phase 2 Ensemble")
    if comparator_curve is not None:
        plt.plot(comparator_curve["coverage"], comparator_curve["accuracy"],
                 "s--", linewidth=2.0, label="FullControl-MatchedBudget")
    plt.xlabel("Coverage"); plt.ylabel("Accuracy")
    plt.title("Selective Prediction"); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(out_png, dpi=CFG.FIG_DPI, bbox_inches="tight")
    plt.close()


def plot_seed_vs_ensemble(per_seed_df, ensemble_metrics, out_png):
    metrics = ["test_f1", "test_balanced_accuracy", "test_specificity", "test_auc"]
    display = {
        "test_f1":                "Macro-F1",
        "test_balanced_accuracy": "Balanced Accuracy",
        "test_specificity":       "Specificity",
        "test_auc":               "AUC",
    }
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
    for ax, metric in zip(axes, metrics):
        vals  = per_seed_df[metric].astype(float).values
        seeds = per_seed_df["seed"].astype(str).values
        ax.bar(range(len(vals)), vals, alpha=0.85, color="#4C78A8", label="Seeds")
        ens_val = ensemble_metrics.get(metric, float("nan"))
        if not np.isnan(ens_val):
            ax.axhline(ens_val, color="#D62728", linestyle="--", linewidth=2, label="Ensemble")
        ax.set_title(display[metric])
        ax.set_xticks(range(len(vals)))
        ax.set_xticklabels(seeds)
        ax.set_xlabel("Seed")
        lo = max(0.0, min(vals.min(), ens_val if not np.isnan(ens_val) else vals.min()) - 0.03)
        hi = min(1.0, max(vals.max(), ens_val if not np.isnan(ens_val) else vals.max()) + 0.03)
        ax.set_ylim(lo, hi)
        ax.grid(axis="y", alpha=0.3)
    handles, labels_l = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels_l, loc="upper center", ncol=2, frameon=False)
    fig.suptitle("Per-seed Stability vs Ensemble", y=1.03, fontsize=14)
    plt.tight_layout()
    plt.savefig(out_png, dpi=CFG.FIG_DPI, bbox_inches="tight")
    plt.close()


# ── FIX 2: completely rewritten subgroup plot ───────────────
def plot_subgroup_balacc(subgroup_df, out_png):
    """
    Horizontal bar chart of balanced-accuracy per slice role.

    Fixes vs original:
    - NaN balanced_accuracy (roles with only one class) → 0.0 instead of crashing
    - xlim(0, 1.05) so bars always render even when all values are small
    - Value labels on each bar
    - Reference line at 0.5 (random-chance baseline)
    - Figure height auto-scales with number of roles
    """
    plot_df = subgroup_df[["slice_role", "balanced_accuracy"]].copy()
    plot_df["balanced_accuracy"] = (
        pd.to_numeric(plot_df["balanced_accuracy"], errors="coerce").fillna(0.0)
    )
    plot_df = plot_df.sort_values("balanced_accuracy", ascending=True).reset_index(drop=True)

    n_roles = max(len(plot_df), 1)
    fig_h   = max(3.5, n_roles * 0.65)
    fig, ax = plt.subplots(figsize=(9, fig_h))

    bars = ax.barh(
        plot_df["slice_role"],
        plot_df["balanced_accuracy"],
        color="#59A14F", alpha=0.88, edgecolor="white", linewidth=0.6
    )

    # Reference line at random-chance (0.50)
    ax.axvline(0.5, color="#888888", linestyle="--", linewidth=1.3,
               alpha=0.75, label="Random chance (0.50)")

    # Value annotation on each bar
    for bar, val in zip(bars, plot_df["balanced_accuracy"]):
        x_pos = bar.get_width() + 0.012
        ax.text(
            x_pos, bar.get_y() + bar.get_height() / 2.0,
            f"{val:.3f}", va="center", ha="left", fontsize=9
        )

    ax.set_xlim(0.0, 1.08)
    ax.set_xlabel("Balanced Accuracy", fontsize=11)
    ax.set_ylabel("Slice Role",        fontsize=11)
    ax.set_title("Subgroup Performance by Slice Role", fontsize=13, fontweight="bold")
    ax.grid(axis="x", alpha=0.3)
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(out_png, dpi=CFG.FIG_DPI, bbox_inches="tight")
    plt.close()
    print(f"  ✓ {os.path.basename(out_png)}")


# ── FIX 3: per-baseline and per-ablation CMs ───────────────
def plot_all_comparator_cms(slice_df, phase3_baseline_pred, phase3_ablation_pred):
    """
    Save a separate confusion-matrix PNG for every baseline and ablation variant.
    Uses aligned labels so every CM is index-matched to the Phase 2 test set.
    """
    def _safe_name(s):
        return s.replace(" ", "_").replace("-", "_").replace("/", "_")

    for name, store in phase3_baseline_pred.items():
        try:
            aligned_df = align_method_to_reference(slice_df, store)
            labels = aligned_df["label"].astype(int).values
            preds  = aligned_df["pred_cmp"].astype(int).values
        except Exception as e:
            print(f"  [WARN] baseline CM skipped for {name}: {e}")
            labels = np.asarray(store["label"], dtype=int)
            preds  = np.asarray(store["pred"],  dtype=int)

        out = os.path.join(CFG.WORK_DIR, f"phase4a_cm_baseline_{_safe_name(name)}.png")
        plot_confusion_matrix(labels, preds, out,
                              title=f"Baseline: {name}\nTest Set Confusion Matrix")

    for name, store in phase3_ablation_pred.items():
        try:
            aligned_df = align_method_to_reference(slice_df, store)
            labels = aligned_df["label"].astype(int).values
            preds  = aligned_df["pred_cmp"].astype(int).values
        except Exception as e:
            print(f"  [WARN] ablation CM skipped for {name}: {e}")
            labels = np.asarray(store["label"], dtype=int)
            preds  = np.asarray(store["pred"],  dtype=int)

        tag = "fullcontrol" if name == "FullControl-MatchedBudget" else "ablation"
        out = os.path.join(CFG.WORK_DIR, f"phase4a_cm_{tag}_{_safe_name(name)}.png")
        plot_confusion_matrix(labels, preds, out,
                              title=f"Ablation: {name}\nTest Set Confusion Matrix")

    print("  ✓ per-baseline and per-ablation confusion matrices saved")


# ============================================================
# MAIN
# ============================================================
def run_phase4a():
    start = time.time()
    ensure_dir(CFG.WORK_DIR)

    print("\n" + "=" * 72)
    print("PHASE 4A (V3 — CORRECTED): LOCKED EVALUATION & STATISTICS")
    print("=" * 72)

    # ----------------------------
    # Load artifacts
    # ----------------------------
    agg, per_seed_df, slice_df, patient_df, aux_patient, threshold = load_phase2_outputs()
    phase3_results, phase3_baseline_pred, phase3_ablation_pred     = load_phase3_outputs()

    print(f"Loaded Phase 2 threshold: {threshold:.2f}")
    print(f"Loaded test slices:       {len(slice_df)}")
    print(f"Loaded Phase 3 baselines: {len(phase3_baseline_pred)}")
    print(f"Loaded Phase 3 ablations: {len(phase3_ablation_pred)}")

    # ----------------------------
    # Primary endpoint: slice-center
    # ----------------------------
    labels = slice_df["label"].astype(int).values
    prob1  = slice_df["prob_1"].astype(float).values

    main_metrics, preds, probs = compute_metrics(labels, prob1, threshold)

    metric_names = [
        "accuracy", "f1", "balanced_accuracy", "specificity",
        "sensitivity", "auc", "pr_auc", "ece", "brier", "nll"
    ]
    ci_dict = {
        m: bootstrap_ci(labels, prob1, threshold, m,
                        n_boot=CFG.BOOTSTRAP_N, seed=CFG.BOOTSTRAP_SEED)
        for m in metric_names
    }

    # ----------------------------
    # Auxiliary patient-level
    # ----------------------------
    patient_metrics = None
    if patient_df is not None and {"label", "prob_1"}.issubset(set(patient_df.columns)):
        patient_metrics, _, _ = compute_metrics(
            patient_df["label"].astype(int).values,
            patient_df["prob_1"].astype(float).values,
            0.50
        )

    # ----------------------------
    # Reliability & selective prediction
    # ----------------------------
    reliability_df   = build_reliability_table(probs, labels, n_bins=CFG.RELIABILITY_BINS)
    reliability_df.to_csv(OUT_RELIABILITY_CSV, index=False)

    main_selective_df       = selective_curve(labels, prob1, threshold)
    comparator_selective_df = None

    # ----------------------------
    # Comparator curves from Phase 3
    # ----------------------------
    comparator_curves = {}

    if "FullControl-MatchedBudget" in phase3_ablation_pred:
        full_df = align_method_to_reference(slice_df, phase3_ablation_pred["FullControl-MatchedBudget"])
        comparator_curves["FullControl"] = full_df["prob_1_cmp"].astype(float).values
        comparator_selective_df = selective_curve(
            full_df["label"].astype(int).values,
            full_df["prob_1_cmp"].astype(float).values,
            float(phase3_results["ablations"]["FullControl-MatchedBudget"]["threshold"])
        )

    baselines_block = phase3_results.get("baselines", {})
    if len(baselines_block) > 0:
        best_baseline_name = max(baselines_block, key=lambda k: baselines_block[k].get("f1", -1))
        base_df = align_method_to_reference(slice_df, phase3_baseline_pred[best_baseline_name])
        comparator_curves[f"BestBaseline:{best_baseline_name}"] = base_df["prob_1_cmp"].astype(float).values

    ablations_block = phase3_results.get("ablations", {})
    ablation_only   = {k: v for k, v in ablations_block.items() if k != "FullControl-MatchedBudget"}
    if len(ablation_only) > 0:
        best_ablation_name = max(ablation_only, key=lambda k: ablation_only[k].get("f1", -1))
        abl_df = align_method_to_reference(slice_df, phase3_ablation_pred[best_ablation_name])
        comparator_curves[f"BestAblation:{best_ablation_name}"] = abl_df["prob_1_cmp"].astype(float).values

    # ----------------------------
    # Subgroup analysis by role
    # ----------------------------
    subgroup_rows = []
    if "slice_role" in slice_df.columns:
        for role_name, g in slice_df.groupby("slice_role", sort=True):
            gm, _, _ = compute_metrics(
                g["label"].astype(int).values,
                g["prob_1"].astype(float).values,
                threshold
            )
            row = {"slice_role": role_name}
            row.update(gm)
            subgroup_rows.append(row)
    subgroup_df = pd.DataFrame(subgroup_rows)
    if len(subgroup_df) > 0:
        subgroup_df.to_csv(OUT_SUBGROUP_CSV, index=False)

    # ----------------------------
    # Statistical tests vs Phase 3
    # ----------------------------
    stat_rows          = []
    all_method_entries = []

    for name, store in phase3_baseline_pred.items():
        all_method_entries.append(("Baseline", name, store, phase3_results["baselines"][name]))
    for name, store in phase3_ablation_pred.items():
        all_method_entries.append(("Ablation", name, store, phase3_results["ablations"][name]))

    for family, name, store, result_block in all_method_entries:
        cmp_df    = align_method_to_reference(slice_df, store)
        cmp_prob1 = cmp_df["prob_1_cmp"].astype(float).values
        cmp_pred  = cmp_df["pred_cmp"].astype(int).values
        y         = cmp_df["label"].astype(int).values

        ens_correct = preds == y
        cmp_correct = cmp_pred == y
        b, c, p_raw = exact_mcnemar_pvalue(ens_correct, cmp_correct)

        cmp_threshold      = float(result_block.get("threshold", 0.50))
        cmp_metrics, _, _  = compute_metrics(y, cmp_prob1, cmp_threshold)

        stat_rows.append({
            "family":                             family,
            "method":                             name,
            "discordant_phase2_better":           int(b),
            "discordant_comparator_better":       int(c),
            "p_raw":                              float(p_raw),
            "phase2_macro_f1":                    float(main_metrics["f1"]),
            "method_macro_f1":                    float(cmp_metrics["f1"]),
            "delta_macro_f1_phase2_minus_method": float(main_metrics["f1"] - cmp_metrics["f1"]),
            "phase2_balacc":                      float(main_metrics["balanced_accuracy"]),
            "method_balacc":                      float(cmp_metrics["balanced_accuracy"]),
            "delta_balacc_phase2_minus_method":   float(main_metrics["balanced_accuracy"] - cmp_metrics["balanced_accuracy"])
                                                  if not (np.isnan(main_metrics["balanced_accuracy"]) or np.isnan(cmp_metrics["balanced_accuracy"])) else float("nan"),
            "phase2_specificity":                 float(main_metrics["specificity"]),
            "method_specificity":                 float(cmp_metrics["specificity"]),
            "delta_specificity_phase2_minus_method": float(main_metrics["specificity"] - cmp_metrics["specificity"]),
            "phase2_auc":  float(main_metrics["auc"])  if not np.isnan(main_metrics["auc"])  else float("nan"),
            "method_auc":  float(cmp_metrics["auc"])   if not np.isnan(cmp_metrics["auc"])   else float("nan"),
            "delta_auc_phase2_minus_method":
                float(main_metrics["auc"] - cmp_metrics["auc"])
                if not (np.isnan(main_metrics["auc"]) or np.isnan(cmp_metrics["auc"])) else float("nan"),
        })

    stats_df = pd.DataFrame(stat_rows)
    if len(stats_df) > 0:
        stats_df["p_holm"] = holm_bonferroni(stats_df["p_raw"].values.astype(float))
        stats_df["significance"] = stats_df["p_holm"].apply(
            lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
        )
        stats_df = stats_df.sort_values(
            ["family", "p_holm", "delta_macro_f1_phase2_minus_method"],
            ascending=[True, True, False]
        )
        stats_df.to_csv(OUT_STATS_CSV, index=False)

    # ----------------------------
    # Failure analysis
    # ----------------------------
    error_df          = slice_df.copy()
    error_df["pred_phase2"]  = preds.astype(int)
    error_df["confidence"]   = np.maximum(prob1, 1.0 - prob1)
    error_df["uncertainty"]  = 1.0 - error_df["confidence"]
    error_df["correct"]      = (error_df["pred_phase2"].values == error_df["label"].values).astype(int)

    fp_df = error_df[(error_df["label"] == 0) & (error_df["pred_phase2"] == 1)].sort_values(
        ["confidence", "prob_1"], ascending=[False, False]).head(25)
    fn_df = error_df[(error_df["label"] == 1) & (error_df["pred_phase2"] == 0)].sort_values(
        ["confidence", "prob_1"], ascending=[False, True]).head(25)

    fp_df.to_csv(OUT_FAILURE_FP_CSV, index=False)
    fn_df.to_csv(OUT_FAILURE_FN_CSV, index=False)

    # ----------------------------
    # Main metrics table
    # ----------------------------
    main_rows = []
    for m in metric_names:
        val = main_metrics[m]
        ci  = ci_dict[m]
        main_rows.append({
            "endpoint":        "slice_center_primary",
            "metric":          m,
            "value":           float(val)     if not np.isnan(val)     else float("nan"),
            "bootstrap_mean":  float(ci["mean"])  if not np.isnan(ci["mean"])  else float("nan"),
            "ci95_lower":      float(ci["lower"]) if not np.isnan(ci["lower"]) else float("nan"),
            "ci95_upper":      float(ci["upper"]) if not np.isnan(ci["upper"]) else float("nan"),
            "bootstrap_std":   float(ci["std"])   if not np.isnan(ci["std"])   else float("nan"),
        })

    if patient_metrics is not None:
        for k in ["accuracy", "f1", "balanced_accuracy", "specificity", "sensitivity", "auc", "pr_auc", "ece"]:
            main_rows.append({
                "endpoint": "patient_level_auxiliary", "metric": k,
                "value":           float(patient_metrics[k]) if not np.isnan(patient_metrics[k]) else float("nan"),
                "bootstrap_mean":  float("nan"), "ci95_lower": float("nan"),
                "ci95_upper":      float("nan"), "bootstrap_std": float("nan"),
            })

    pd.DataFrame(main_rows).to_csv(OUT_MAIN_CSV, index=False)

    # ----------------------------
    # Seed summary
    # ----------------------------
    seed_summary = per_seed_df.copy()
    if "test_f1" in seed_summary.columns:
        seed_summary["rank_by_test_f1"] = seed_summary["test_f1"].rank(ascending=False, method="dense")
    seed_summary.to_csv(OUT_SEED_CSV, index=False)

    # ----------------------------
    # Selective curve CSV
    # ----------------------------
    export_selective = main_selective_df.copy()
    if comparator_selective_df is not None:
        export_selective = export_selective.rename(columns={
            "accuracy":         "accuracy_phase2_ensemble",
            "macro_f1":         "macro_f1_phase2_ensemble",
            "balanced_accuracy": "balanced_accuracy_phase2_ensemble",
        })
        export_selective["accuracy_full_control"]          = comparator_selective_df["accuracy"].values
        export_selective["macro_f1_full_control"]          = comparator_selective_df["macro_f1"].values
        export_selective["balanced_accuracy_full_control"] = comparator_selective_df["balanced_accuracy"].values
    export_selective.to_csv(OUT_SELECTIVE_CSV, index=False)

    # ----------------------------
    # Plots — ensemble
    # ----------------------------
    print("\nSaving plots...")

    plot_confusion_matrix(labels, preds, OUT_CM_PNG,
                          title="PUMA-MIL-Lite Ensemble\nTest Set Confusion Matrix")
    print(f"  ✓ {os.path.basename(OUT_CM_PNG)}")

    plot_roc_curve(labels, prob1, comparator_curves, OUT_ROC_PNG)
    print(f"  ✓ {os.path.basename(OUT_ROC_PNG)}")

    plot_pr_curve(labels, prob1, comparator_curves, OUT_PR_PNG)
    print(f"  ✓ {os.path.basename(OUT_PR_PNG)}")

    plot_reliability_diagram(reliability_df, OUT_RELIABILITY_PNG)
    print(f"  ✓ {os.path.basename(OUT_RELIABILITY_PNG)}")

    plot_selective_prediction(main_selective_df, comparator_selective_df, OUT_SELECTIVE_PNG)
    print(f"  ✓ {os.path.basename(OUT_SELECTIVE_PNG)}")

    ensemble_metrics_for_plot = {
        "test_f1":                main_metrics["f1"],
        "test_balanced_accuracy": main_metrics["balanced_accuracy"],
        "test_specificity":       main_metrics["specificity"],
        "test_auc":               main_metrics["auc"],
    }
    plot_seed_vs_ensemble(per_seed_df, ensemble_metrics_for_plot, OUT_SEED_PNG)
    print(f"  ✓ {os.path.basename(OUT_SEED_PNG)}")

    # FIX 1 — subgroup bar chart (now robust)
    if len(subgroup_df) > 0:
        plot_subgroup_balacc(subgroup_df, OUT_SUBGROUP_PNG)

    # FIX 2 — per-baseline & per-ablation CMs
    print("\nSaving per-comparator confusion matrices...")
    plot_all_comparator_cms(slice_df, phase3_baseline_pred, phase3_ablation_pred)

    # ----------------------------
    # JSON summary
    # ----------------------------
    summary = {
        "phase4a_version":   "v3_corrected_q1_ready",
        "threshold":         float(threshold),
        "n_test_slices":     int(len(labels)),
        "n_baselines":       int(len(phase3_baseline_pred)),
        "n_ablations":       int(len(phase3_ablation_pred)),
        "primary_metrics":   {k: (float(v) if not np.isnan(v) else None)
                              for k, v in main_metrics.items()},
        "ci_95": {m: {
            "lower": float(ci_dict[m]["lower"]) if not np.isnan(ci_dict[m]["lower"]) else None,
            "upper": float(ci_dict[m]["upper"]) if not np.isnan(ci_dict[m]["upper"]) else None,
        } for m in metric_names},
        "patient_metrics":   {k: (float(v) if not np.isnan(v) else None)
                              for k, v in patient_metrics.items()} if patient_metrics else None,
        "runtime_minutes":   round((time.time() - start) / 60.0, 2),
    }
    save_json(summary, OUT_JSON)

    # ----------------------------
    # Summary printout
    # ----------------------------
    elapsed = (time.time() - start) / 60.0
    print(f"\n{'=' * 72}")
    print(f"✅  PHASE 4A (V3) COMPLETE  ({elapsed:.1f} min)")
    print(f"{'=' * 72}")
    print(f"Primary slice-centre metrics (threshold={threshold:.2f}):")
    print(f"  Macro-F1:  {main_metrics['f1']:.4f}  [{ci_dict['f1']['lower']:.4f}–{ci_dict['f1']['upper']:.4f}]")
    print(f"  Bal-Acc:   {main_metrics['balanced_accuracy']:.4f}")
    print(f"  AUC:       {main_metrics['auc']:.4f}  [{ci_dict['auc']['lower']:.4f}–{ci_dict['auc']['upper']:.4f}]")
    print(f"  Spec:      {main_metrics['specificity']:.4f}")
    print(f"  Sens:      {main_metrics['sensitivity']:.4f}")
    print(f"  ECE:       {main_metrics['ece']:.4f}")


if __name__ == "__main__":
    run_phase4a()

In [ ]:
# ============================================
# CELL 1: BRA TS2020 EXTERNAL FEATURE EXTRACTION
# Writes:
# - /kaggle/working/brats2020patientsummaryv3.csv
# - /kaggle/working/brats2020slicemetadatav3.csv
# - /kaggle/working/brats2020externalfeatures4modv3.npz
# ============================================

import os
import gc
import glob
import json
import time
import random
import warnings
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import nibabel as nib
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import timm

warnings.filterwarnings("ignore")

@dataclass
class Config:
    SEED: int = 42
    BRATS2020DIR: str = "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
    WORKDIR: str = "/kaggle/working"
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    NUMWORKERS: int = 2
    IMGSIZE: int = 224
    MODALITIES: tuple = ("t1", "t1ce", "t2", "flair")
    FEATUREDIM: int = 384
    BACKBONE: str = "vit_small_patch16_224"
    TUMORPIXELTHRESHOLD: int = 100
    MAXSELECTEDSLICES: int = 48
    MAXPOSITIVEFRACTION: float = 0.60
    BOUNDARYNEGRADIUS: int = 6
    FARNEGMINDISTANCE: int = 12
    TARGETBOUNDARYNEG: int = 12
    TARGETFARNEG: int = 10
    MINSELECTEDIFNOTUMOR: int = 24
    BATCHSIZE: int = 16

CFG = Config()

SLICEROLETOID = {"farneg": 0, "boundaryneg": 1, "positive": 2, "midneg": 3, "notumor": 4}
IDTOSLICEROLE = {v: k for k, v in SLICEROLETOID.items()}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def find_first(*patterns):
    for p in patterns:
        files = glob.glob(p)
        if files:
            return files[0]
    return None

def percentile_normalize(vol):
    vol = np.asarray(vol, dtype=np.float32)
    p1, p99 = np.percentile(vol, [1, 99])
    vol = np.clip(vol, p1, p99)
    mean, std = vol.mean(), vol.std()
    if std > 1e-6:
        vol = (vol - mean) / std
    vmin, vmax = vol.min(), vol.max()
    if (vmax - vmin) > 1e-6:
        vol = (vol - vmin) / (vmax - vmin)
    return vol.astype(np.float32)

def spread_sample_indices(indices, k):
    indices = sorted(set(int(x) for x in indices))
    if k <= 0 or len(indices) == 0:
        return []
    if len(indices) <= k:
        return indices
    pos = np.linspace(0, len(indices) - 1, k)
    picked, used = [], set()
    for p in pos:
        j = int(round(float(p)))
        j = max(0, min(j, len(indices) - 1))
        if j not in used:
            used.add(j)
            picked.append(indices[j])
    if len(picked) < k:
        for x in indices:
            if x not in picked:
                picked.append(x)
            if len(picked) == k:
                break
    return sorted(picked[:k])

def nearest_positive_distance(all_indices, pos_indices):
    if len(pos_indices) == 0:
        return {int(i): -1 for i in all_indices}
    pos_arr = np.asarray(sorted(pos_indices), dtype=np.int64)
    out = {}
    for i in all_indices:
        out[int(i)] = int(np.min(np.abs(pos_arr - int(i))))
    return out

def sample_notumor_indices(depth, k):
    k = min(int(k), int(depth))
    if k <= 0:
        return []
    centerk = min(k // 2, max(8, k // 3))
    mid = depth // 2
    half = centerk // 2
    center_block = list(range(max(0, mid - half), min(depth, mid - half + centerk)))
    remain = k - len(center_block)
    others = [i for i in range(depth) if i not in set(center_block)]
    spread = spread_sample_indices(others, remain)
    selected = sorted(set(center_block + spread))
    if len(selected) < k:
        for i in range(depth):
            if i not in selected:
                selected.append(i)
            if len(selected) == k:
                break
    return sorted(selected[:k])

def select_slices_v4(
    tumorpixels,
    threshold,
    maxselected,
    maxpositivefraction,
    boundarynegradius,
    farnegmindistance,
    targetboundaryneg,
    targetfarneg,
    minnotumor,
):
    tumorpixels = np.asarray(tumorpixels, dtype=np.int64)
    depth = len(tumorpixels)
    allidx = list(range(depth))
    pos = np.where(tumorpixels >= threshold)[0].astype(np.int64).tolist()

    if len(pos) == 0:
        selected = sample_notumor_indices(depth, min(minnotumor, maxselected))
        rolemap = {int(i): "notumor" for i in selected}
        distmap = {int(i): -1 for i in selected}
        meta = {
            "haspositive": 0,
            "firstpos": -1,
            "lastpos": -1,
            "roistart": -1,
            "roiend": -1,
            "nselected": int(len(selected)),
            "nselectedpositive": 0,
            "nselectedboundaryneg": 0,
            "nselectedfarneg": 0,
            "nselectedmidneg": 0,
            "nselectednotumor": int(len(selected)),
        }
        return selected, rolemap, distmap, meta

    firstpos, lastpos = int(min(pos)), int(max(pos))
    roistart = max(0, firstpos - boundarynegradius)
    roiend = min(depth - 1, lastpos + boundarynegradius)
    distmapfull = nearest_positive_distance(allidx, pos)

    neg = [i for i in allidx if i not in set(pos)]
    boundaryneg = [i for i in neg if distmapfull[i] <= boundarynegradius]
    farneg = [i for i in neg if distmapfull[i] >= farnegmindistance]
    midneg = [i for i in neg if i not in set(boundaryneg) and i not in set(farneg)]

    maxposkeep = max(1, int(round(maxselected * maxpositivefraction)))
    selpos = spread_sample_indices(pos, min(len(pos), maxposkeep))
    remaining = maxselected - len(selpos)

    boundarynegsorted = sorted(boundaryneg, key=lambda x: (distmapfull[x], abs(x - (firstpos + lastpos) / 2.0)))
    selboundary = boundarynegsorted[:min(targetboundaryneg, remaining, len(boundarynegsorted))]
    remaining -= len(selboundary)

    selfar = spread_sample_indices(farneg, min(targetfarneg, remaining, len(farneg)))
    remaining -= len(selfar)

    midsorted = sorted(midneg, key=lambda x: (distmapfull[x], abs(x - (firstpos + lastpos) / 2.0)))
    selmid = spread_sample_indices(midsorted, min(remaining, len(midsorted)))
    remaining -= len(selmid)

    selected = sorted(set(selpos + selboundary + selfar + selmid))

    if len(selected) < maxselected:
        remainingpool = [i for i in allidx if i not in set(selected)]
        fill = spread_sample_indices(remainingpool, maxselected - len(selected))
        selected = sorted(set(selected + fill))

    if len(selected) > maxselected:
        selected = spread_sample_indices(selected, maxselected)

    selset = set(selected)
    boundaryset = set(selboundary) & selset
    farset = set(selfar) & selset
    posset = set(selpos) & selset

    rolemap = {}
    for i in selected:
        if i in posset:
            rolemap[int(i)] = "positive"
        elif i in boundaryset:
            rolemap[int(i)] = "boundaryneg"
        elif i in farset:
            rolemap[int(i)] = "farneg"
        else:
            rolemap[int(i)] = "midneg"

    distmap = {int(i): int(distmapfull[i]) for i in selected}
    meta = {
        "haspositive": 1,
        "firstpos": int(firstpos),
        "lastpos": int(lastpos),
        "roistart": int(roistart),
        "roiend": int(roiend),
        "nselected": int(len(selected)),
        "nselectedpositive": int(sum(rolemap[i] == "positive" for i in selected)),
        "nselectedboundaryneg": int(sum(rolemap[i] == "boundaryneg" for i in selected)),
        "nselectedfarneg": int(sum(rolemap[i] == "farneg" for i in selected)),
        "nselectedmidneg": int(sum(rolemap[i] == "midneg" for i in selected)),
        "nselectednotumor": 0,
    }
    return selected, rolemap, distmap, meta

def list_patient_dirs(base_dir, tag="BraTS2020"):
    if tag == "BraTS2020":
        dirs = sorted(glob.glob(os.path.join(base_dir, "BraTS20*")))
        if not dirs:
            dirs = sorted(glob.glob(os.path.join(base_dir, "*", "BraTS20*")))
    else:
        dirs = []
    return [d for d in dirs if os.path.isdir(d)]

def get_bundle(patient_dir):
    bundle = {}
    for mod in CFG.MODALITIES:
        bundle[mod] = find_first(
            os.path.join(patient_dir, f"*{mod}.nii.gz"),
            os.path.join(patient_dir, f"*{mod}.nii"),
            os.path.join(patient_dir, f"{mod}.nii.gz"),
            os.path.join(patient_dir, f"{mod}.nii"),
        )
    bundle["seg"] = find_first(
        os.path.join(patient_dir, "*seg.nii.gz"),
        os.path.join(patient_dir, "*seg.nii"),
        os.path.join(patient_dir, "seg.nii.gz"),
        os.path.join(patient_dir, "seg.nii"),
    )
    return bundle

def build_metadata(dataset_dir, dataset_tag="BraTS2020"):
    patient_dirs = list_patient_dirs(dataset_dir, dataset_tag)
    print(f"Building metadata for {dataset_tag}: {len(patient_dirs)} patients")

    patient_rows, slice_rows = [], []
    rowid = 0

    for i, patient_dir in enumerate(tqdm(patient_dirs, desc=f"{dataset_tag} audit")):
        pid = os.path.basename(patient_dir)
        bundle = get_bundle(patient_dir)

        if bundle["seg"] is None:
            continue
        if any(bundle[m] is None for m in CFG.MODALITIES):
            continue

        try:
            seg = nib.load(bundle["seg"]).get_fdata()
        except Exception:
            continue

        depth = seg.shape[2]
        tumorpixels = [int(np.sum(seg[:, :, z] > 0)) for z in range(depth)]

        selected, rolemap, distmap, meta = select_slices_v4(
            tumorpixels=tumorpixels,
            threshold=CFG.TUMORPIXELTHRESHOLD,
            maxselected=CFG.MAXSELECTEDSLICES,
            maxpositivefraction=CFG.MAXPOSITIVEFRACTION,
            boundarynegradius=CFG.BOUNDARYNEGRADIUS,
            farnegmindistance=CFG.FARNEGMINDISTANCE,
            targetboundaryneg=CFG.TARGETBOUNDARYNEG,
            targetfarneg=CFG.TARGETFARNEG,
            minnotumor=CFG.MINSELECTEDIFNOTUMOR,
        )

        patient_rows.append({
            "patientid": pid,
            "patientpath": patient_dir,
            "depth": int(depth),
            "totaltumorvoxels": int(np.sum(tumorpixels)),
            "maxtumorpixels": int(np.max(tumorpixels)),
            "positiveslicesthr100": int(np.sum(np.array(tumorpixels) >= 100)),
            "selectedslices": int(meta["nselected"]),
            "selectedpositive": int(meta["nselectedpositive"]),
            "selectedboundaryneg": int(meta["nselectedboundaryneg"]),
            "selectedfarneg": int(meta["nselectedfarneg"]),
            "selectedmidneg": int(meta["nselectedmidneg"]),
            "selectednotumor": int(meta["nselectednotumor"]),
            "haspositive": int(meta["haspositive"]),
            "roistart": int(meta["roistart"]),
            "roiend": int(meta["roiend"]),
            "firstpos": int(meta["firstpos"]),
            "lastpos": int(meta["lastpos"]),
        })

        for orderidx, z in enumerate(selected):
            tp = int(tumorpixels[z])
            label = int(tp >= CFG.TUMORPIXELTHRESHOLD)
            role = rolemap[int(z)]
            slice_rows.append({
                "rowid": rowid,
                "patientid": pid,
                "patientpath": patient_dir,
                "sliceidx": int(z),
                "sliceorder": int(orderidx),
                "tumorpixels": tp,
                "label": label,
                "slicerole": role,
                "sliceroleid": int(SLICEROLETOID[role]),
                "distancetopositive": int(distmap[int(z)]),
                "roistart": int(meta["roistart"]),
                "roiend": int(meta["roiend"]),
                "haspositivepatient": int(meta["haspositive"]),
                "t1path": bundle["t1"],
                "t1cepath": bundle["t1ce"],
                "t2path": bundle["t2"],
                "flairpath": bundle["flair"],
                "segpath": bundle["seg"],
                "modalitymaskt1": 1,
                "modalitymaskt1ce": 1,
                "modalitymaskt2": 1,
                "modalitymaskflair": 1,
            })
            rowid += 1

        del seg
        if (i + 1) % 25 == 0:
            clear_memory()

    return pd.DataFrame(patient_rows), pd.DataFrame(slice_rows)

class FrozenViT(nn.Module):
    def __init__(self, model_name=CFG.BACKBONE):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.backbone.eval()
        for p in self.backbone.parameters():
            p.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            return self.backbone(x)

class SlicePreprocessor:
    def __init__(self):
        self.transform = transforms.Compose([
            transforms.Resize((CFG.IMGSIZE, CFG.IMGSIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def __call__(self, arr2d):
        arr2d = np.asarray(arr2d, dtype=np.float32)
        arr2d = np.clip(arr2d, 0.0, 1.0)
        img = Image.fromarray((arr2d * 255).astype(np.uint8)).convert("RGB")
        return self.transform(img)

def load_norm_vol(path):
    vol = nib.load(path).get_fdata()
    vol = percentile_normalize(vol)
    return vol.astype(np.float32)

def extract_features_for_split(df, split_name):
    print(f"Extracting features for {split_name}: {len(df)} slices")

    save_path = os.path.join(CFG.WORKDIR, f"{split_name}features4modv3.npz")
    if len(df) == 0:
        out = {
            "featurest1": np.zeros((0, CFG.FEATUREDIM), dtype=np.float32),
            "featurest1ce": np.zeros((0, CFG.FEATUREDIM), dtype=np.float32),
            "featurest2": np.zeros((0, CFG.FEATUREDIM), dtype=np.float32),
            "featuresflair": np.zeros((0, CFG.FEATUREDIM), dtype=np.float32),
            "labels": np.zeros((0,), dtype=np.int64),
            "patientids": np.array([], dtype=str),
            "sliceidxs": np.zeros((0,), dtype=np.int64),
            "modalitymask": np.zeros((0, 4), dtype=np.int64),
            "tumorpixels": np.zeros((0,), dtype=np.int64),
            "distancetopositive": np.zeros((0,), dtype=np.int64),
            "sliceroleid": np.zeros((0,), dtype=np.int64),
            "haspositivepatient": np.zeros((0,), dtype=np.int64),
        }
        np.savez_compressed(save_path, **out)
        return out

    model = FrozenViT().to(CFG.DEVICE)
    preproc = SlicePreprocessor()

    featst1, featst1ce, featst2, featsflair = [], [], [], []
    labels_all, patientids_all, sliceidxs_all = [], [], []
    modmasks_all, tumorpixels_all, distance_all, roleid_all, haspositive_all = [], [], [], [], []

    patient_groups = list(df.groupby("patientid", sort=True))
    for _, pdf in tqdm(patient_groups, desc=f"Features {split_name}"):
        pdf = pdf.sort_values("sliceidx").reset_index(drop=True)
        try:
            volt1 = load_norm_vol(pdf.loc[0, "t1path"])
            volt1ce = load_norm_vol(pdf.loc[0, "t1cepath"])
            volt2 = load_norm_vol(pdf.loc[0, "t2path"])
            volflair = load_norm_vol(pdf.loc[0, "flairpath"])
        except Exception:
            continue

        slice_indices = pdf["sliceidx"].astype(int).tolist()
        labels_np = pdf["label"].astype(np.int64).values
        modmasks_np = pdf[["modalitymaskt1", "modalitymaskt1ce", "modalitymaskt2", "modalitymaskflair"]].astype(np.int64).values
        tumorpixels_np = pdf["tumorpixels"].astype(np.int64).values
        distance_np = pdf["distancetopositive"].astype(np.int64).values
        roleid_np = pdf["sliceroleid"].astype(np.int64).values
        haspos_np = pdf["haspositivepatient"].astype(np.int64).values
        pid_np = pdf["patientid"].astype(str).values

        for start in range(0, len(slice_indices), CFG.BATCHSIZE):
            end = min(start + CFG.BATCHSIZE, len(slice_indices))
            idxs = slice_indices[start:end]

            batcht1 = torch.stack([preproc(volt1[:, :, z]) for z in idxs], dim=0).to(CFG.DEVICE)
            batcht1ce = torch.stack([preproc(volt1ce[:, :, z]) for z in idxs], dim=0).to(CFG.DEVICE)
            batcht2 = torch.stack([preproc(volt2[:, :, z]) for z in idxs], dim=0).to(CFG.DEVICE)
            batchflair = torch.stack([preproc(volflair[:, :, z]) for z in idxs], dim=0).to(CFG.DEVICE)

            ft1 = model(batcht1).cpu().numpy().astype(np.float32)
            ft1ce = model(batcht1ce).cpu().numpy().astype(np.float32)
            ft2 = model(batcht2).cpu().numpy().astype(np.float32)
            fflair = model(batchflair).cpu().numpy().astype(np.float32)

            featst1.append(ft1)
            featst1ce.append(ft1ce)
            featst2.append(ft2)
            featsflair.append(fflair)
            labels_all.append(labels_np[start:end])
            patientids_all.extend(pid_np[start:end].tolist())
            sliceidxs_all.append(np.asarray(idxs, dtype=np.int64))
            modmasks_all.append(modmasks_np[start:end].astype(np.int64))
            tumorpixels_all.append(tumorpixels_np[start:end].astype(np.int64))
            distance_all.append(distance_np[start:end].astype(np.int64))
            roleid_all.append(roleid_np[start:end].astype(np.int64))
            haspositive_all.append(haspos_np[start:end].astype(np.int64))

        del volt1, volt1ce, volt2, volflair
        clear_memory()

    out = {
        "featurest1": np.concatenate(featst1, axis=0).astype(np.float32),
        "featurest1ce": np.concatenate(featst1ce, axis=0).astype(np.float32),
        "featurest2": np.concatenate(featst2, axis=0).astype(np.float32),
        "featuresflair": np.concatenate(featsflair, axis=0).astype(np.float32),
        "labels": np.concatenate(labels_all, axis=0).astype(np.int64),
        "patientids": np.asarray(patientids_all),
        "sliceidxs": np.concatenate(sliceidxs_all, axis=0).astype(np.int64),
        "modalitymask": np.concatenate(modmasks_all, axis=0).astype(np.int64),
        "tumorpixels": np.concatenate(tumorpixels_all, axis=0).astype(np.int64),
        "distancetopositive": np.concatenate(distance_all, axis=0).astype(np.int64),
        "sliceroleid": np.concatenate(roleid_all, axis=0).astype(np.int64),
        "haspositivepatient": np.concatenate(haspositive_all, axis=0).astype(np.int64),
    }

    np.savez_compressed(save_path, **out)
    print(f"Saved: {save_path}")
    del model
    clear_memory()
    return out

ensure_dir(CFG.WORKDIR)
set_seed(CFG.SEED)

external_npz = os.path.join(CFG.WORKDIR, "brats2020externalfeatures4modv3.npz")
patient_csv = os.path.join(CFG.WORKDIR, "brats2020patientsummaryv3.csv")
slice_csv = os.path.join(CFG.WORKDIR, "brats2020slicemetadatav3.csv")

if os.path.exists(external_npz):
    print(f"Already exists: {external_npz}")
else:
    start = time.time()
    patientdf, slicedf = build_metadata(CFG.BRATS2020DIR, "BraTS2020")
    patientdf.to_csv(patient_csv, index=False)
    slicedf.to_csv(slice_csv, index=False)
    extract_features_for_split(slicedf, "brats2020external")
    manifest = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "dataset": "BraTS2020",
        "config": asdict(CFG),
        "outputs": [
            "brats2020patientsummaryv3.csv",
            "brats2020slicemetadatav3.csv",
            "brats2020externalfeatures4modv3.npz",
        ],
        "elapsed_min": round((time.time() - start) / 60.0, 2),
    }
    with open(os.path.join(CFG.WORKDIR, "brats2020external_manifestv3.json"), "w") as f:
        json.dump(manifest, f, indent=2)
    print("Done.")


In [ ]:
# ============================================
# CELL 2: LOCKED EXTERNAL VALIDATION
# Uses saved internal temperatures + frozen ensemble threshold
# Writes:
# - /kaggle/working/external_perseed_metrics_locked_v3.csv
# - /kaggle/working/external_aggregated_metrics_locked_v3.json
# - /kaggle/working/external_slice_predictions_locked_v3.csv
# - /kaggle/working/external_patient_predictions_locked_v3.csv
# - /kaggle/working/external_ensemble_probs_locked_v3.npz
# ============================================

import os
import gc
import json
import math
import random
import re
import glob
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    average_precision_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

class Config:
    WORKDIR = "/kaggle/working"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    DIN = 384
    DMODEL = 128
    MAXSLICES = 48
    TRAINWINDOW = 11
    TOPK = 4
    PROTOMOMENTUM = 0.90
    EVALBATCHSIZE = 64

CFG = Config()

SLICEROLETOID = {"farneg": 0, "boundaryneg": 1, "positive": 2, "midneg": 3, "notumor": 4}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_npz(path):
    with np.load(path, allow_pickle=True) as data:
        return {k: data[k] for k in data.files}

def zeros_if_missing(datadict, key, shape, dtype):
    if key in datadict:
        return datadict[key]
    return np.zeros(shape, dtype=dtype)

def safe_auc(labels, prob1):
    labels = np.asarray(labels)
    prob1 = np.asarray(prob1)
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return float("nan")
    try:
        return float(roc_auc_score(labels, prob1))
    except Exception:
        return float("nan")

def safe_prauc(labels, prob1):
    labels = np.asarray(labels)
    prob1 = np.asarray(prob1)
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return float("nan")
    try:
        return float(average_precision_score(labels, prob1))
    except Exception:
        return float("nan")

def calculate_ece(probs, labels, nbins=15):
    probs = np.asarray(probs)
    labels = np.asarray(labels)
    if len(labels) == 0:
        return 0.0
    conf = np.max(probs, axis=1)
    pred = np.argmax(probs, axis=1)
    acc = (pred == labels).astype(float)
    bins = np.linspace(0, 1, nbins + 1)
    ece = 0.0
    for i in range(nbins):
        m = (conf > bins[i]) & (conf <= bins[i + 1]) if i > 0 else (conf >= bins[i]) & (conf <= bins[i + 1])
        if m.sum() > 0:
            ece += (m.sum() / len(labels)) * abs(acc[m].mean() - conf[m].mean())
    return float(ece)

def metrics_from_probs(probs, labels, threshold=0.5):
    probs = np.asarray(probs)
    labels = np.asarray(labels)
    if len(labels) == 0:
        out = {
            "accuracy": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0,
            "auc": float("nan"),
            "prauc": float("nan"),
            "ece": 0.0,
            "balancedaccuracy": 0.0,
            "specificity": 0.0,
            "sensitivity": 0.0,
        }
        return out, np.zeros((0,), dtype=np.int64)

    preds = (probs[:, 1] >= threshold).astype(int)
    out = {
        "accuracy": float(accuracy_score(labels, preds)),
        "precision": float(precision_score(labels, preds, average="macro", zero_division=0)),
        "recall": float(recall_score(labels, preds, average="macro", zero_division=0)),
        "f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "auc": safe_auc(labels, probs[:, 1]),
        "prauc": safe_prauc(labels, probs[:, 1]),
        "ece": float(calculate_ece(probs, labels)),
        "balancedaccuracy": float(balanced_accuracy_score(labels, preds)) if len(np.unique(labels)) > 1 else 0.0,
        "specificity": float(recall_score(labels, preds, pos_label=0, zero_division=0)),
        "sensitivity": float(recall_score(labels, preds, pos_label=1, zero_division=0)),
    }
    return out, preds

def aggregate_probs_by_patient(probs, labels, pids, sliceidxs=None, topk=4):
    df = pd.DataFrame({
        "patientid": pids,
        "label": labels.astype(int),
        "prob1": probs[:, 1].astype(float),
    })
    df["sliceidx"] = sliceidxs.astype(int) if sliceidxs is not None else np.arange(len(df))

    rows = []
    for pid, g in df.groupby("patientid", sort=True):
        g = g.sort_values("sliceidx")
        vals = np.sort(g["prob1"].values)[::-1]
        k = min(int(topk), len(vals))
        p1 = float(vals[:k].mean()) if k > 0 else 0.5
        rows.append({
            "patientid": pid,
            "label": int(g["label"].max()),
            "prob0": float(1.0 - p1),
            "prob1": float(p1),
        })

    outdf = pd.DataFrame(rows)
    probspt = outdf[["prob0", "prob1"]].values.astype(np.float32)
    labelspt = outdf["label"].values.astype(np.int64)
    pidspt = outdf["patientid"].values.astype(str)
    return probspt, labelspt, pidspt, outdf

class CenterWindowDataset(Dataset):
    def __init__(self, datadict, windowsize=11):
        assert windowsize % 2 == 1
        self.windowsize = windowsize
        self.half = windowsize // 2

        patientids = np.asarray(datadict["patientids"]).astype(str)
        sliceidxs = np.asarray(datadict["sliceidxs"]).astype(np.int64)
        ft1 = np.asarray(datadict["featurest1"], dtype=np.float32)
        ft1ce = np.asarray(datadict["featurest1ce"], dtype=np.float32)
        ft2 = np.asarray(datadict["featurest2"], dtype=np.float32)
        flair = np.asarray(datadict["featuresflair"], dtype=np.float32)
        labels = np.asarray(datadict["labels"], dtype=np.int64)
        modmask = np.asarray(datadict["modalitymask"], dtype=np.int64)

        n = len(labels)
        dist = zeros_if_missing(datadict, "distancetopositive", (n,), np.int64).astype(np.int64)
        roleid = zeros_if_missing(datadict, "sliceroleid", (n,), np.int64).astype(np.int64)
        tumorpixels = zeros_if_missing(datadict, "tumorpixels", (n,), np.int64).astype(np.int64)
        haspositivepatient = zeros_if_missing(datadict, "haspositivepatient", (n,), np.int64).astype(np.int64)

        self.patientdata = {}
        pidtoindices = {}
        for i, pid in enumerate(patientids):
            pidtoindices.setdefault(pid, []).append(i)

        for pid in sorted(pidtoindices.keys()):
            idxs = np.asarray(pidtoindices[pid], dtype=np.int64)
            order = np.argsort(sliceidxs[idxs], kind="mergesort")
            idxs = idxs[order]
            self.patientdata[pid] = {
                "t1": ft1[idxs],
                "t1ce": ft1ce[idxs],
                "t2": ft2[idxs],
                "flair": flair[idxs],
                "labels": labels[idxs],
                "sliceidxs": sliceidxs[idxs],
                "modalitymask": modmask[idxs],
                "distancetopositive": dist[idxs],
                "sliceroleid": roleid[idxs],
                "tumorpixels": tumorpixels[idxs],
                "haspositivepatient": haspositivepatient[idxs],
            }

        self.index = []
        for pid in sorted(self.patientdata.keys()):
            y = self.patientdata[pid]["labels"]
            roles = self.patientdata[pid]["sliceroleid"]
            sx = self.patientdata[pid]["sliceidxs"]
            dd = self.patientdata[pid]["distancetopositive"]
            tp = self.patientdata[pid]["tumorpixels"]
            baglabel = int(y.max() > 0)
            for localidx in range(len(y)):
                self.index.append({
                    "pid": pid,
                    "localidx": int(localidx),
                    "centerlabel": int(y[localidx]),
                    "baglabel": baglabel,
                    "sliceroleid": int(roles[localidx]),
                    "sliceidx": int(sx[localidx]),
                    "distancetopositive": int(dd[localidx]),
                    "tumorpixels": int(tp[localidx]),
                })

    def __len__(self):
        return len(self.index)

    def build_window(self, arr, centeridx):
        out, mask = [], []
        for offset in range(-self.half, self.half + 1):
            j = centeridx + offset
            if 0 <= j < len(arr):
                out.append(arr[j])
                mask.append(True)
            else:
                out.append(np.zeros(arr.shape[1], dtype=np.float32))
                mask.append(False)
        return np.stack(out, axis=0), np.array(mask, dtype=bool)

    def __getitem__(self, idx):
        meta = self.index[idx]
        pid = meta["pid"]
        c = meta["localidx"]
        d = self.patientdata[pid]

        t1, seqmask = self.build_window(d["t1"], c)
        t1ce, _ = self.build_window(d["t1ce"], c)
        t2, _ = self.build_window(d["t2"], c)
        flair, _ = self.build_window(d["flair"], c)

        validmm = d["modalitymask"]
        bagmodmask = (validmm.sum(axis=0) > 0).astype(np.int64)
        if bagmodmask.sum() == 0:
            bagmodmask[:] = 1

        return (
            torch.tensor(t1, dtype=torch.float32),
            torch.tensor(t1ce, dtype=torch.float32),
            torch.tensor(t2, dtype=torch.float32),
            torch.tensor(flair, dtype=torch.float32),
            torch.tensor(int(meta["baglabel"]), dtype=torch.long),
            torch.tensor(int(meta["centerlabel"]), dtype=torch.long),
            torch.tensor(seqmask, dtype=torch.bool),
            torch.tensor(bagmodmask, dtype=torch.bool),
            torch.tensor(self.half, dtype=torch.long),
            torch.tensor(int(meta["sliceroleid"]), dtype=torch.long),
            str(pid),
            torch.tensor(int(meta["sliceidx"]), dtype=torch.long),
            torch.tensor(int(meta["distancetopositive"]), dtype=torch.long),
            torch.tensor(int(meta["tumorpixels"]), dtype=torch.long),
        )

def masked_topk_mean(seqfeats, scores, seqmask, topk):
    bsz, _, dim = seqfeats.shape
    out = []
    device = seqfeats.device
    dtype = seqfeats.dtype
    for b in range(bsz):
        valididx = torch.where(seqmask[b])[0]
        if valididx.numel() == 0:
            out.append(torch.zeros(dim, device=device, dtype=dtype))
            continue
        vals = scores[b, valididx]
        k = min(int(topk), int(valididx.numel()))
        chosen_local = torch.topk(vals, k=k, dim=0).indices
        chosen_idx = valididx[chosen_local]
        out.append(seqfeats[b, chosen_idx].mean(dim=0))
    return torch.stack(out, dim=0)

class ReliabilityFusion(nn.Module):
    def __init__(self, din=384, dmodel=128):
        super().__init__()
        self.proj = nn.ModuleList([
            nn.Sequential(nn.Linear(din, dmodel), nn.LayerNorm(dmodel), nn.GELU())
            for _ in range(4)
        ])
        self.rel = nn.ModuleList([nn.Linear(dmodel, 1) for _ in range(4)])
        self.out = nn.Sequential(nn.Linear(dmodel * 2, dmodel), nn.LayerNorm(dmodel), nn.GELU())

    def forward(self, mods, modmask):
        projfeats, rellogits = [], []
        for i in range(4):
            p = self.proj[i](mods[i])
            projfeats.append(p)
            rellogits.append(self.rel[i](p))
        featstack = torch.stack(projfeats, dim=2)
        relstack = torch.cat(rellogits, dim=-1)
        relstack = relstack.masked_fill(~modmask[:, None, :], -1e4)
        weights = torch.softmax(relstack, dim=-1)
        fused = (featstack * weights.unsqueeze(-1)).sum(dim=2)

        valid = modmask[:, None, :, None].float()
        meanfeat = (featstack * valid).sum(dim=2) / valid.sum(dim=2).clamp_min(1.0)
        out = self.out(torch.cat([fused, meanfeat], dim=-1))
        return out, weights

class HybridMILPool(nn.Module):
    def __init__(self, dmodel=128, topk=4):
        super().__init__()
        self.topk = topk
        self.attn = nn.Sequential(nn.Linear(dmodel, dmodel // 2), nn.Tanh(), nn.Linear(dmodel // 2, 1))

    def forward(self, seqfeats, slicelogits, seqmask):
        probs = torch.softmax(slicelogits, dim=-1)
        entropy = -torch.sum(probs * torch.log(probs.clamp_min(1e-8)), dim=-1) / math.log(2.0)
        reliability = 1.0 - entropy
        attnlogits = self.attn(seqfeats).squeeze(-1)
        attnlogits = attnlogits + torch.log(reliability.clamp_min(1e-6))
        attnlogits = attnlogits.masked_fill(~seqmask, -1e9)
        attnw = torch.softmax(attnlogits, dim=1)
        attnpool = torch.sum(seqfeats * attnw.unsqueeze(-1), dim=1)
        posfeat = masked_topk_mean(seqfeats, probs[:, :, 1], seqmask, self.topk)
        negfeat = masked_topk_mean(seqfeats, probs[:, :, 0], seqmask, self.topk)
        bagembed = torch.cat([attnpool, posfeat, negfeat], dim=-1)
        return bagembed, attnw, reliability

class PUMAMILLite(nn.Module):
    def __init__(self, din=384, dmodel=128, maxslices=48, topk=4, protomomentum=0.9):
        super().__init__()
        self.dmodel = dmodel
        self.maxslices = maxslices
        self.topk = topk
        self.protomomentum = protomomentum

        self.fusion = ReliabilityFusion(din=din, dmodel=dmodel)
        self.posembed = nn.Parameter(torch.zeros(1, maxslices, dmodel))

        encoderlayer = nn.TransformerEncoderLayer(
            d_model=dmodel,
            nhead=4,
            dim_feedforward=256,
            dropout=0.10,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoderlayer, num_layers=2)
        self.slicehead = nn.Sequential(nn.LayerNorm(dmodel), nn.Linear(dmodel, 2))
        self.pool = HybridMILPool(dmodel=dmodel, topk=topk)

        self.register_buffer("prototypes", torch.zeros(2, dmodel * 3))
        self.register_buffer("protocounts", torch.zeros(2))

        self.finalhead = nn.Sequential(
            nn.LayerNorm(dmodel * 3 + 2),
            nn.Linear(dmodel * 3 + 2, dmodel),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(dmodel, 2),
        )
        nn.init.trunc_normal_(self.posembed, std=0.02)

    def prototypesims(self, bagembed):
        sims = []
        for c in [0, 1]:
            if self.protocounts[c] <= 0 or torch.norm(self.prototypes[c]) < 1e-8:
                sims.append(torch.zeros(bagembed.size(0), 1, device=bagembed.device))
            else:
                s = F.cosine_similarity(bagembed, self.prototypes[c].unsqueeze(0), dim=-1).unsqueeze(-1)
                sims.append(s)
        return torch.cat(sims, dim=-1)

    def forward(self, t1, t1ce, t2, flair, seqmask, modmask, returnaux=False):
        assert t1.size(1) <= self.maxslices
        fused, modweights = self.fusion([t1, t1ce, t2, flair], modmask)
        x = fused + self.posembed[:, :t1.size(1), :]
        x = self.encoder(x, src_key_padding_mask=~seqmask)
        slicelogits = self.slicehead(x)
        bagembed, attnw, reliability = self.pool(x, slicelogits, seqmask)
        protos = self.prototypesims(bagembed)
        baglogits = self.finalhead(torch.cat([bagembed, protos], dim=-1))
        if returnaux:
            return baglogits, slicelogits, bagembed, modweights, attnw, reliability
        return baglogits

def evaluate_window_dataset(model, dataset, batchsize=64, temperature=1.0):
    model.eval()
    loader = DataLoader(dataset, batch_size=batchsize, shuffle=False, num_workers=0)

    all_logits, all_probs, all_labels = [], [], []
    all_pids, all_sliceidxs = [], []
    all_roles, all_dists, all_tumorpixels = [], [], []

    with torch.no_grad():
        for batch in loader:
            t1, t1ce, t2, flair, bagy, centery, seqmask, modmask, centerpos, roleid, pid, sliceidx, dist, tumorpixels = batch

            t1 = t1.to(CFG.DEVICE)
            t1ce = t1ce.to(CFG.DEVICE)
            t2 = t2.to(CFG.DEVICE)
            flair = flair.to(CFG.DEVICE)
            seqmask = seqmask.to(CFG.DEVICE)
            modmask = modmask.to(CFG.DEVICE)
            centerpos = centerpos.to(CFG.DEVICE)

            _, slicelogits, _, _, _, _ = model(t1, t1ce, t2, flair, seqmask, modmask, returnaux=True)
            centerlogits = slicelogits[torch.arange(slicelogits.size(0), device=slicelogits.device), centerpos]
            centerlogits = centerlogits / max(float(temperature), 1e-6)
            centerprobs = torch.softmax(centerlogits, dim=-1)

            all_logits.append(centerlogits.cpu().numpy())
            all_probs.append(centerprobs.cpu().numpy())
            all_labels.append(centery.numpy())
            all_pids.extend(list(pid))
            all_sliceidxs.append(sliceidx.numpy())
            all_roles.append(roleid.numpy())
            all_dists.append(dist.numpy())
            all_tumorpixels.append(tumorpixels.numpy())

    logits = np.concatenate(all_logits, axis=0) if all_logits else np.zeros((0, 2), dtype=np.float32)
    probs = np.concatenate(all_probs, axis=0) if all_probs else np.zeros((0, 2), dtype=np.float32)
    labels = np.concatenate(all_labels, axis=0) if all_labels else np.zeros((0,), dtype=np.int64)
    pids = np.asarray(all_pids).astype(str)
    sliceidxs = np.concatenate(all_sliceidxs, axis=0) if all_sliceidxs else np.zeros((0,), dtype=np.int64)
    roles = np.concatenate(all_roles, axis=0) if all_roles else np.zeros((0,), dtype=np.int64)
    dists = np.concatenate(all_dists, axis=0) if all_dists else np.zeros((0,), dtype=np.int64)
    tumorpixels = np.concatenate(all_tumorpixels, axis=0) if all_tumorpixels else np.zeros((0,), dtype=np.int64)

    return logits, probs, labels, pids, sliceidxs, roles, dists, tumorpixels

def unwrap_state_dict(state):
    if isinstance(state, dict):
        for key in ["state_dict", "model_state_dict", "model", "net"]:
            if key in state and isinstance(state[key], dict):
                return state[key]
    return state

def remap_checkpoint_keys(state_dict):
    fixed = {}
    for k, v in state_dict.items():
        if k.startswith("module."):
            k = k[len("module."):]

        k = k.replace("pos_embed", "posembed")
        k = k.replace("proto_counts", "protocounts")
        k = k.replace("slice_head", "slicehead")
        k = k.replace("final_head", "finalhead")

        fixed[k] = v
    return fixed

def load_checkpoint_compat(model, ckpt_path):
    state = torch.load(ckpt_path, map_location=CFG.DEVICE)
    state = unwrap_state_dict(state)
    state = remap_checkpoint_keys(state)

    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        raise RuntimeError(
            f"Checkpoint mismatch after remap for {ckpt_path}\n"
            f"Missing: {missing}\n"
            f"Unexpected: {unexpected}"
        )
    model.eval()
    return model

def resolve_checkpoint_path(seed):
    seed = int(seed)

    preferred = [
        os.path.join(CFG.WORKDIR, f"pumamil_best_seed{seed}.pth"),
        os.path.join(CFG.WORKDIR, f"pumamilbestseed{seed}.pth"),
        os.path.join(CFG.WORKDIR, f"pumamil_bestseed{seed}.pth"),
        os.path.join(CFG.WORKDIR, f"pumamilbest_seed{seed}.pth"),
    ]

    for p in preferred:
        if os.path.exists(p):
            return p

    if "detected_checkpoints" in globals():
        for p in detected_checkpoints:
            name = os.path.basename(p)
            m = re.search(r"seed(\d+)", name)
            if m and int(m.group(1)) == seed and os.path.exists(p):
                return p

    globbed = glob.glob(os.path.join(CFG.WORKDIR, f"*seed{seed}*.pth"))
    for p in sorted(globbed):
        name = os.path.basename(p)
        m = re.search(r"seed(\d+)", name)
        if m and int(m.group(1)) == seed:
            return p

    raise FileNotFoundError(f"No checkpoint found for seed {seed}")

set_seed(42)

external_path = os.path.join(CFG.WORKDIR, "brats2020externalfeatures4modv3.npz")
perseed_path = os.path.join(CFG.WORKDIR, "per_seed_test_metrics_v3.csv")
agg_path = os.path.join(CFG.WORKDIR, "aggregated_test_metrics_v3.json")

assert os.path.exists(external_path), f"Missing: {external_path}"
assert os.path.exists(perseed_path), f"Missing: {perseed_path}"
assert os.path.exists(agg_path), f"Missing: {agg_path}"

external_data = load_npz(external_path)
metrics_df = pd.read_csv(perseed_path)
with open(agg_path, "r") as f:
    agg = json.load(f)

if isinstance(agg.get("ensemble"), dict):
    frozen_threshold = float(agg["ensemble"]["threshold"])
else:
    frozen_threshold = float(metrics_df["threshold"].mean())

dataset = CenterWindowDataset(external_data, windowsize=CFG.TRAINWINDOW)

perseed_rows = []
all_probs = []
resolved_checkpoint_files = []

labels_ref = None
pids_ref = None
sliceidxs_ref = None
roles_ref = None
dists_ref = None
tumorpixels_ref = None

for _, row in metrics_df.iterrows():
    seed = int(row["seed"])
    temperature = float(row["temperature"])
    ckpt_path = resolve_checkpoint_path(seed)
    resolved_checkpoint_files.append(os.path.basename(ckpt_path))

    model = PUMAMILLite(
        din=CFG.DIN,
        dmodel=CFG.DMODEL,
        maxslices=CFG.MAXSLICES,
        topk=CFG.TOPK,
        protomomentum=CFG.PROTOMOMENTUM,
    ).to(CFG.DEVICE)

    model = load_checkpoint_compat(model, ckpt_path)

    logits, probs, labels, pids, sliceidxs, roles, dists, tumorpixels = evaluate_window_dataset(
        model, dataset, batchsize=CFG.EVALBATCHSIZE, temperature=temperature
    )
    met, preds = metrics_from_probs(probs, labels, threshold=frozen_threshold)

    perseed_rows.append({
        "seed": seed,
        "checkpoint_file": os.path.basename(ckpt_path),
        "temperature": temperature,
        "frozen_threshold": frozen_threshold,
        "accuracy": float(met["accuracy"]),
        "precision": float(met["precision"]),
        "recall": float(met["recall"]),
        "macro_f1": float(met["f1"]),
        "auc": float(met["auc"]) if not np.isnan(met["auc"]) else float("nan"),
        "prauc": float(met["prauc"]) if not np.isnan(met["prauc"]) else float("nan"),
        "balanced_accuracy": float(met["balancedaccuracy"]),
        "specificity": float(met["specificity"]),
        "sensitivity": float(met["sensitivity"]),
        "ece": float(met["ece"]),
    })

    all_probs.append(probs)

    if labels_ref is None:
        labels_ref = labels
        pids_ref = pids
        sliceidxs_ref = sliceidxs
        roles_ref = roles
        dists_ref = dists
        tumorpixels_ref = tumorpixels
    else:
        assert np.array_equal(labels_ref, labels)
        assert np.array_equal(pids_ref, pids)
        assert np.array_equal(sliceidxs_ref, sliceidxs)

    del model
    clear_memory()

avg_probs = np.mean(all_probs, axis=0)
ensmetrics, enspreds = metrics_from_probs(avg_probs, labels_ref, threshold=frozen_threshold)

slice_pred_df = pd.DataFrame({
    "patientid": pids_ref,
    "sliceidx": sliceidxs_ref.astype(int),
    "label": labels_ref.astype(int),
    "prob0": avg_probs[:, 0].astype(float),
    "prob1": avg_probs[:, 1].astype(float),
    "pred": enspreds.astype(int),
    "sliceroleid": roles_ref.astype(int),
    "distancetopositive": dists_ref.astype(int),
    "tumorpixels": tumorpixels_ref.astype(int),
})
slice_pred_df.to_csv(os.path.join(CFG.WORKDIR, "external_slice_predictions_locked_v3.csv"), index=False)

patient_probs, patient_labels, patient_ids, patient_df = aggregate_probs_by_patient(
    avg_probs, labels_ref, pids_ref, sliceidxs_ref, topk=CFG.TOPK
)
patient_aux_metrics, patient_preds = metrics_from_probs(patient_probs, patient_labels, threshold=0.50)
patient_df["pred"] = patient_preds.astype(int)
patient_df.to_csv(os.path.join(CFG.WORKDIR, "external_patient_predictions_locked_v3.csv"), index=False)

pd.DataFrame(perseed_rows).to_csv(os.path.join(CFG.WORKDIR, "external_perseed_metrics_locked_v3.csv"), index=False)

np.savez_compressed(
    os.path.join(CFG.WORKDIR, "external_ensemble_probs_locked_v3.npz"),
    probs=avg_probs,
    labels=labels_ref,
    pids=pids_ref,
    sliceidxs=sliceidxs_ref,
    sliceroleid=roles_ref,
    distancetopositive=dists_ref,
    tumorpixels=tumorpixels_ref,
)

summary = {
    "protocol": {
        "type": "locked_external_validation",
        "external_dataset": "BraTS2020",
        "external_features": "brats2020externalfeatures4modv3.npz",
        "checkpoint_files": resolved_checkpoint_files,
        "per_seed_temperature_source": "per_seed_test_metrics_v3.csv",
        "frozen_threshold_source": "aggregated_test_metrics_v3.json",
        "frozen_threshold": float(frozen_threshold),
        "retrained_on_external": False,
        "retuned_on_external": False,
    },
    "slice_level_external": {
        "accuracy": float(ensmetrics["accuracy"]),
        "precision": float(ensmetrics["precision"]),
        "recall": float(ensmetrics["recall"]),
        "macro_f1": float(ensmetrics["f1"]),
        "auc": float(ensmetrics["auc"]) if not np.isnan(ensmetrics["auc"]) else float("nan"),
        "prauc": float(ensmetrics["prauc"]) if not np.isnan(ensmetrics["prauc"]) else float("nan"),
        "balanced_accuracy": float(ensmetrics["balancedaccuracy"]),
        "specificity": float(ensmetrics["specificity"]),
        "sensitivity": float(ensmetrics["sensitivity"]),
        "ece": float(ensmetrics["ece"]),
        "confusion_matrix": confusion_matrix(labels_ref, enspreds).tolist(),
    },
    "patient_level_auxiliary": {
        "threshold": 0.50,
        "accuracy": float(patient_aux_metrics["accuracy"]),
        "precision": float(patient_aux_metrics["precision"]),
        "recall": float(patient_aux_metrics["recall"]),
        "macro_f1": float(patient_aux_metrics["f1"]),
        "auc": float(patient_aux_metrics["auc"]) if not np.isnan(patient_aux_metrics["auc"]) else float("nan"),
        "prauc": float(patient_aux_metrics["prauc"]) if not np.isnan(patient_aux_metrics["prauc"]) else float("nan"),
        "balanced_accuracy": float(patient_aux_metrics["balancedaccuracy"]),
        "specificity": float(patient_aux_metrics["specificity"]),
        "sensitivity": float(patient_aux_metrics["sensitivity"]),
        "ece": float(patient_aux_metrics["ece"]),
    },
}

with open(os.path.join(CFG.WORKDIR, "external_aggregated_metrics_locked_v3.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 72)
print("LOCKED EXTERNAL VALIDATION COMPLETE")
print("=" * 72)
print(f"Frozen ensemble threshold: {frozen_threshold:.4f}")
print(f"External Slice Accuracy: {ensmetrics['accuracy']:.4f}")
print(f"External Slice Macro-F1: {ensmetrics['f1']:.4f}")
print(f"External Slice Balanced Accuracy: {ensmetrics['balancedaccuracy']:.4f}")
print(f"External Slice Specificity: {ensmetrics['specificity']:.4f}")
print(f"External Slice Sensitivity: {ensmetrics['sensitivity']:.4f}")
print(f"External Slice PR-AUC: {0.0 if np.isnan(ensmetrics['prauc']) else ensmetrics['prauc']:.4f}")
print(f"External Slice AUC: {0.0 if np.isnan(ensmetrics['auc']) else ensmetrics['auc']:.4f}")
print("Confusion Matrix:", confusion_matrix(labels_ref, enspreds).tolist())
print("- external_perseed_metrics_locked_v3.csv")
print("- external_aggregated_metrics_locked_v3.json")
print("- external_slice_predictions_locked_v3.csv")
print("- external_patient_predictions_locked_v3.csv")
print("- external_ensemble_probs_locked_v3.npz")
print("=" * 72)


In [ ]:
# ============================================================
# PHASE4B_IMAGE_INTERPRETABILITY.py
# PROPER image-level GradCAM, GradCAM++, LIME, SHAP
# as shown in the paper figures (heatmaps overlaid on MRI slices)
#
# HOW IT WORKS
# ─────────────────────────────────────────────────────────────
# Your model pipeline is:
#   raw MRI slice  →  ViT backbone  →  384-d feature  →  PUMA-MIL-Lite
#
# For image-level explanations we run the ViT + PUMA end-to-end on the
# actual raw MRI pixel arrays and compute gradients / perturbations at
# the INPUT IMAGE level — exactly as in the paper figures you showed.
#
# OUTPUTS (one figure grid per explanation method):
#   interp_gradcam.png        — 3-col grid: MRI | GradCAM | GradCAM++
#   interp_lime.png           — 2-col grid: MRI | LIME superpixels
#   interp_shap.png           — 3-col grid: MRI | SHAP(pos) | SHAP(neg)
#
# Each row in the grid = one MRI slice (2 positive + 2 negative samples)
# across all 4 modalities (T1, T1ce, T2, FLAIR) for context.
# ============================================================

import os, gc, json, math, warnings, glob
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
import timm

warnings.filterwarnings("ignore")

# ── CONFIG ─────────────────────────────────────────────────────────────────
WORK_DIR     = "/kaggle/working"
BACKBONE     = "vit_small_patch16_224"
IMG_SIZE     = 224
PATCH_SIZE   = 16
N_PATCHES_1D = IMG_SIZE // PATCH_SIZE    # 14
FEATURE_DIM  = 384
D_MODEL      = 128
MAX_SLICES   = 48
TOP_K        = 4
FIG_DPI      = 300
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

# Slice CSV from Phase 1 — contains raw NIfTI paths per slice
SLICE_META_CSV = os.path.join(WORK_DIR, "slice_metadata_v3.csv")

# Pre-extracted feature NPZ from Phase 1 (used to build proper windows)
_TEST_NPZ_CANDIDATES = [
    os.path.join(WORK_DIR, "test_features_4mod_v3.npz"),
    os.path.join(WORK_DIR, "testfeatures4modv3.npz"),
]
TEST_NPZ = next((p for p in _TEST_NPZ_CANDIDATES if os.path.exists(p)),
                _TEST_NPZ_CANDIDATES[0])

TRAIN_WINDOW = 11   # must match Phase 2 CFG.TRAIN_WINDOW

# Checkpoint
CKPT_CANDIDATES = [
    os.path.join(WORK_DIR, "pumamil_best_v3.pth"),
    os.path.join(WORK_DIR, "pumamilbestv3.pth"),
    os.path.join(WORK_DIR, "pumamil_best_seed42.pth"),
]

# How many samples per class to show
N_POS = 3   # positive (tumor) slices
N_NEG = 3   # negative (non-tumor) slices

MODALITIES = ["t1", "t1ce", "t2", "flair"]   # lowercase, matches CSV cols

# ── OUTPUTS ────────────────────────────────────────────────────────────────
OUT_GRADCAM   = os.path.join(WORK_DIR, "interp_gradcam.png")
OUT_GRADCAMPP = os.path.join(WORK_DIR, "interp_gradcampp.png")
OUT_LIME      = os.path.join(WORK_DIR, "interp_lime.png")
OUT_SHAP      = os.path.join(WORK_DIR, "interp_shap.png")

# ── TAR PATHS ──────────────────────────────────────────────────────────────
# BraTS 2021 tar lives in Kaggle input; extracted files go to WORK_DIR so
# they persist across sessions (unlike /kaggle/temp which is wiped on restart)
BRATS_TAR      = "/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_Training_Data.tar"
BRATS_EXTRACT  = os.path.join(WORK_DIR, "brats2021_extracted")   # persistent

# ── HELPERS ────────────────────────────────────────────────────────────────
def ensure_dir(p): os.makedirs(p, exist_ok=True)
def clear_mem():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


def ensure_brats_extracted():
    """
    Extract only the patient folders needed by the selected samples
    from the tar into BRATS_EXTRACT (inside /kaggle/working, which
    persists between sessions).

    Returns the extraction root so callers can remap paths.
    """
    ensure_dir(BRATS_EXTRACT)
    # Check if already extracted (at least one patient folder present)
    existing = glob.glob(os.path.join(BRATS_EXTRACT, "BraTS2021_*"))
    if existing:
        print(f"  [TAR] already extracted: {len(existing)} patient folders in {BRATS_EXTRACT}")
        return BRATS_EXTRACT

    if not os.path.exists(BRATS_TAR):
        raise FileNotFoundError(
            f"BraTS 2021 tar not found at {BRATS_TAR}\n"
            "Please make sure the dataset is attached to your Kaggle notebook."
        )

    print(f"  [TAR] Extracting BraTS 2021 from tar → {BRATS_EXTRACT} ...")
    print("        (this takes ~3-5 min, runs once, then cached in /kaggle/working)")
    import tarfile, time
    t0 = time.time()
    with tarfile.open(BRATS_TAR, "r") as tf:
        tf.extractall(BRATS_EXTRACT)
    elapsed = time.time() - t0
    extracted = glob.glob(os.path.join(BRATS_EXTRACT, "BraTS2021_*"))
    # Sometimes tar puts everything inside a subdirectory
    if not extracted:
        extracted = glob.glob(os.path.join(BRATS_EXTRACT, "*", "BraTS2021_*"))
    print(f"  [TAR] Done in {elapsed/60:.1f} min — {len(extracted)} patient folders")
    return BRATS_EXTRACT


def repair_paths(df, extract_root):
    """
    The CSV was written with paths pointing to /kaggle/temp/brats2021_extracted/...
    which is wiped between sessions.  Re-map to the persistent extract_root.

    Handles both:
      /kaggle/temp/brats2021_extracted/BraTS2021_00001/BraTS2021_00001_t1.nii.gz
      /kaggle/working/brats2021_extracted/BraTS2021_00001/BraTS2021_00001_t1.nii.gz
    """
    path_cols = [c for c in ["t1_path","t1ce_path","t2_path","flair_path","seg_path"]
                 if c in df.columns]
    if not path_cols:
        return df

    df = df.copy()
    for col in path_cols:
        def _remap(p):
            p = str(p)
            # Extract the patient_id and filename from the end of the old path
            # Works regardless of old prefix (/kaggle/temp/... or /kaggle/working/...)
            parts = p.replace("\\", "/").split("/")
            # Find the BraTS2021_XXXXX part
            for i, part in enumerate(parts):
                if part.startswith("BraTS2021_") and i + 1 < len(parts):
                    pid_dir  = part                 # e.g. BraTS2021_00001
                    filename = parts[i + 1]         # e.g. BraTS2021_00001_t1.nii.gz
                    # Try direct: EXTRACT/BraTS2021_XXXXX/filename
                    cand1 = os.path.join(extract_root, pid_dir, filename)
                    if os.path.exists(cand1):
                        return cand1
                    # Try one level deeper (tar may have a subfolder)
                    for sub in os.listdir(extract_root):
                        cand2 = os.path.join(extract_root, sub, pid_dir, filename)
                        if os.path.exists(cand2):
                            return cand2
            return p   # return original if remapping failed (will be filtered out)
        df[col] = df[col].apply(_remap)
    return df


def percentile_normalize(vol):
    lo = np.percentile(vol[vol > 0], 1) if np.any(vol > 0) else 0.0
    hi = np.percentile(vol[vol > 0], 99) if np.any(vol > 0) else 1.0
    if hi == lo: return np.zeros_like(vol, dtype=np.float32)
    return np.clip((vol - lo) / (hi - lo + 1e-8), 0.0, 1.0).astype(np.float32)


def load_slice(nii_path, slice_idx):
    """Load one 2D slice from a NIfTI volume, percentile-normalised to [0,1]."""
    vol = nib.load(nii_path).get_fdata().astype(np.float32)
    vol = percentile_normalize(vol)
    return vol[:, :, int(slice_idx)]      # (H, W) float32 in [0, 1]


def arr2d_to_tensor(arr2d):
    """Convert a [0,1] float32 2D array → ImageNet-normalised (1,3,224,224) tensor."""
    arr2d = np.clip(arr2d, 0.0, 1.0)
    img   = Image.fromarray((arr2d * 255).astype(np.uint8)).convert("RGB")
    tfm   = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return tfm(img).unsqueeze(0)           # (1, 3, 224, 224)


def arr2d_to_display(arr2d):
    """Resize 2D MRI slice to 224×224 for display."""
    h, w = arr2d.shape
    return cv2.resize(arr2d, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)


def overlay_heatmap(base_img, cam, alpha=0.65, colormap=cv2.COLORMAP_JET, gamma=0.4):
    """
    Overlay a [0,1] CAM heatmap onto a [0,1] grayscale image.
    Uses gamma correction to suppress low activations and make
    hot regions pop with vivid jet colors (red/yellow = high, blue = low).
    Returns an RGB float [0,1] array of shape (H, W, 3).
    """
    # Gamma correction: suppresses weak activations, boosts strong ones
    cam_gamma = np.power(np.clip(cam, 0, 1), gamma)
    base_rgb  = np.stack([base_img] * 3, axis=-1)   # (H, W, 3)
    cam_uint8 = np.uint8(255 * cam_gamma)
    heat_bgr  = cv2.applyColorMap(cam_uint8, colormap)
    heat_rgb  = heat_bgr[:, :, ::-1].astype(np.float32) / 255.0
    # Use cam intensity as per-pixel alpha for cleaner overlay
    pixel_alpha = (cam_gamma[..., np.newaxis] * alpha)
    blended = (1 - pixel_alpha) * base_rgb + pixel_alpha * heat_rgb
    return np.clip(blended, 0, 1)


# ── MODEL ──────────────────────────────────────────────────────────────────
# ── ViT backbone with ATTENTION-BASED GradCAM hooks ──────────────────────────
class ViTForCAM(nn.Module):
    """
    ViT wrapper with ATTENTION-WEIGHT-BASED GradCAM.

    IMPORTANT: Standard GradCAM fails on ViTs that use CLS-token pooling
    because backward() sends gradients ONLY to the CLS token (position 0),
    leaving all 196 patch-token gradients at ZERO.  Hooking the block output
    and looking at patch gradients therefore produces an all-zero CAM.

    FIX: We hook the self-attention layer inside the last transformer block
    to capture the ATTENTION WEIGHTS.  The CLS token's attention over the
    196 patch tokens IS the spatial map of "where the model looks".

    GradCAM   = CLS→patch attention (averaged across heads)
    GradCAM++ = CLS→patch attention × gradient magnitude of the CLS feature
                (gradient-weighted attention — patches that receive more
                 gradient-informed CLS attention get higher weight)
    """
    def __init__(self, model_name=BACKBONE):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        self._attn_weights = None    # (B, heads, 197, 197)
        self._activations  = None    # (B, 197, D)  — block output
        self._gradients    = None    # (B, 197, D)  — gradient at block output

        last_block = self.backbone.blocks[-1]
        # Hook attention module to capture attention weights
        last_block.attn.register_forward_hook(self._attn_hook)
        # Hook block output for gradient-weighted refinement
        last_block.register_forward_hook(self._fwd_hook)
        last_block.register_full_backward_hook(self._bwd_hook)

    def _attn_hook(self, module, inp, out):
        """Capture raw attention weights from qkv attention."""
        # timm's Attention.forward computes attn_weights internally;
        # we need to recompute them from the input to get the weights.
        x = inp[0]  # (B, 197, D)
        B, N, C = x.shape
        qkv = module.qkv(x).reshape(B, N, 3, module.num_heads, C // module.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, heads, N, head_dim)
        q, k, _ = qkv.unbind(0)
        scale = (C // module.num_heads) ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale  # (B, heads, N, N)
        attn = attn.softmax(dim=-1)
        self._attn_weights = attn.detach()  # (B, heads, 197, 197)

    def _fwd_hook(self, module, inp, out):
        self._activations = out  # (B, 197, D)

    def _bwd_hook(self, module, grad_in, grad_out):
        self._gradients = grad_out[0]  # (B, 197, D)

    def forward(self, x):
        return self.backbone(x)   # (1, D=384)

    def get_cam(self, method="gradcam"):
        """
        Compute spatial CAM from attention weights.
        Returns a (14, 14) numpy array (NOT yet upsampled).
        """
        # CLS token's attention over 196 patches, averaged across heads
        # attn_weights shape: (B, heads, 197, 197)
        # CLS is position 0; patches are positions 1-196
        cls_attn = self._attn_weights[:, :, 0, 1:]  # (B, heads, 196)
        cam = cls_attn.mean(dim=1)                   # (B, 196) — average across heads

        if method == "gradcampp" and self._gradients is not None:
            # Gradient-weighted: modulate attention by gradient magnitude at CLS
            # The CLS gradient tells us how important different features are;
            # we use it to reweight the attention heads before averaging
            cls_grad = self._gradients[:, 0, :]  # (B, D) — gradient at CLS token
            head_dim = cls_grad.shape[-1] // self._attn_weights.shape[1]
            n_heads  = self._attn_weights.shape[1]
            # Per-head gradient magnitude
            grad_per_head = cls_grad.reshape(-1, n_heads, head_dim)  # (B, heads, head_dim)
            head_importance = grad_per_head.abs().mean(dim=-1)       # (B, heads)
            if head_importance.sum() > 1e-10:
                head_importance = head_importance / head_importance.sum(dim=-1, keepdim=True)
            else:
                head_importance = torch.ones_like(head_importance) / n_heads
            # Weighted average of per-head CLS→patch attention
            cam = (cls_attn * head_importance.unsqueeze(-1)).sum(dim=1)  # (B, 196)

        cam = cam.reshape(1, N_PATCHES_1D, N_PATCHES_1D)  # (1, 14, 14)
        cam = cam.squeeze(0).detach().cpu().numpy()        # (14, 14)
        # Normalize to [0, 1]
        if cam.max() > 1e-10:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-10)
        return cam


def make_brain_mask(arr2d, threshold=0.02):
    """Binary mask: 1 inside brain, 0 in black background."""
    mask   = (arr2d > threshold).astype(np.uint8)
    kernel = np.ones((11, 11), np.uint8)
    mask   = cv2.dilate(mask, kernel, iterations=3)
    return mask.astype(np.float32)


def compute_saliency_cam(vit, puma, slice_dict, true_feats, target_mod, method="gradcam"):
    """
    TRUE feature-map Grad-CAM / Grad-CAM++ via ViT last-block hooks.

    The gradient flows:
      image_pixels → ViT → (hooked feature maps) → CLS → PUMA → logit → backward
      Hook captures activations A and gradients dY/dA at the last transformer block.
      CAM = ReLU(sum_c(alpha_c * A_c)), reshaped 14×14 → upsampled to 224×224.

    true_feats: pre-extracted {mod: tensor(1,384)} for all 4 modalities
                (only the non-target mods are used — they stay fixed)
    """
    vit.eval()
    puma.eval()
    # Need train mode for backward hooks to fire reliably, but eval norm stats
    vit.backbone.blocks[-1].train()

    # Input tensor with gradient tracking (needed for backward to flow through ViT)
    img = arr2d_to_tensor(slice_dict[target_mod]).to(DEVICE)
    img.requires_grad_(True)

    # Extract feature for the target modality (gradient flows through)
    feat_target = vit(img)          # (1, 384) — hooks capture activations

    # Build per-modality feature tensors: shape (1, 1, 384)
    def _f(mod):
        if mod == target_mod:
            return feat_target.unsqueeze(1)
        return true_feats[mod].detach().unsqueeze(1)

    sm = torch.ones(1, 1, dtype=torch.bool, device=DEVICE)
    mm = torch.ones(1, 4, dtype=torch.bool, device=DEVICE)

    logits = puma(_f('t1'), _f('t1ce'), _f('t2'), _f('flair'), sm, mm)
    pred   = int(logits.argmax(1).item())
    conf   = float(torch.softmax(logits, 1).max().item())

    # Backpropagate — hooks capture gradients at last block
    logits[0, pred].backward()

    # Compute CAM from hooked activations/gradients
    cam_14 = vit.get_cam(method=method)  # (14, 14)

    # Upsample to 224×224 with bicubic interpolation for smooth heatmap
    cam = cv2.resize(cam_14, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)

    # Apply brain mask to zero out black background
    base  = arr2d_to_display(slice_dict[target_mod])
    bmask = make_brain_mask(base)
    cam   = cam * bmask

    # Light Gaussian blur for visual smoothness
    cam = cv2.GaussianBlur(cam, (5, 5), 0)
    if cam.max() > 1e-10:
        cam = cam / cam.max()

    if img.grad is not None:
        img.grad = None
    return np.clip(cam, 0, 1), pred, conf, base


# ── PUMA-MIL-Lite (minimal, for inference only) ────────────────────────────
def masked_topk_mean(sf, scores, mask, topk):
    bsz, _, dim = sf.shape; out = []
    for b in range(bsz):
        vi = torch.where(mask[b])[0]
        if vi.numel() == 0: out.append(torch.zeros(dim, device=sf.device)); continue
        k = min(int(topk), int(vi.numel()))
        ch = vi[torch.topk(scores[b, vi], k).indices]
        out.append(sf[b, ch].mean(0))
    return torch.stack(out)

class ReliabilityFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.ModuleList([
            nn.Sequential(nn.Linear(FEATURE_DIM, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU())
            for _ in range(4)])
        self.rel = nn.ModuleList([nn.Linear(D_MODEL, 1) for _ in range(4)])
        self.out = nn.Sequential(nn.Linear(D_MODEL*2, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU())
    def forward(self, mods, mm):
        pf = [self.proj[i](mods[i]) for i in range(4)]
        rl = [self.rel[i](pf[i]) for i in range(4)]
        fs = torch.stack(pf, 2); rs = torch.cat(rl, -1).masked_fill(~mm[:,None,:], -1e4)
        w = torch.softmax(rs, -1)
        fu = (fs * w.unsqueeze(-1)).sum(2)
        v = mm[:,None,:,None].float()
        mf = (fs*v).sum(2)/v.sum(2).clamp_min(1)
        return self.out(torch.cat([fu,mf],-1)), w

class HybridMILPool(nn.Module):
    def __init__(self):
        super().__init__()
        self.topk = TOP_K
        self.attn = nn.Sequential(nn.Linear(D_MODEL, D_MODEL//2), nn.Tanh(), nn.Linear(D_MODEL//2, 1))
    def forward(self, sf, sl, sm):
        p = torch.softmax(sl, -1)
        e = -(p*torch.log(p.clamp_min(1e-8))).sum(-1)/math.log(2)
        r = 1-e
        al = self.attn(sf).squeeze(-1)+torch.log(r.clamp_min(1e-6))
        al = al.masked_fill(~sm, -1e9)
        aw = torch.softmax(al, 1)
        ap = (sf*aw.unsqueeze(-1)).sum(1)
        pp = masked_topk_mean(sf, p[...,1], sm, self.topk)
        np_ = masked_topk_mean(sf, p[...,0], sm, self.topk)
        return torch.cat([ap,pp,np_],-1), aw, r

class PUMAMILLite(nn.Module):
    def __init__(self):
        super().__init__()
        self.fusion    = ReliabilityFusion()
        self.pos_embed = nn.Parameter(torch.zeros(1, MAX_SLICES, D_MODEL))
        el = nn.TransformerEncoderLayer(D_MODEL,4,256,0.1,batch_first=True,activation="gelu")
        self.encoder    = nn.TransformerEncoder(el, 2)
        self.slice_head = nn.Sequential(nn.LayerNorm(D_MODEL), nn.Linear(D_MODEL,2))
        self.pool       = HybridMILPool()
        self.register_buffer("prototypes",   torch.zeros(2, D_MODEL*3))
        self.register_buffer("proto_counts", torch.zeros(2))
        self.final_head = nn.Sequential(
            nn.LayerNorm(D_MODEL*3+2), nn.Linear(D_MODEL*3+2, D_MODEL),
            nn.GELU(), nn.Dropout(0.25), nn.Linear(D_MODEL, 2))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
    def proto_sims(self, be):
        s = []
        for c in [0,1]:
            if self.proto_counts[c]<=0 or torch.norm(self.prototypes[c])<1e-8:
                s.append(torch.zeros(be.size(0),1,device=be.device))
            else:
                s.append(F.cosine_similarity(be, self.prototypes[c].unsqueeze(0),-1).unsqueeze(-1))
        return torch.cat(s,-1)
    def forward(self, t1,t1ce,t2,flair, sm,mm):
        fu, _ = self.fusion([t1,t1ce,t2,flair], mm)
        x = fu + self.pos_embed[:,:t1.size(1),:]
        x = self.encoder(x, src_key_padding_mask=~sm)
        sl = self.slice_head(x)
        # ── CRITICAL FIX ──────────────────────────────────────────────
        # Phase 2 evaluates via:  center_logits = slice_logits[:, center_pos, :]
        # NOT via bag_logits from final_head (which answers "does this
        # PATIENT have any tumor somewhere?" → nearly always yes).
        # Center position = middle of the window (WINDOW_HALF for 11-slice)
        center_pos = t1.size(1) // 2
        return sl[:, center_pos, :]   # (B, 2) — center-slice logits


def load_models():
    """Load ViT (with gradients) and PUMA checkpoint."""
    vit = ViTForCAM(BACKBONE).to(DEVICE)
    # keep gradients enabled for CAM
    for p in vit.parameters():
        p.requires_grad_(True)
    vit.train()   # need train mode so backward works through BatchNorm etc.

    puma = PUMAMILLite().to(DEVICE)
    ckpt = next((p for p in CKPT_CANDIDATES if os.path.exists(p)), None)
    if ckpt is None:
        raise FileNotFoundError("No PUMA checkpoint found")
    state = torch.load(ckpt, map_location=DEVICE)
    if any(k.startswith("module.") for k in state):
        state = {k.replace("module.",""):v for k,v in state.items()}
    puma.load_state_dict(state, strict=False)
    puma.eval()
    return vit, puma, ckpt


# ── SAMPLE SELECTOR ────────────────────────────────────────────────────────
def select_samples(n_pos=N_POS, n_neg=N_NEG):
    """
    Pick n_pos tumor slices and n_neg non-tumor slices from the
    test split of slice_metadata_v3.csv.

    Automatically extracts the BraTS tar if the NIfTI files are no
    longer present at the paths stored in the CSV (/kaggle/temp gets
    wiped between Kaggle sessions).

    Returns a list of dicts: {pid, slice_idx, label, t1_path, …}
    """
    if not os.path.exists(SLICE_META_CSV):
        raise FileNotFoundError(
            f"Not found: {SLICE_META_CSV}\n"
            "Phase 1 must have run first to create slice_metadata_v3.csv.")

    df = pd.read_csv(SLICE_META_CSV)

    for col in ["t1_path","t1ce_path","t2_path","flair_path"]:
        if col not in df.columns:
            raise KeyError(f"slice_metadata_v3.csv missing column: {col}")

    # Prefer test split; fall back to all rows if column absent or split empty
    if "split" in df.columns:
        test_df = df[df["split"] == "test"].copy()
        if len(test_df) == 0:
            test_df = df.copy()
    else:
        test_df = df.copy()

    # ── First check: are the stored paths already valid? ─────────────────
    def _all_exist(row):
        return all(os.path.exists(str(row[c]))
                   for c in ["t1_path","t1ce_path","t2_path","flair_path"])

    valid_mask = test_df.apply(_all_exist, axis=1)
    n_valid    = valid_mask.sum()

    if n_valid == 0:
        # Paths are stale (/kaggle/temp was wiped).
        # Extract the tar into /kaggle/working (persistent) and remap.
        print("  [INFO] Stored NIfTI paths are stale (likely /kaggle/temp was wiped).")
        print("  [INFO] Will extract BraTS 2021 tar into /kaggle/working/brats2021_extracted/")
        extract_root = ensure_brats_extracted()
        test_df   = repair_paths(test_df, extract_root)
        valid_mask = test_df.apply(_all_exist, axis=1)
        n_valid    = valid_mask.sum()
        print(f"  [INFO] After remapping: {n_valid}/{len(test_df)} slices have valid paths")

    test_df = test_df[valid_mask].reset_index(drop=True)

    if len(test_df) == 0:
        raise RuntimeError(
            "No slices found with valid NIfTI paths even after tar extraction.\n"
            f"Extraction root checked: {BRATS_EXTRACT}\n"
            "Please verify the BraTS 2021 dataset is attached to this notebook.")

    # Sample pos and neg rows
    pos_pool = test_df[test_df["label"] == 1]
    neg_pool = test_df[test_df["label"] == 0]
    pos_rows = pos_pool.sample(n=min(n_pos, len(pos_pool)), random_state=42)
    neg_rows = neg_pool.sample(n=min(n_neg, len(neg_pool)), random_state=42)
    samples  = pd.concat([pos_rows, neg_rows]).reset_index(drop=True)

    result = []
    for _, row in samples.iterrows():
        result.append({
            "pid":       str(row["patient_id"]),
            "slice_idx": int(row["slice_idx"]),
            "label":     int(row["label"]),
            "role":      str(row.get("slice_role", "unknown")),
            "t1":        str(row["t1_path"]),
            "t1ce":      str(row["t1ce_path"]),
            "t2":        str(row["t2_path"]),
            "flair":     str(row["flair_path"]),
        })
    return result


WINDOW_HALF = 5   # TRAIN_WINDOW = 11 → half = 5

def load_sample_slices(sample):
    """Load 2D centre slice (for display). Returns {mod: arr(H,W)}."""
    return {mod: load_slice(sample[mod], sample["slice_idx"]) for mod in MODALITIES}


def load_window_vols(sample):
    """
    Load all 4 NIfTI volumes for a sample (lazy, cached per call).
    Returns {mod: vol(H,W,D) float32 normalised}.
    """
    vols = {}
    for mod in MODALITIES:
        path = sample[mod]
        vol  = nib.load(path).get_fdata().astype(np.float32)
        vols[mod] = percentile_normalize(vol)
    return vols


def extract_window_features(vit, vols, center_idx, override_mod=None, override_img_u8=None):
    """
    Extract ViT features for an 11-slice window centred on center_idx.

    vols          : {mod: vol(H,W,D)}
    center_idx    : int, the slice we want to explain
    override_mod  : modality name to replace at center position (for LIME/SHAP)
    override_img_u8 : (224,224,3) uint8 — replaces center slice of override_mod

    Returns:
        feats : {mod: tensor(1, W, 384)}  W=window_size=11
        seq_mask : tensor(1, W) bool
        center_local : int — position of center slice within window (=WINDOW_HALF)
    """
    D          = vols["t1"].shape[2]
    half       = WINDOW_HALF
    win_size   = 2 * half + 1   # 11

    vit.eval()
    tfm = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    feats    = {mod: [] for mod in MODALITIES}
    seq_mask = []

    for offset in range(-half, half + 1):
        z     = center_idx + offset
        valid = (0 <= z < D)
        seq_mask.append(valid)

        for mod in MODALITIES:
            if not valid:
                feats[mod].append(torch.zeros(FEATURE_DIM, device=DEVICE))
                continue

            # Override center slice for explanation methods
            if (override_mod == mod) and (offset == 0) and (override_img_u8 is not None):
                t = tfm(Image.fromarray(override_img_u8)).unsqueeze(0).to(DEVICE)
            else:
                arr2d = vols[mod][:, :, z]
                arr2d = np.clip(arr2d, 0, 1)
                img   = Image.fromarray((arr2d * 255).astype(np.uint8)).convert("RGB")
                t     = tfm(img).unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                f = vit(t)       # (1, 384)
            feats[mod].append(f.squeeze(0))

    # Stack to (1, W, D)
    result = {}
    for mod in MODALITIES:
        result[mod] = torch.stack(feats[mod], dim=0).unsqueeze(0)  # (1, W, 384)
    seq_mask_t = torch.tensor(seq_mask, dtype=torch.bool, device=DEVICE).unsqueeze(0)  # (1, W)
    return result, seq_mask_t, half


def run_puma_window(puma, feats, seq_mask):
    """Run PUMA on a full 11-slice window. Returns logits (1,2)."""
    puma.eval()
    mm = torch.ones(1, 4, dtype=torch.bool, device=DEVICE)
    with torch.no_grad():
        logits = puma(feats["t1"], feats["t1ce"], feats["t2"], feats["flair"],
                      seq_mask, mm)
    return logits


# ── GRAD-CAM: hook-based at center slice with full window context ──────────
def forward_center_grad(vit, puma, vols, center_idx, target_mod="t1ce"):
    """
    Extract features for the full 11-slice window with grad enabled only for
    the center slice of target_mod.  After backward(), use vit.get_cam() to
    retrieve the attention-based GradCAM from the hooked transformer block.

    IMPORTANT: The attention hooks fire on EVERY vit(t) call, so we explicitly
    snapshot the attention weights from the target slice's forward pass.
    Otherwise they'd be overwritten by subsequent modality/offset calls.

    Returns (logits, pred, conf).
    """
    vit.eval(); puma.eval()
    # Enable train on last block so backward hooks fire
    vit.backbone.blocks[-1].train()
    D    = vols["t1"].shape[2]
    half = WINDOW_HALF
    tfm  = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    feats_all  = {mod: [] for mod in MODALITIES}
    seq_mask   = []
    saved_attn = None  # will hold attention weights from the TARGET slice

    for offset in range(-half, half + 1):
        z     = center_idx + offset
        valid = (0 <= z < D)
        seq_mask.append(valid)
        for mod in MODALITIES:
            if not valid:
                feats_all[mod].append(torch.zeros(FEATURE_DIM, device=DEVICE))
                continue
            arr2d = np.clip(vols[mod][:, :, z], 0, 1)
            img   = Image.fromarray((arr2d * 255).astype(np.uint8)).convert("RGB")
            t     = tfm(img).unsqueeze(0).to(DEVICE)
            if mod == target_mod and offset == 0:
                t.requires_grad_(True)
                f = vit(t)           # gradient flows; hooks capture activations
                # ── SNAPSHOT attention weights from this specific forward pass ──
                saved_attn = vit._attn_weights.clone() if vit._attn_weights is not None else None
            else:
                with torch.no_grad():
                    f = vit(t)
            feats_all[mod].append(f.squeeze(0))

    # Restore the target slice's attention weights (overwritten by subsequent calls)
    if saved_attn is not None:
        vit._attn_weights = saved_attn

    f_dict = {mod: torch.stack(feats_all[mod],0).unsqueeze(0) for mod in MODALITIES}
    sm     = torch.tensor(seq_mask, dtype=torch.bool, device=DEVICE).unsqueeze(0)
    mm     = torch.ones(1,4,dtype=torch.bool,device=DEVICE)
    logits = puma(f_dict["t1"],f_dict["t1ce"],f_dict["t2"],f_dict["flair"],sm,mm)
    pred   = int(logits.argmax(1).item())
    conf   = float(torch.softmax(logits,1).max().item())
    return logits, pred, conf


# ══════════════════════════════════════════════════════════════════════════
#  1. GRAD-CAM / GRAD-CAM++   (SEPARATE PNGs for each method)
# ══════════════════════════════════════════════════════════════════════════
def _pre_extract_features(vit, slice_dict):
    """Extract ViT features for all 4 modalities (single slice). Returns {mod: tensor(1,384)}."""
    feats = {}
    with torch.no_grad():
        for mod in MODALITIES:
            t = arr2d_to_tensor(slice_dict[mod]).to(DEVICE)
            feats[mod] = vit(t)   # (1, 384)
    return feats


def _collect_cams(vit, puma, samples, target_mod="t1ce"):
    """
    TRUE feature-map Grad-CAM & Grad-CAM++ for each sample.
    Uses the full 11-slice window context (matching Phase 2 training) via
    forward_center_grad + vit.get_cam().

    The ViT hooks capture activations/gradients at the last transformer block,
    then get_cam() computes the 14×14 CAM which is upsampled to 224×224.
    """
    results = []
    for sample in samples:
        slices = load_sample_slices(sample)
        base   = arr2d_to_display(slices[target_mod])

        # Load full 4-mod volumes for the windowed approach
        vols = load_window_vols(sample)
        center_idx = sample["slice_idx"]

        # --- GradCAM ---
        logits, pred, conf = forward_center_grad(
            vit, puma, vols, center_idx, target_mod=target_mod)
        logits[0, pred].backward(retain_graph=False)
        cam_gc_14 = vit.get_cam(method="gradcam")   # (14, 14)
        cam_gc = cv2.resize(cam_gc_14, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)

        # --- GradCAM++ ---
        logits2, _, _ = forward_center_grad(
            vit, puma, vols, center_idx, target_mod=target_mod)
        logits2[0, pred].backward(retain_graph=False)
        cam_gpp_14 = vit.get_cam(method="gradcampp")  # (14, 14)
        cam_gpp = cv2.resize(cam_gpp_14, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)

        # Apply brain mask & light Gaussian blur
        bmask = make_brain_mask(base)
        cam_gc  = cam_gc  * bmask
        cam_gpp = cam_gpp * bmask
        cam_gc  = cv2.GaussianBlur(cam_gc,  (5, 5), 0)
        cam_gpp = cv2.GaussianBlur(cam_gpp, (5, 5), 0)
        if cam_gc.max()  > 1e-10: cam_gc  = cam_gc  / cam_gc.max()
        if cam_gpp.max() > 1e-10: cam_gpp = cam_gpp / cam_gpp.max()
        cam_gc  = np.clip(cam_gc,  0, 1)
        cam_gpp = np.clip(cam_gpp, 0, 1)

        results.append({
            "base": base, "cam_gc": cam_gc, "cam_gpp": cam_gpp,
            "pred": pred, "conf": conf,
            "label": sample["label"], "pid": sample["pid"],
        })
        clear_mem()
    return results


def _save_cam_figure(cam_data, method_name, out_path):
    """
    Save a 2-column figure:  col0=Original MRI  |  col1=CAM heatmap overlay
    One row per sample.  SEPARATE file per method.
    """
    n = len(cam_data)
    if n == 0:
        print(f"  [SKIP] no data for {method_name}")
        return
    fig, axes = plt.subplots(n, 2, figsize=(8, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]
    for i, d in enumerate(cam_data):
        lbl_str  = "Tumor" if d["label"] == 1 else "Non-Tumor"
        pred_str = "Tumor" if d["pred"]  == 1 else "Non-Tumor"
        cam_key  = "cam_gc" if method_name == "Grad-CAM" else "cam_gpp"
        axes[i, 0].imshow(d["base"], cmap="gray", vmin=0, vmax=1)
        axes[i, 0].set_title(f"Original T1ce\nTrue: {lbl_str}", fontsize=10)
        axes[i, 0].axis("off")
        axes[i, 1].imshow(overlay_heatmap(d["base"], d[cam_key]))
        axes[i, 1].set_title(f"{method_name} Heatmap\nPred: {pred_str}  (conf={d['conf']:.2f})", fontsize=10)
        axes[i, 1].axis("off")
    plt.suptitle(f"{method_name} — PUMA-MIL-Lite  (T1ce modality)",
                 fontsize=13, fontweight="bold", y=0.99)
    plt.subplots_adjust(top=0.93)   # reserve space for suptitle
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()
    print(f"  ✓ {os.path.basename(out_path)}")


def run_gradcam(vit, puma, samples):
    print("\n[1/3] GradCAM & GradCAM++ ...")

    cam_data = _collect_cams(vit, puma, samples, target_mod="t1ce")

    # ── GradCAM — separate PNG ─────────────────────────────────────────────
    _save_cam_figure(cam_data, "Grad-CAM",   OUT_GRADCAM)

    # ── GradCAM++ — separate PNG ───────────────────────────────────────────
    _save_cam_figure(cam_data, "Grad-CAM++", OUT_GRADCAMPP)

    # ── All-4-modality combined figure (first sample only) ─────────────────
    # Uses forward_center_grad with FULL 11-slice window context per modality
    sample = samples[0]
    slices_4mod = load_sample_slices(sample)
    vols_4mod   = load_window_vols(sample)
    center_idx  = sample["slice_idx"]
    out_4mod = os.path.join(WORK_DIR, "interp_gradcam_4modalities.png")
    fig2, axes2 = plt.subplots(4, 3, figsize=(12, 15))
    for mi, mod in enumerate(MODALITIES):
        base = arr2d_to_display(slices_4mod[mod])
        # GradCAM with full 11-slice window context
        logits_gc, pred, conf = forward_center_grad(
            vit, puma, vols_4mod, center_idx, target_mod=mod)
        logits_gc[0, pred].backward(retain_graph=False)
        cam_gc_14 = vit.get_cam(method="gradcam")
        cam_gc = cv2.resize(cam_gc_14, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
        # GradCAM++ with full 11-slice window context
        logits_gpp, _, _ = forward_center_grad(
            vit, puma, vols_4mod, center_idx, target_mod=mod)
        logits_gpp[0, pred].backward(retain_graph=False)
        cam_gpp_14 = vit.get_cam(method="gradcampp")
        cam_gpp = cv2.resize(cam_gpp_14, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
        # Apply brain mask & Gaussian blur
        bmask = make_brain_mask(base)
        cam_gc  = cam_gc  * bmask
        cam_gpp = cam_gpp * bmask
        cam_gc  = cv2.GaussianBlur(cam_gc,  (5, 5), 0)
        cam_gpp = cv2.GaussianBlur(cam_gpp, (5, 5), 0)
        if cam_gc.max()  > 1e-10: cam_gc  = cam_gc  / cam_gc.max()
        if cam_gpp.max() > 1e-10: cam_gpp = cam_gpp / cam_gpp.max()
        cam_gc  = np.clip(cam_gc,  0, 1)
        cam_gpp = np.clip(cam_gpp, 0, 1)
        lbl_str  = "Tumor" if sample["label"]==1 else "Non-Tumor"
        pred_str = "Tumor" if pred==1 else "Non-Tumor"
        axes2[mi, 0].imshow(base, cmap="gray", vmin=0, vmax=1)
        axes2[mi, 0].set_title(f"{mod.upper()}  |  True: {lbl_str}", fontsize=9)
        axes2[mi, 1].imshow(overlay_heatmap(base, cam_gc))
        axes2[mi, 1].set_title(f"Grad-CAM  |  Pred: {pred_str} ({conf:.2f})", fontsize=9)
        axes2[mi, 2].imshow(overlay_heatmap(base, cam_gpp))
        axes2[mi, 2].set_title(f"Grad-CAM++  |  Pred: {pred_str} ({conf:.2f})", fontsize=9)
        for ax in axes2[mi]: ax.axis("off")
        clear_mem()
    plt.suptitle(
        f"Grad-CAM — All 4 Modalities\n"
        f"Patient: {sample['pid']}  |  True: {'Tumor' if sample['label']==1 else 'Non-Tumor'}",
        fontsize=12, fontweight="bold", y=0.99)
    plt.subplots_adjust(top=0.92)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(out_4mod, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()
    print(f"  ✓ {os.path.basename(out_4mod)}")



# ══════════════════════════════════════════════════════════════════════════
#  2. LIME  — Yellow boundary contour style (matching reference Fig. 10)
# ══════════════════════════════════════════════════════════════════════════
def run_lime(vit, puma, samples):
    print("\n[2/3] LIME ...")
    try:
        from lime import lime_image
    except ImportError:
        print("  [SKIP] pip install lime  then re-run"); return

    tfm = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    explainer = lime_image.LimeImageExplainer()
    n_rows = len(samples)
    fig, axes = plt.subplots(n_rows, 2, figsize=(8, 4*n_rows))
    if n_rows == 1: axes = axes[np.newaxis, :]

    for row_i, sample in enumerate(samples):
        slices = load_sample_slices(sample)
        arr    = arr2d_to_display(slices["t1ce"])
        img_rgb = np.stack([arr]*3, axis=-1)
        img_u8  = np.uint8(np.clip(img_rgb, 0, 1) * 255)

        # Load full volumes for windowed extraction
        vols       = load_window_vols(sample)
        center_idx = sample["slice_idx"]

        def lime_predict(imgs_uint8, _vols=vols, _ci=center_idx):
            """Perturb ONLY center T1ce slice within a proper 11-slice window."""
            probs = []
            vit.eval(); puma.eval()
            for img_p in imgs_uint8:
                feats, seq_mask, _ = extract_window_features(
                    vit, _vols, _ci,
                    override_mod="t1ce", override_img_u8=img_p)
                with torch.no_grad():
                    logit = run_puma_window(puma, feats, seq_mask)
                    prob  = torch.softmax(logit, 1).cpu().numpy()[0]
                probs.append(prob)
            return np.asarray(probs, dtype=np.float32)

        brain_mean = int(img_u8[img_u8 > 10].mean()) if (img_u8 > 10).any() else 60

        explanation = explainer.explain_instance(
            img_u8, lime_predict,
            top_labels=2, hide_color=brain_mean,
            num_samples=500, random_seed=42,
        )
        pred_label = explanation.top_labels[0]
        lbl_str    = "Tumor" if sample["label"]==1 else "Non-Tumor"
        pred_str   = "Tumor" if pred_label==1     else "Non-Tumor"

        sp_weights = dict(explanation.local_exp[pred_label])
        segs       = explanation.segments

        sp_sorted = sorted(sp_weights.items(), key=lambda x: x[1], reverse=True)
        n_pos_sp  = sum(1 for _,w in sp_sorted if w > 0)
        n_neg_sp  = sum(1 for _,w in sp_sorted if w < 0)
        TOP_K_LIME = min(8, max(n_pos_sp, 1))

        print(f"  {sample['pid']}  true={'T' if sample['label']==1 else 'N'}  "
              f"pred={'T' if pred_label==1 else 'N'}  "
              f"n_pos={n_pos_sp}  n_neg={n_neg_sp}  "
              f"max_w={max((abs(w) for _,w in sp_sorted), default=0):.5f}")

        # ── Yellow boundary contours on grayscale (reference paper style) ──
        pos_sps = [sp_id for sp_id, w in sp_sorted if w > 0][:TOP_K_LIME]
        # Build grayscale RGB canvas
        canvas = np.uint8(np.clip(arr, 0, 1) * 255)
        canvas_rgb = cv2.cvtColor(canvas, cv2.COLOR_GRAY2BGR)
        # Draw yellow contours ONLY for superpixels inside the brain
        YELLOW = (0, 255, 255)  # BGR for OpenCV
        for sp_id in pos_sps:
            sp_mask = np.uint8(segs == sp_id)
            # Skip superpixels in black background (mean intensity < 5%)
            sp_mean = arr[sp_mask > 0].mean() if (sp_mask > 0).any() else 0
            if sp_mean < 0.05:
                continue
            contours, _ = cv2.findContours(sp_mask, cv2.RETR_EXTERNAL,
                                           cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(canvas_rgb, contours, -1, YELLOW, thickness=2)
        lime_vis = canvas_rgb[:, :, ::-1].astype(np.float32) / 255.0  # BGR→RGB

        # Plot: Original | LIME contours
        axes[row_i,0].imshow(arr, cmap="gray", vmin=0, vmax=1)
        axes[row_i,0].set_title(f"Original T1ce\nTrue: {lbl_str}", fontsize=10)
        axes[row_i,0].axis("off")
        axes[row_i,1].imshow(lime_vis)
        axes[row_i,1].set_title(f"LIME Explanation\nPred: {pred_str}", fontsize=10)
        axes[row_i,1].axis("off")
        clear_mem()

    plt.suptitle("LIME Superpixel Explanations — T1ce (11-slice window)",
                 fontsize=13, fontweight="bold", y=0.99)
    plt.subplots_adjust(top=0.93)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(OUT_LIME, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()
    print(f"  ✓ {os.path.basename(OUT_LIME)}")


# ══════════════════════════════════════════════════════════════════════════
#  3. SHAP  — GradientExplainer, red-green diverging colormap (ref Fig. 13)
# ══════════════════════════════════════════════════════════════════════════
def run_shap(vit, puma, samples):
    print("\n[3/3] SHAP (image-level, 11-slice window context) ...")
    try:
        import shap
    except ImportError:
        print("  [SKIP] pip install shap  then re-run"); return
    from matplotlib.colors import TwoSlopeNorm
    from mpl_toolkits.axes_grid1 import make_axes_locatable

    n_rows = len(samples)
    fig, axes = plt.subplots(n_rows, 3, figsize=(14, 4.5*n_rows))
    if n_rows == 1: axes = axes[np.newaxis, :]

    for row_i, sample in enumerate(samples):
        slices = load_sample_slices(sample)
        base   = arr2d_to_display(slices["t1ce"])
        t_img  = arr2d_to_tensor(slices["t1ce"]).to(DEVICE)

        # Load full volumes for 11-slice window context
        vols       = load_window_vols(sample)
        center_idx = sample["slice_idx"]

        # Pre-extract NON-T1ce window features (these stay fixed during SHAP)
        D    = vols["t1"].shape[2]
        half = WINDOW_HALF
        tfm  = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])

        fixed_feats = {mod: [] for mod in MODALITIES}
        seq_mask_list = []
        for offset in range(-half, half + 1):
            z     = center_idx + offset
            valid = (0 <= z < D)
            seq_mask_list.append(valid)
            for mod in MODALITIES:
                if not valid:
                    fixed_feats[mod].append(torch.zeros(FEATURE_DIM, device=DEVICE))
                    continue
                if mod == "t1ce" and offset == 0:
                    fixed_feats[mod].append(None)
                    continue
                arr2d = np.clip(vols[mod][:, :, z], 0, 1)
                img   = Image.fromarray((arr2d * 255).astype(np.uint8)).convert("RGB")
                t     = tfm(img).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    f = vit(t).squeeze(0)
                fixed_feats[mod].append(f)

        sm_w = torch.tensor(seq_mask_list, dtype=torch.bool, device=DEVICE).unsqueeze(0)
        mm_w = torch.ones(1, 4, dtype=torch.bool, device=DEVICE)
        center_local = half

        class E2E(nn.Module):
            def forward(self_inner, x):
                bsz = x.size(0)
                f_center = vit(x)
                results_list = []
                for b in range(bsz):
                    feat_window = {mod: [] for mod in MODALITIES}
                    for w_i in range(len(seq_mask_list)):
                        for mod in MODALITIES:
                            if mod == "t1ce" and w_i == center_local:
                                feat_window[mod].append(f_center[b])
                            else:
                                feat_window[mod].append(fixed_feats[mod][w_i])
                    fw = {mod: torch.stack(feat_window[mod], 0).unsqueeze(0)
                          for mod in MODALITIES}
                    logit = puma(fw["t1"], fw["t1ce"], fw["t2"], fw["flair"],
                                sm_w, mm_w)
                    results_list.append(logit)
                return torch.cat(results_list, dim=0)

        e2e = E2E().to(DEVICE)
        e2e.train()

        bg_pool = [s for s in samples if s["label"] != sample["label"]]
        if not bg_pool: bg_pool = [s for s in samples if s is not sample]
        bg = torch.cat([arr2d_to_tensor(load_sample_slices(s)["t1ce"]).to(DEVICE)
                        for s in bg_pool[:2]], dim=0)

        explainer = shap.GradientExplainer(e2e, bg)
        raw = explainer.shap_values(t_img)

        if isinstance(raw, tuple): raw = raw[0]
        if not isinstance(raw, list): raw = [raw]
        raw = [np.asarray(v) for v in raw]

        def _to2d(arr_in):
            arr_in = np.asarray(arr_in)
            if arr_in.ndim == 5: arr_in = arr_in[..., 1]
            if arr_in.ndim == 4: arr_in = arr_in.squeeze(0)
            if arr_in.ndim == 3 and arr_in.shape[0]==3: arr_in = arr_in.mean(0)
            if arr_in.ndim == 3 and arr_in.shape[-1]==3: arr_in = arr_in.mean(-1)
            return arr_in.astype(np.float32)

        if len(raw) >= 2:
            sv_tumor    = _to2d(raw[1])
            sv_nontumor = _to2d(raw[0])
        else:
            sv_all      = _to2d(raw[0])
            sv_tumor    = sv_all.copy()
            sv_nontumor = -sv_all.copy()

        # ── Red-green diverging SHAP overlay on grayscale (ref Fig. 13) ──
        # Build a direct red↔green colormap (no yellow middle) for max contrast
        from matplotlib.colors import LinearSegmentedColormap
        _rdgn = LinearSegmentedColormap.from_list(
            "RdGn", ["#d62728", "#f0f0f0", "#2ca02c"], N=256)

        def _shap_diverging(ax, grey, sv, title):
            """
            Overlay SHAP values using Red↔Green diverging colormap on grayscale.
            Red = negative SHAP (opposing), Green = positive SHAP (supporting).
            Includes a colorbar below the plot.
            """
            bmask = make_brain_mask(grey)
            sv_masked = sv * bmask

            # Symmetric normalization around 0
            vmax = max(np.abs(sv_masked).max(), 1e-10)
            norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

            # Show grayscale base
            ax.imshow(grey, cmap="gray", vmin=0, vmax=1)
            # Overlay SHAP with transparency; alpha keyed to magnitude
            # Use aggressive alpha so colors are clearly visible
            sv_alpha = np.clip(np.abs(sv_masked) / vmax, 0, 1)
            # Power-scale alpha for stronger visibility of moderate values
            sv_alpha = np.power(sv_alpha, 0.5) * 0.92
            sv_alpha[bmask < 0.5] = 0  # zero outside brain
            rgba = _rdgn(norm(sv_masked))  # (H,W,4) RGBA — red-green
            rgba[..., 3] = sv_alpha       # per-pixel alpha
            ax.imshow(rgba)
            ax.set_title(title, fontsize=10)
            ax.axis("off")
            # Add colorbar
            divider = make_axes_locatable(ax)
            cax = divider.append_axes("bottom", size="5%", pad=0.08)
            sm_cb = cm.ScalarMappable(cmap=_rdgn, norm=norm)
            sm_cb.set_array([])
            cb = plt.colorbar(sm_cb, cax=cax, orientation="horizontal")
            cb.set_label("SHAP value", fontsize=8)
            cb.ax.tick_params(labelsize=7)

        lbl_str = "Tumor" if sample["label"]==1 else "Non-Tumor"
        axes[row_i,0].imshow(base, cmap="gray", vmin=0, vmax=1)
        axes[row_i,0].set_title(f"Original T1ce\nTrue: {lbl_str}", fontsize=10)
        axes[row_i,0].axis("off")
        _shap_diverging(axes[row_i,1], base, sv_tumor,
                        "SHAP — Tumor class")
        _shap_diverging(axes[row_i,2], base, sv_nontumor,
                        "SHAP — Non-Tumor class")
        del raw; clear_mem()

    plt.suptitle("SHAP Gradient Explanations — T1ce modality (11-slice window)",
                 fontsize=13, fontweight="bold", y=0.99)
    plt.subplots_adjust(top=0.93, hspace=0.35)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(OUT_SHAP, dpi=FIG_DPI, bbox_inches="tight")
    plt.close()
    print(f"  ✓ {os.path.basename(OUT_SHAP)}")



# ══════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    ensure_dir(WORK_DIR)

    print("=" * 70)
    print("PHASE 4B — IMAGE-LEVEL INTERPRETABILITY")
    print("GradCAM | GradCAM++ | LIME | SHAP  on raw MRI slices")
    print("=" * 70)

    print("\nLoading models ...")
    vit, puma, ckpt = load_models()
    print(f"  ✓ PUMA checkpoint: {os.path.basename(ckpt)}")
    print(f"  ✓ ViT backbone:    {BACKBONE}")
    print(f"  ✓ Device:          {DEVICE}")

    print("\nSelecting representative slices ...")
    samples = select_samples(n_pos=N_POS, n_neg=N_NEG)
    print(f"  ✓ {len(samples)} slices selected "
          f"({sum(s['label']==1 for s in samples)} tumor, "
          f"{sum(s['label']==0 for s in samples)} non-tumor)")

    run_gradcam(vit, puma, samples)
    run_lime(vit, puma, samples)
    run_shap(vit, puma, samples)

    print("\n" + "=" * 70)
    print("✅  COMPLETE")
    print("=" * 70)
    for f in [OUT_GRADCAM,
              OUT_GRADCAMPP,
              os.path.join(WORK_DIR, "interp_gradcam_4modalities.png"),
              OUT_LIME, OUT_SHAP]:
        if os.path.exists(f):
            kb = os.path.getsize(f)/1024
            print(f"  ✓ {os.path.basename(f):50s} ({kb:.0f} KB)")

In [ ]:
import shutil, os

folder = "/kaggle/working/brats2021_extracted"

if os.path.exists(folder):
    print(f"Deleting {folder} ...")
    shutil.rmtree(folder)
    print("✓ Done. Folder deleted.")
else:
    print("Folder does not exist — nothing to delete.")